In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pymongo import MongoClient
from sklearn.metrics import confusion_matrix
from IPython.display import display
from sklearn.metrics import precision_score, recall_score, f1_score
import statsmodels.api as sm
from statsmodels.formula.api import ols


In [ ]:
MONGODB_HOST = 'your host'
MONGODB_PORT = 27017
MONGODB_NAME = 'RAGQA'

# Función para obtener la base de datos
def get_db():
    client = MongoClient(MONGODB_HOST, MONGODB_PORT)
    return client[MONGODB_NAME]

def load_data(collection):
    db = get_db()
    eval_collection = db[collection]
    eval_data = list(eval_collection.find({}))
    return pd.DataFrame(eval_data)


In [ ]:
EVAL_COLLECTION_LIST_SUPERVISED = ["ia_eval_supervised", "ia_deepseek_eval_supervised", "ia_deepseek-r1_70b_eval_supervised", "ia_phi4_eval_supervised", "ia_qwen_2_5_eval_supervised", "eval_supervised"]
EVAL_COLLECTION_LIST = [ "ia_eval", "ia_deepseek_eval", "ia_deepseek-r1_70b_eval", "ia_phi_4_eval", "ia_qwen_2_5_eval", "eval"]
EVAL_NAMES = [ "Llama3.1:7b", "Deepseek-r1:7b", "Deepseek-r1:70b", "Phi4", "Qwen2.5", "Human"]
EVAL_COLLECTION_LIST_BIASED = [ "ia_biased_eval", "ia_biased_deepseek_eval", "ia_biased_deepseek-r1_70b_eval", "ia_biased_phi4_eval", "ia_biased_qwen_2_5_eval"]


In [ ]:
def create_connection_data(df_score, x_positions, llm_types):
    """
    Crea los datos para las líneas de conexión en el orden correcto.
    - Conecta Human (Start) -> Basic AI -> Advanced AI.
    - Conecta Advanced AI -> Human (End) si Human (End) está presente.
    """
    connections = []
    for idx in df_score.index:
        # Obtener los valores originales (sin duplicar)
        original_values = df_score.loc[idx, ['Human (Start)', 'Basic AI', 'Advanced AI', 'Human (End)']].dropna()
        
        if len(original_values) >= 2:
            # Conexión entre Human (Start), Basic AI y Advanced AI
            if 'Human (Start)' in original_values and 'Basic AI' in original_values:
                x_values = [x_positions['Human (Start)'], x_positions['Basic AI']]
                y_values = [original_values['Human (Start)'], original_values['Basic AI']]
                connections.append((x_values, y_values))
            
            if 'Basic AI' in original_values and 'Advanced AI' in original_values:
                x_values = [x_positions['Basic AI'], x_positions['Advanced AI']]
                y_values = [original_values['Basic AI'], original_values['Advanced AI']]
                connections.append((x_values, y_values))
            
            # Conexión entre Advanced AI y Human (End)
            if 'Advanced AI' in original_values and 'Human (End)' in original_values:
                x_values = [x_positions['Advanced AI'], x_positions['Human (End)']]
                y_values = [original_values['Advanced AI'], original_values['Human (End)']]
                connections.append((x_values, y_values))
    
    return connections

In [ ]:
def filter_and_analyze_responses(df):
    patterns = [
        r"i\s*(?:do\s*not|don'?t)\s*know",
        r"no\s*(?:lo)?\s*s[ée]",
 
    ]
    
    mask = df['original_answer'].str.lower().str.contains('|'.join(patterns), regex=True, na=False)
    df_filtered = df[~mask].copy()
    removed_counts = (
        df[mask]
        .groupby(['LLM', 'original_answer_id'])
        .size()
        .reset_index(name='removed_count')
    )
    
    class_summary = (
        removed_counts
        .groupby('LLM')
        .agg({
            'original_answer_id': 'nunique',
            'removed_count': 'sum'
        })
        .rename(columns={
            'original_answer_id': 'unique_answers_removed',
            'removed_count': 'total_responses_removed'
        })
    )
    
    class_mapping = {
        0: 'Humano',
        1: 'IA',
        2: 'IA Avanzada'
    }
    
    class_summary.index = class_summary.index.map(class_mapping)
    
    return df_filtered, class_summary, removed_counts

In [ ]:
def compare_evaluations_ploting_equal_n_plotly(df_1, df_2, df_3, name1, name2, name3):
    # Importamos pandas si es necesario
    import pandas as pd
    from plotly.subplots import make_subplots
    import plotly.graph_objects as go
    
    try:
        df_1['original_answer'] = df_1["response"]
    except:
        pass
    
    try:
        df_2['original_answer'] = df_2["response"]
    except:
        pass
    
    try:
        df_3['original_answer'] = df_3["response"]
    except:
        pass
   
    df_1_filtered, summary_1, detailed_counts_1 = filter_and_analyze_responses(df_1)
    df_2_filtered, summary_2, detailed_counts_2 = filter_and_analyze_responses(df_2)
    try:
        df_3_filtered, summary_3, detailed_counts_3 = filter_and_analyze_responses(df_3)
    except: 
        pass
    
    df_agregado_filtrado_1 = df_1_filtered.groupby(['question_id', 'LLM'])[['accuracy_score', 'clarity_score', 'completeness_score']].mean().reset_index()
    df_agregado_filtrado_2 = df_2_filtered.groupby(['question_id', 'LLM'])[['accuracy_score', 'clarity_score', 'completeness_score']].mean().reset_index()
    try:
        df_agregado_filtrado_3 = df_3_filtered.groupby(['question_id', 'LLM'])[['accuracy_score', 'clarity_score', 'completeness_score']].mean().reset_index()
    except:
        pass
    
    # Duplicar las entradas de Human y agregarlas al final
    llm_names = {0: 'Human (Start)', 1: 'Basic AI', 2: 'Advanced AI', 3: 'Human (End)'}
    
    # Para el primer dataframe

    df_human_filtered_1 = df_agregado_filtrado_1[df_agregado_filtrado_1['LLM'] == 0].copy()
    df_human_filtered_1['LLM'] = 3
    df_agregado_filtrado_1 = pd.concat([df_agregado_filtrado_1, df_human_filtered_1])
    df_agregado_filtrado_1['LLM_type'] = df_agregado_filtrado_1['LLM'].map(llm_names)
    
    # Para el segundo dataframe

    
    df_human_filtered_2 = df_agregado_filtrado_2[df_agregado_filtrado_2['LLM'] == 0].copy()
    df_human_filtered_2['LLM'] = 3
    df_agregado_filtrado_2 = pd.concat([df_agregado_filtrado_2, df_human_filtered_2])
    df_agregado_filtrado_2['LLM_type'] = df_agregado_filtrado_2['LLM'].map(llm_names)
    
    # Para el tercer dataframe

    try:
        df_human_filtered_3 = df_agregado_filtrado_3[df_agregado_filtrado_3['LLM'] == 0].copy()
        df_human_filtered_3['LLM'] = 3
        df_agregado_filtrado_3 = pd.concat([df_agregado_filtrado_3, df_human_filtered_3])
        df_agregado_filtrado_3['LLM_type'] = df_agregado_filtrado_3['LLM'].map(llm_names)
    except:    
        pass
    # Identificar las preguntas que tienen TODAS las categorías (las 4 en este caso) para cada dataframe

    preguntas_con_todas_filtrado_1 = df_agregado_filtrado_1.groupby("question_id")["LLM_type"].nunique() == 4
    preguntas_validas_filtrado_1 = preguntas_con_todas_filtrado_1[preguntas_con_todas_filtrado_1].index
    df_filtrado_filtrado_1 = df_agregado_filtrado_1[df_agregado_filtrado_1["question_id"].isin(preguntas_validas_filtrado_1)]
    
    preguntas_con_todas_filtrado_2 = df_agregado_filtrado_2.groupby("question_id")["LLM_type"].nunique() == 4
    preguntas_validas_filtrado_2 = preguntas_con_todas_filtrado_2[preguntas_con_todas_filtrado_2].index
    df_filtrado_filtrado_2 = df_agregado_filtrado_2[df_agregado_filtrado_2["question_id"].isin(preguntas_validas_filtrado_2)]
    try:
        preguntas_con_todas_filtrado_3 = df_agregado_filtrado_3.groupby("question_id")["LLM_type"].nunique() == 4
        preguntas_validas_filtrado_3 = preguntas_con_todas_filtrado_3[preguntas_con_todas_filtrado_3].index
        df_filtrado_filtrado_3 = df_agregado_filtrado_3[df_agregado_filtrado_3["question_id"].isin(preguntas_validas_filtrado_3)]
    except:
        preguntas_con_todas_filtrado_3 = preguntas_con_todas_filtrado_2
        preguntas_validas_filtrado_3 = preguntas_validas_filtrado_2
        df_filtrado_filtrado_3 = df_filtrado_filtrado_2
    scores = ['accuracy_score']
    score_names = {
        'accuracy_score': 'Accuracy',
    }
    
    colors = {
        'Human (Start)': 'rgb(31, 119, 180)',
        'Basic AI': 'rgb(255, 127, 14)',
        'Advanced AI': 'rgb(44, 160, 44)',
        'Human (End)': 'rgb(31, 119, 180)'
    }
    
    # Función para crear datos de conexión
    def create_connection_data(df, x_positions, LLMs):
        connections = []
        for idx in df.index:
            row = df.loc[idx]
            xs = []
            ys = []
            for LLM in llm_types:
                if not pd.isna(row.get(llm_type, pd.NA)):
                    xs.append(x_positions[llm_type])
                    ys.append(row[llm_type])
            if len(xs) > 1:  # Solo añadir si hay más de un punto para conectar
                connections.append((xs, ys))
        return connections
    
    for score in scores:

        df_score_filtrado_1 = df_filtrado_filtrado_1.pivot(index='question_id', columns='LLM_type', values=score)
        df_score_filtrado_2 = df_filtrado_filtrado_2.pivot(index='question_id', columns='LLM_type', values=score)
        try:
            df_score_filtrado_3 = df_filtrado_filtrado_3.pivot(index='question_id', columns='LLM_type', values=score)
        except:
            df_score_filtrado_3 = df_score_filtrado_2

        # Crear subplots con tres columnas
        fig = make_subplots(rows=1, cols=3, subplot_titles=(f"{name1} evaluations", f"{name2} evaluations", f"{name3} evaluations"))
        
        # Definir el orden específico de los tipos y sus posiciones en el eje x
        llm_types = ['Human (Start)', 'Basic AI', 'Advanced AI', 'Human (End)']
        x_positions = {
            'Human (Start)': 0,
            'Basic AI': 1,
            'Advanced AI': 2,
            'Human (End)': 3
        }
        
        # Añadir gráficos para los tres dataframes originales (una columna para cada uno)
        for col, (df_score_filtrado) in enumerate([(df_score_filtrado_1), 
                                                            (df_score_filtrado_2),
                                                            (df_score_filtrado_3)], 1):
            # Gráfico original
            for llm_type in llm_types:
                if llm_type in df_score_filtrado.columns:
                    data_points = df_score_filtrado[llm_type].dropna()
                    # Añadir el número de puntos al nombre para el hover
                    display_name = f"{llm_type} (n={len(data_points)})"
                    
                    # Mostrar en la leyenda solo en el primer gráfico
                    show_legend = (col == 1) and ('Start' in llm_type or 'Human' not in llm_type)
                    
                    fig.add_trace(go.Violin(
                        y=data_points,
                        name='Human' if 'Human' in llm_type else llm_type,
                        marker_color=colors[llm_type],
                        #points='all',
                        jitter=0.3,
                        width=0.5,
                        box_visible=True,
                        meanline_visible=True,
                        #pointpos=2,
                        showlegend=show_legend,
                        x0=x_positions[llm_type],
                        hovertext=[f"{display_name}: {val:.2f}" for val in data_points]
                    ), row=1, col=col)
            
            # Conexiones para el gráfico original
            connections = create_connection_data(df_score_filtrado, x_positions, llm_types)
            for x_values, y_values in connections:
                fig.add_trace(go.Scatter(
                    x=x_values,
                    y=y_values,
                    mode='lines+markers',
                    line=dict(color="rgba(128, 128, 128, 0.3)", width=1, dash='dot'),
                    showlegend=False,
                    hoverinfo='none'
                ), row=1, col=col)
            
            # Añadir anotaciones con el conteo de puntos
            for i, llm_type in enumerate(llm_types):
                if llm_type in df_score_filtrado.columns:
                    count = len(df_score_filtrado[llm_type].dropna())
                    fig.add_annotation(
                        x=i,
                        y=7.5,  # Posición en el eje y (ajustar según sea necesario)
                        text=f"n={count}",
                        showarrow=False,
                        font=dict(size=10),
                        row=1, col=col
                    )
        
        # Layout general
        fig.update_layout(
            title=f"Comparison of {score_names[score]} Scores Across Response Types for AI models and AI evaluations",
            width=2400,  # Aumentar el ancho para acomodar tres gráficos
            height=600,
            template="plotly_white",
            boxmode='group',
            plot_bgcolor='white'
        )
        
        # Configurar los ejes para cada columna
        for i in range(1, 4):
            # Configure el eje y
            fig.update_yaxes(
                title="Score" if i == 1 else None,  # Solo mostrar título en el primer eje y
                range=[0, 8],
                gridcolor='rgba(128, 128, 128, 0.2)',
                zeroline=False,
                row=1, col=i
            )
            
            # Configure el eje x
            fig.update_xaxes(
                ticktext=['Human', 'qwen2:7b', 'Deepseek-r1:7b', 'Human'],
                tickvals=[0, 1, 2, 3],
                gridcolor='rgba(128, 128, 128, 0.2)',
                zeroline=False,
                row=1, col=i
            )
        
        # Añadir un subtítulo con información sobre la cantidad de preguntas para cada conjunto de datos
        fig.add_annotation(
            x=0.5,
            y=-0.15,
            xref="paper",
            yref="paper",
            text=(f"Left: {name1} (n={len(preguntas_validas_filtrado_1)}) | " +
                  f"Center: {name2} (n={len(preguntas_validas_filtrado_2)}) | " +
                  f"Right: {name3} (n={len(preguntas_validas_filtrado_3)})"),
            showarrow=False,
            font=dict(size=12)
        )
        
        fig.show()

In [ ]:
for collec, collec_biased, collec_supervised, name in zip(EVAL_COLLECTION_LIST, EVAL_COLLECTION_LIST_BIASED, EVAL_COLLECTION_LIST_SUPERVISED, EVAL_NAMES):
    compare_evaluations_ploting_equal_n_plotly(load_data(collec), load_data(collec_biased), load_data(collec_supervised), name, f"{name} biased", f"{name} supervised")

In [ ]:

def create_evaluation_heatmap(
    eval_collections, 
    eval_collections_biased, 
    eval_collections_supervised, 
    eval_names,
    score_type='accuracy_score',  # Tipo de puntuación específico a visualizar
    llm_filter=None,
    height=800,
    width=1200,
    colorscale="YlGnBu",
    show_scores=True
):
    """
    Crea un heatmap que visualiza la distribución de puntuaciones (1-7) para el score_type seleccionado.
    
    Args:
        eval_collections: Lista de colecciones de datos normales
        eval_collections_biased: Lista de colecciones de datos sesgados
        eval_collections_supervised: Lista de colecciones de datos supervisados
        eval_names: Lista de nombres de los modelos
        score_type: Tipo de puntuación a visualizar (ej: 'accuracy_score')
        llm_filter: IDs de LLM a incluir, None para todos
        height: Altura de la figura
        width: Ancho de la figura
        colorscale: Escala de colores para el heatmap
        show_scores: Si es True, muestra los valores numéricos en las celdas
    """
    import pandas as pd
    import plotly.graph_objects as go
    import numpy as np
    
    # Definir nombres
    llm_names = {0: 'Human', 1: 'Basic AI', 2: 'Advanced AI'}
    score_titles = {
        'accuracy_score': 'Accuracy',
        'clarity_score': 'Clarity',
        'completeness_score': 'Completeness',
        'conciseness_score': 'Conciseness',
        'coherence_score': 'Coherence',
        'helpfulness_score': 'Helpfulness',
        'correctness_score': 'Correctness'
    }
    
    if llm_filter is None:
        llm_filter = list(llm_names.keys())
    
    # Preparar datos procesados
    all_data = []
    
    # Función auxiliar para cargar datos con control de errores
    def safe_load_data(collection, model_name, eval_type, model_idx):
        try:
            if collection is None:
                print(f"Advertencia: Colección para {model_name} ({eval_type}) no está disponible.")
                return None
            
            # Cargar datos
            df = load_data(collection)
            
            # Preparar datos agregados
            df_agregado = df
            df_agregado['LLM_type'] = df_agregado['LLM'].map(llm_names)
            df_agregado['eval_type'] = eval_type
            df_agregado['model_name'] = model_name
            df_agregado['model_idx'] = model_idx
            
            # Filtrar LLMs si es necesario
            df_agregado = df_agregado[df_agregado['LLM'].isin(llm_filter)]
            
            if len(df_agregado) > 0:
                return df_agregado
            else:
                print(f"Advertencia: No hay datos después del filtrado para {model_name} ({eval_type}).")
                return None
        except Exception as e:
            print(f"Error al cargar datos para {model_name} ({eval_type}): {str(e)}")
            return None
    
    # Procesar todos los datos con control de errores
    for eval_type, collections in [
        ("normal", eval_collections),
        ("biased", eval_collections_biased),
        ("supervised", eval_collections_supervised)
    ]:
        if collections is not None:
            for model_idx, (collection, name) in enumerate(zip(collections, eval_names)):
                try:
                    df_agregado = safe_load_data(collection, name, eval_type, model_idx)
                    if df_agregado is not None:
                        all_data.append(df_agregado)
                except Exception as e:
                    print(f"Error al procesar {name} ({eval_type}): {str(e)}")
    
    # Verificar si tenemos datos disponibles
    if not all_data:
        print("Error: No hay datos disponibles para visualizar.")
        return None
    
    # Combinar todos los datos
    combined_df = pd.concat(all_data)
    
    # Verificar si el score_type está disponible
    if score_type not in combined_df.columns:
        print(f"Error: El tipo de puntuación '{score_type}' no está disponible en los datos.")
        return None
    
    # Crear un DataFrame para almacenar las frecuencias de cada puntuación
    score_values = list(range(1, 8))  # Puntuaciones del 1 al 7
    heatmap_data = []
    
    # Calcular distribución de puntuaciones para cada combinación
    for model_name in eval_names:
        for eval_type in ["normal", "biased", "supervised"]:
            for llm_id in llm_filter:
                subset = combined_df[
                    (combined_df['model_name'] == model_name) & 
                    (combined_df['eval_type'] == eval_type) & 
                    (combined_df['LLM'] == llm_id)
                ]
                
                if len(subset) > 0:
                    # Crear nombre de fila para el heatmap
                    row_name = f"{model_name} ({eval_type.capitalize()} - {llm_names[llm_id]})"
                    
                    # Obtener las puntuaciones
                    scores = subset[score_type].values
                    
                    # Calcular frecuencia para cada valor de puntuación
                    for score_val in score_values:
                        count = np.sum(scores == score_val)
                        percentage = (count / len(scores)) * 100 if len(scores) > 0 else 0
                        
                        heatmap_data.append({
                            'row_name': row_name,
                            'model': model_name,
                            'eval_type': eval_type,
                            'llm_type': llm_names[llm_id],
                            'score_value': score_val,
                            'count': count,
                            'percentage': percentage
                        })
    
    if not heatmap_data:
        print("Error: No hay suficientes datos para crear el heatmap.")
        return None
    
    # Crear DataFrame para el heatmap
    heatmap_df = pd.DataFrame(heatmap_data)
    
    # Pivotar el DataFrame para formato de heatmap
    pivot_df = heatmap_df.pivot_table(
        index='row_name', 
        columns='score_value', 
        values='percentage',
        fill_value=0
    )
    
    # Ordenar para mejor visualización (opcional)
    model_order = []
    for model in eval_names:
        for eval_type in ["normal", "biased", "supervised"]:
            for llm_id in llm_filter:
                row_name = f"{model} ({eval_type.capitalize()} - {llm_names[llm_id]})"
                if row_name in pivot_df.index:
                    model_order.append(row_name)
    
    if model_order:
        pivot_df = pivot_df.reindex(model_order)
    
    # Preparar datos para Plotly
    y_labels = pivot_df.index.tolist()
    x_labels = [str(i) for i in score_values]
    z_values = pivot_df.values
    
    # Crear figura
    fig = go.Figure(data=go.Heatmap(
        z=z_values,
        x=x_labels,
        y=y_labels,
        colorscale=colorscale,
        zmin=0,
        zmax=100,
        colorbar=dict(title="Percentage (%)"),
    ))
    
    # Crear anotaciones (valores en las celdas)
    annotations = []
    if show_scores:
        for i in range(len(y_labels)):
            for j in range(len(x_labels)):
                value = z_values[i, j]
                if value > 0:  # Solo mostrar valores mayores que cero
                    annotations.append(
                        dict(
                            text=f"{value:.1f}%",
                            x=j,  # La posición correcta es j, no x_labels[j]
                            y=i,  # La posición correcta es i, no y_labels[i]
                            xref='x',
                            yref='y',
                            font=dict(color='white' if value > 50 else 'black'),
                            showarrow=False
                        )
                    )
    
    # Agregar las anotaciones a la figura
    fig.update_layout(annotations=annotations)
    
    # Actualizar layout
    fig.update_layout(
        title=f"Distribution of {score_titles.get(score_type, score_type)} Scores",
        height=height,
        width=width,
        xaxis=dict(title="Score Value (1-7)", tickmode='linear', dtick=1, tickvals=list(range(len(x_labels))), ticktext=x_labels),
        yaxis=dict(title="Model / Condition / LLM", autorange="reversed", tickvals=list(range(len(y_labels))), ticktext=y_labels),
        template="plotly_white"
    )
    
    # Permitir que la altura se ajuste automáticamente según el número de filas
    if len(y_labels) > 10:
        adjusted_height = height + (len(y_labels) - 10) * 30
        fig.update_layout(height=adjusted_height)
    
    return fig

In [ ]:
def compare_evaluations_visualization(
    eval_collections, 
    eval_collections_supervised,
    eval_names,
    score_type='accuracy_score',
    llm_filter=None,
    height=800,
    width=1900,
    statistical_test='mann-whitney',  # 'mann-whitney' o 't-test'
    pvalue_correction='fdr_bh'  # 'bonferroni', 'holm', 'fdr_bh', None
):
    """
    Función para crear visualizaciones comparativas estilo paper con análisis estadístico.
    
    Args:
        eval_collections: Lista de colecciones de datos normales
        eval_collections_supervised: Lista de colecciones de datos supervisados
        eval_names: Lista de nombres de los modelos
        score_type: Tipo de puntuación a visualizar
        llm_filter: IDs de LLM a incluir, None para todos
        height: Altura de la figura
        width: Ancho de la figura
        statistical_test: Tipo de test estadístico ('mann-whitney' o 't-test')
        pvalue_correction: Método de corrección de p-values ('bonferroni', 'holm', 'fdr_bh', None)
    """
    import pandas as pd
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    import numpy as np
    from scipy import stats
    from statsmodels.stats.multitest import multipletests
    
    # Definir nombres y colores
    llm_names = {0: 'Human', 1: 'Basic AI', 2: 'Advanced AI'}
    
    if llm_filter is None:
        llm_filter = list(llm_names.keys())
    
    # Colores por tipo de LLM
    llm_colors = {
        0: 'rgb(31, 119, 180)',    # Human - azul
        1: 'rgb(255, 127, 14)',     # Basic AI - naranja
        2: 'rgb(44, 160, 44)'       # Advanced AI - verde
    }
    
    # Colores con transparencia para las categorías
    category_styles = {
        'normal': {'opacity': 0.7, 'dash': 'solid'},
        'supervised': {'opacity': 0.5, 'dash': 'dot'}
    }
    
    score_titles = {
        'accuracy_score': 'Accuracy',
        'clarity_score': 'Clarity',
        'completeness_score': 'Completeness',
        'conciseness_score': 'Conciseness',
        'coherence_score': 'Coherence',
        'helpfulness_score': 'Helpfulness',
        'correctness_score': 'Correctness'
    }
    
    def calculate_pvalue(data1, data2, test_type='mann-whitney'):
        """Calcula el p-value entre dos distribuciones"""
        if len(data1) < 2 or len(data2) < 2:
            return np.nan
        
        try:
            if test_type == 'mann-whitney':
                statistic, pvalue = stats.mannwhitneyu(data1, data2, alternative='two-sided')
            else:  # t-test
                statistic, pvalue = stats.ttest_ind(data1, data2)
            return pvalue
        except:
            return np.nan
    
    def format_pvalue(p, is_adjusted=False):
        """Formatea el p-value para mostrar"""
        prefix = "adj." if is_adjusted else ""
        if np.isnan(p):
            return "N/A"
        elif p < 0.001:
            return f"{prefix}p<0.001***"
        elif p < 0.01:
            return f"{prefix}p={p:.3f}**"
        elif p < 0.05:
            return f"{prefix}p={p:.3f}*"
        else:
            return f"{prefix}p={p:.3f}"
    
    # Preparar datos procesados
    all_data = []
    available_models = []
    
    # Función auxiliar para cargar datos
    def safe_load_data(collection, model_name, eval_type):
        try:
            if collection is None:
                return None
            
            df = load_data(collection)
            df['LLM_type'] = df['LLM'].map(llm_names)
            df['eval_type'] = eval_type
            df['model_name'] = model_name
            df = df[df['LLM'].isin(llm_filter)]
            
            if len(df) > 0:
                if model_name not in available_models:
                    available_models.append(model_name)
                return df
            return None
        except Exception as e:
            print(f"Error loading data for {model_name} ({eval_type}): {str(e)}")
            return None
    
    # Procesar datos normales y supervised
    for eval_type, collections in [
        ("normal", eval_collections),
        ("supervised", eval_collections_supervised)
    ]:
        if collections is not None:
            for collection, name in zip(collections, eval_names):
                df_agregado = safe_load_data(collection, name, eval_type)
                if df_agregado is not None:
                    all_data.append(df_agregado)
    
    if not all_data:
        print("Error: No data available to visualize.")
        return
    
    all_data_1 = []
    for df in all_data:
        try:  
            df['original_answer'] = df["response"]
        except:
            pass

        df, summary_1, detailed_counts_1 = filter_and_analyze_responses(df)
        all_data_1.append(df)
        
    all_data = all_data_1
    combined_df = pd.concat(all_data)
    
    # Configurar subplots por modelo
    n_models = len(available_models)
    if n_models == 0:
        print("Error: No models with available data.")
        return
    
    # Calcular layout (3 columnas máximo)
    n_cols = min(3, n_models)
    n_rows = (n_models + n_cols - 1) // n_cols
    
    fig = make_subplots(
        rows=n_rows, 
        cols=n_cols,
        subplot_titles=available_models,
        vertical_spacing=0.20,
        horizontal_spacing=0.12
    )
    
    # Posiciones x para los violines
    x_positions = {0: 0, 1: 1, 2: 2}  # Human, Basic AI, Advanced AI
    
    # Almacenar estadísticas para cada modelo
    all_statistics = {}
    all_pvalues = []  # Para ajuste global
    
    for model_idx, name in enumerate(available_models):
        row = model_idx // n_cols + 1
        col = model_idx % n_cols + 1
        
        model_data = combined_df[combined_df['model_name'] == name]
        
        # Almacenar datos para tests estadísticos
        model_statistics = {}
        
        # Para cada tipo de evaluación (normal y supervised)
        for eval_type in ['normal', 'supervised']:
            eval_data = model_data[model_data['eval_type'] == eval_type]
            
            if len(eval_data) == 0:
                continue
            
            # Agregar violines para cada LLM
            for llm_id in llm_filter:
                llm_data = eval_data[eval_data['LLM'] == llm_id][score_type].dropna()
                
                if len(llm_data) > 0:
                    # Guardar datos para análisis estadístico
                    if llm_id not in model_statistics:
                        model_statistics[llm_id] = {}
                    model_statistics[llm_id][eval_type] = llm_data.values
                    
                    trace_name = f"{llm_names[llm_id]} - {eval_type.capitalize()}"
                    
                    # Ajustar posición x según el tipo de evaluación
                    x_offset = -0.2 if eval_type == 'normal' else 0.2
                    
                    fig.add_trace(go.Violin(
                        y=llm_data,
                        name=trace_name,
                        x0=x_positions[llm_id] + x_offset,
                        marker_color=llm_colors[llm_id],
                        opacity=category_styles[eval_type]['opacity'],
                        box_visible=True,
                        meanline_visible=True,
                        width=0.35,
                        showlegend=(model_idx == 0),
                        legendgroup=f"{llm_id}-{eval_type}",
                        line=dict(width=2),
                        scalemode='width'
                    ), row=row, col=col)
        
        # Calcular p-values comparando LLMs dentro de cada categoría
        pvalue_annotations = []
        
        # Comparaciones dentro de cada tipo de evaluación
        for eval_type in ['normal', 'supervised']:
            comparisons = [
                (0, 1, "H-BA"),
                (0, 2, "H-AA"),
                (1, 2, "BA-AA")
            ]
            
            for llm1, llm2, comparison_name in comparisons:
                if llm1 in model_statistics and llm2 in model_statistics:
                    if eval_type in model_statistics[llm1] and eval_type in model_statistics[llm2]:
                        data1 = model_statistics[llm1][eval_type]
                        data2 = model_statistics[llm2][eval_type]
                        pval = calculate_pvalue(data1, data2, statistical_test)
                        
                        if not np.isnan(pval):
                            pvalue_info = {
                                'comparison': comparison_name,
                                'eval_type': eval_type,
                                'pvalue': pval,
                                'model': name
                            }
                            pvalue_annotations.append(pvalue_info)
                            all_pvalues.append(pvalue_info)
        
        all_statistics[name] = {
            'data': model_statistics,
            'pvalues': pvalue_annotations
        }
    
    # Ajustar p-values si se especifica corrección
    if pvalue_correction and len(all_pvalues) > 0:
        pvals_array = np.array([p['pvalue'] for p in all_pvalues])
        
        # Aplicar corrección
        try:
            reject, pvals_corrected, _, _ = multipletests(
                pvals_array, 
                alpha=0.05, 
                method=pvalue_correction
            )
            
            # Actualizar p-values ajustados
            for i, pval_info in enumerate(all_pvalues):
                pval_info['pvalue_adjusted'] = pvals_corrected[i]
                pval_info['reject_null'] = reject[i]
            
            print(f"\nP-value adjustment using {pvalue_correction}:")
            print(f"Total comparisons: {len(all_pvalues)}")
            print(f"Significant after correction: {sum(reject)}")
            
        except Exception as e:
            print(f"Error in p-value correction: {e}")
            # Si falla la corrección, usar p-values sin ajustar
            for pval_info in all_pvalues:
                pval_info['pvalue_adjusted'] = pval_info['pvalue']
                pval_info['reject_null'] = pval_info['pvalue'] < 0.05
    else:
        # Sin corrección
        for pval_info in all_pvalues:
            pval_info['pvalue_adjusted'] = pval_info['pvalue']
            pval_info['reject_null'] = pval_info['pvalue'] < 0.05
    
    # Añadir anotaciones con p-values ajustados
    for model_idx, name in enumerate(available_models):
        row = model_idx // n_cols + 1
        col = model_idx % n_cols + 1
        
        # Filtrar p-values para este modelo
        model_pvalues = [p for p in all_pvalues if p['model'] == name]
        
        if model_pvalues:
            # Crear texto compacto con p-values
            annotation_text = "<b>Tests:</b><br>"
            
            # Agrupar por tipo de evaluación
            for eval_type in ['normal', 'supervised']:
                eval_pvals = [p for p in model_pvalues if p['eval_type'] == eval_type]
                if eval_pvals:
                    annotation_text += f"<i>{eval_type[:4].upper()}:</i><br>"
                    for p in eval_pvals:
                        formatted = format_pvalue(p['pvalue_adjusted'], is_adjusted=bool(pvalue_correction))
                        annotation_text += f"{p['comparison']}: {formatted}<br>"
            
            # Determinar referencias correctas para el subplot
            if n_rows == 1 and n_cols == 1:
                xref = "x domain"
                yref = "y domain"
            elif n_rows == 1:
                xref = f"x{model_idx+1} domain" if model_idx > 0 else "x domain"
                yref = f"y{model_idx+1} domain" if model_idx > 0 else "y domain"
            else:
                subplot_num = (row - 1) * n_cols + col
                xref = f"x{subplot_num} domain" if subplot_num > 1 else "x domain"
                yref = f"y{subplot_num} domain" if subplot_num > 1 else "y domain"
            
            # Posicionar AL LADO del gráfico
            fig.add_annotation(
                x=1.12,
                y=0.5,
                xref=xref,
                yref=yref,
                text=annotation_text,
                showarrow=False,
                font=dict(size=12, family="Arial", color="black"),
                align="left",
                xanchor="left",
                yanchor="middle",
                bgcolor="rgba(255, 255, 255, 0.9)",
                bordercolor="rgba(0, 0, 0, 0.3)",
                borderwidth=1,
                borderpad=6
            )
    
    # Añadir nota sobre corrección de p-values
    correction_note = ""
    if pvalue_correction:
        correction_names = {
            'bonferroni': 'Bonferroni',
            'holm': 'Holm-Bonferroni',
            'fdr_bh': 'Benjamini-Hochberg (FDR)'
        }
        correction_note = f"P-values adjusted using {correction_names.get(pvalue_correction, pvalue_correction)} correction"
    
    # Layout estilo paper
    title_text = f"Comparison of {score_titles[score_type]} Scores by Evaluator Type"
    if correction_note:
        title_text += f"<br><sub>{correction_note}</sub>"
    
    fig.update_layout(
        title=dict(
            text=title_text,
            font=dict(size=24, family="Arial", color="black"),
            y=0.98,
            x=0.5,
            xanchor='center',
            yanchor='top'
        ),
        font=dict(size=18, family="Arial"),
        height=height,
        width=width,
        template="plotly_white",
        plot_bgcolor='white',
        paper_bgcolor='white',
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.05,
            xanchor="center",
            x=0.5,
            font=dict(size=16, family="Arial"),
            bgcolor="rgba(255,255,255,0.8)",
            bordercolor="rgba(0,0,0,0.2)",
            borderwidth=1
        ),
        margin=dict(t=180, b=80, l=80, r=180)
    )
    
    # Configurar ejes para cada subplot
    for i in range(1, n_models + 1):
        row = (i - 1) // n_cols + 1
        col = (i - 1) % n_cols + 1
        
        if n_rows == 1 and n_cols == 1:
            yaxis_key = 'yaxis'
            xaxis_key = 'xaxis'
        elif n_rows == 1:
            yaxis_key = f'yaxis{i}' if i > 1 else 'yaxis'
            xaxis_key = f'xaxis{i}' if i > 1 else 'xaxis'
        else:
            subplot_num = (row - 1) * n_cols + col
            yaxis_key = f'yaxis{subplot_num}' if subplot_num > 1 else 'yaxis'
            xaxis_key = f'xaxis{subplot_num}' if subplot_num > 1 else 'xaxis'
        
        fig.layout[yaxis_key].update(
            title=dict(
                text="Score" if col == 1 else "",
                font=dict(size=20, family="Arial")
            ),
            range=[0, 8],
            gridcolor='rgba(128, 128, 128, 0.2)',
            zeroline=False,
            tickfont=dict(size=18, family="Arial"),
            showline=True,
            linewidth=1,
            linecolor='black',
            mirror=False
        )
        
        fig.layout[xaxis_key].update(
            ticktext=['Human', 'AI', 'CoT AI'],
            tickvals=[0, 1, 2],
            gridcolor='rgba(128, 128, 128, 0.2)',
            zeroline=False,
            tickfont=dict(size=18, family="Arial"),
            showline=True,
            linewidth=1,
            linecolor='black',
            mirror=False
        )
    
    # Actualizar títulos de subplots con estilo paper
    for i, annotation in enumerate(fig.layout.annotations):
        if i < n_models:
            annotation.font.size = 20
            annotation.font.family = "Arial"
            annotation.font.color = "black"
    
    fig.show()
    
    # Retornar estadísticas incluyendo p-values ajustados
    return all_statistics, all_pvalues

In [ ]:
compare_evaluations_visualization(
    EVAL_COLLECTION_LIST, 
 
    EVAL_COLLECTION_LIST_SUPERVISED, 
    EVAL_NAMES,
   
  
    width=2000, height=900
)



In [ ]:
compare_evaluations_visualization(
    EVAL_COLLECTION_LIST, 
    EVAL_COLLECTION_LIST_BIASED, 
    EVAL_COLLECTION_LIST_SUPERVISED, 
    EVAL_NAMES,
    plot_type='histogram',
    organization='by_category')

In [ ]:
def create_clustered_evaluation_heatmap(
    eval_collections, 
    eval_collections_biased, 
    eval_collections_supervised, 
    eval_names,
    score_type='accuracy_score',
    llm_filter=None,
    height=900,
    width=1400,
    colorscale="Tempo",
    show_scores=True,
    n_clusters=None,  # Number of clusters (automatic if None)
    idk=False,
):
    """
    Creates a heatmap with clustering of conditions (rows) based on score distribution,
    adds color bars to identify models, and includes a horizontal dendrogram showing the hierarchical clustering.
    
    Args:
        eval_collections: List of normal data collections
        eval_collections_biased: List of biased data collections
        eval_collections_supervised: List of supervised data collections
        eval_names: List of model names
        score_type: Type of score to visualize (e.g., 'accuracy_score')
        llm_filter: LLM IDs to include, None for all
        height: Figure height
        width: Figure width
        colorscale: Color scale for the heatmap
        show_scores: If True, shows numeric values in cells
        n_clusters: Number of clusters to form (if None, determined automatically)
    """
    import pandas as pd
    import plotly.graph_objects as go
    import numpy as np
    from sklearn.cluster import AgglomerativeClustering
    from sklearn.metrics import silhouette_score
    import plotly.express as px
    import colorsys
    from scipy.cluster.hierarchy import linkage, dendrogram
    from plotly.subplots import make_subplots
    
    # Define names
    llm_names = {0: 'Human', 1: 'Basic AI', 2: 'Advanced AI'}
    score_titles = {
        'accuracy_score': 'Accuracy',
        'clarity_score': 'Clarity',
        'completeness_score': 'Completeness',
    }
    
    if llm_filter is None:
        llm_filter = list(llm_names.keys())
    
    # Prepare processed data
    all_data = []
    
    # Helper function to load data with error control
    def safe_load_data(collection, model_name, eval_type, model_idx):
        try:
            if collection is None:
                print(f"Warning: Collection for {model_name} ({eval_type}) is not available.")
                return None
            
            # Load data
            df = load_data(collection)
            
            # Prepare aggregated data
            df_agregado = df
            df_agregado['LLM_type'] = df_agregado['LLM'].map(llm_names)
            df_agregado['eval_type'] = eval_type
            df_agregado['model_name'] = model_name
            df_agregado['model_idx'] = model_idx
            
            # Filter LLMs if necessary
            df_agregado = df_agregado[df_agregado['LLM'].isin(llm_filter)]
            
            if len(df_agregado) > 0:
                return df_agregado
            else:
                print(f"Warning: No data after filtering for {model_name} ({eval_type}).")
                return None
        except Exception as e:
            print(f"Error loading data for {model_name} ({eval_type}): {str(e)}")
            return None
    
    # Process all data with error control
    for eval_type, collections in [
        ("normal", eval_collections),
        ("biased", eval_collections_biased),
        ("supervised", eval_collections_supervised)
    ]:
        if collections is not None:
            for model_idx, (collection, name) in enumerate(zip(collections, eval_names)):
                try:
                    df_agregado = safe_load_data(collection, name, eval_type, model_idx)
                    if df_agregado is not None:
                            
                        # Filter idk
                        if idk == False:
                            try:
                                df_agregado['original_answer'] = df_agregado["response"]
                            except Exception as e:
                                print(e)

                            df_agregado, summary, detailed_counts = filter_and_analyze_responses(df_agregado)
                            
                        all_data.append(df_agregado)
                except Exception as e:
                    print(f"Error processing {name} ({eval_type}): {str(e)}")
    
    # Check if we have available data
    if not all_data:
        print("Error: No data available to visualize.")
        return None
    
    # Combine all data
    combined_df = pd.concat(all_data)
    
    # Check if score_type is available
    if score_type not in combined_df.columns:
        print(f"Error: Score type '{score_type}' is not available in the data.")
        return None
    
    # Create a DataFrame to store frequency of each score
    score_values = list(range(1, 8))  # Scores from 1 to 7
    heatmap_data = []
    
    # Generate colors for each model and add legend traces
    model_colors = {}
    legend_traces = []
    
    for i, model in enumerate(eval_names):
        # Generate equally spaced HSL colors
        hue = i / len(eval_names)
        # Convert HSL to RGB
        rgb = colorsys.hls_to_rgb(hue, 0.65, 0.6)
        # Convert to hexadecimal format
        hex_color = f'#{int(rgb[0]*255):02x}{int(rgb[1]*255):02x}{int(rgb[2]*255):02x}'
        model_colors[model] = hex_color
        
        # Create legend trace for this model
        legend_traces.append(
            go.Scatter(
                x=[None],
                y=[None],
                mode="markers",
                marker=dict(size=15, color=hex_color),
                name=model,
                showlegend=True
            )
        )
    
    # Calculate score distribution for each combination
    for model_name in eval_names:
        for eval_type in ["normal", "biased", "supervised"]:
            for llm_id in llm_filter:
                subset = combined_df[
                    (combined_df['model_name'] == model_name) & 
                    (combined_df['eval_type'] == eval_type) & 
                    (combined_df['LLM'] == llm_id)
                ]
                
                if len(subset) > 0:
                    # Create row name for heatmap
                    row_name = f"{model_name} ({eval_type.capitalize()} - {llm_names[llm_id]})"
                    
                    # Get scores
                    scores = subset[score_type].values
                    
                    # Calculate frequency for each score value
                    for score_val in score_values:
                        count = np.sum(scores == score_val)
                        percentage = (count / len(scores)) * 100 if len(scores) > 0 else 0
                        
                        heatmap_data.append({
                            'row_name': row_name,
                            'model': model_name,
                            'eval_type': eval_type,
                            'llm_type': llm_names[llm_id],
                            'score_value': score_val,
                            'count': count,
                            'percentage': percentage
                        })
    
    if not heatmap_data:
        print("Error: Not enough data to create the heatmap.")
        return None
    
    # Create DataFrame for heatmap
    heatmap_df = pd.DataFrame(heatmap_data)
    
    # Pivot DataFrame for heatmap format
    pivot_df = heatmap_df.pivot_table(
        index='row_name', 
        columns='score_value', 
        values='percentage',
        fill_value=0
    )
    
    # Extract metadata
    row_metadata = {}
    for _, row in heatmap_df.drop_duplicates(['row_name', 'model']).iterrows():
        row_metadata[row['row_name']] = {
            'model': row['model'],
            'eval_type': row['eval_type'],
            'llm_type': row['llm_type']
        }
    
    # Compute the linkage matrix using ward
    Z = linkage(pivot_df.values, method='ward')
    
    # Perform clustering
    # First determine optimal number of clusters if not specified
    if n_clusters is None:
        max_clusters = min(10, len(pivot_df))  # Limit maximum number of clusters
        best_score = -1
        best_n_clusters = 2  # Default, at least 2 clusters
        
        for i in range(2, max_clusters + 1):
            try:
                clustering = AgglomerativeClustering(n_clusters=i, linkage='ward')
                labels = clustering.fit_predict(pivot_df.values)
                
                if len(set(labels)) > 1:  # Ensure there are at least 2 different clusters
                    silhouette = silhouette_score(pivot_df.values, labels)
                    if silhouette > best_score:
                        best_score = silhouette
                        best_n_clusters = i
            except Exception as e:
                print(f"Error calculating silhouette score for {i} clusters: {str(e)}")
                continue
        
        n_clusters = best_n_clusters
        print(f"Optimal number of clusters determined: {n_clusters}")
    
    # Apply clustering
    clustering = AgglomerativeClustering(n_clusters=n_clusters, linkage='ward')
    cluster_labels = clustering.fit_predict(pivot_df.values)
    
    # Create dendrogram data - specifically using left orientation
    dendro = dendrogram(Z, no_plot=True, orientation='left')
    
    # Get the order of rows according to the dendrogram
    dendro_leaves = dendro['leaves']
    
    # Reorder rows according to dendrogram
    pivot_df_reordered = pivot_df.iloc[dendro_leaves]
    
    # Reorder clusters according to dendrogram
    cluster_labels_reordered = np.zeros_like(cluster_labels)
    for i, idx in enumerate(dendro_leaves):
        cluster_labels_reordered[i] = cluster_labels[idx]
    
    # Prepare data for Plotly
    y_labels = pivot_df_reordered.index.tolist()
    x_labels = [str(i) for i in score_values]
    z_values = pivot_df_reordered.values
    
    # Create figure with 2 columns: heatmap and dendrogram
    # Note: We're using column_widths to give more space to the heatmap
    fig = make_subplots(
        rows=1, 
        cols=2,
        column_widths=[0.75, 0.25],  # 75% for heatmap, 25% for dendrogram
        subplot_titles=[f"Distribution of {score_titles.get(score_type, score_type)} Scores", "Hierarchical Clustering"],
        horizontal_spacing=0.02,
    )
    
    # Add heatmap to the first column
    fig.add_trace(
        go.Heatmap(
            z=z_values,
            x=x_labels,
            y=y_labels,
            colorscale=colorscale,
            zmin=0,
            zmax=100,
            showscale=False,  # Remove colorbar
        ),
        row=1, col=1
    )
    
    # Add dendrogram to the second column
    # For a horizontal dendrogram, we swap x and y coordinates
    # and make sure they align with our row labels
    
    # Extract dendrogram coordinates
    dendro_x = dendro['icoord']
    dendro_y = dendro['dcoord']
    
    # The dendrogram data from scipy needs to be transformed for Plotly
    # For 'left' orientation, we need to draw each segment of the dendrogram separately
    for i in range(len(dendro_x)):
        # For horizontal orientation with left direction
        # Map y coordinates to row indices
        y_coords = []
        for j in range(len(dendro_x[i])):
            # The original coordinates need to be mapped to row indices
            # Since we're using 'left' orientation, the mapping is more direct
            y_val = dendro_x[i][j]
            # Scale to match the number of rows
            y_coords.append(y_val)
        
        # x coordinates map directly for 'left' orientation
        x_coords = [x for x in dendro_y[i]]  # Negate to point toward heatmap
        
        fig.add_trace(
            go.Scatter(
                x=x_coords,
                y=y_coords,
                mode='lines',
                line=dict(color='black', width=2),
                hoverinfo='skip',
                showlegend=False
            ),
            row=1, col=2
        )
    
    # Create annotations (cell values)
    annotations = []
    if show_scores:
        for i in range(len(y_labels)):
            for j in range(len(x_labels)):
                value = z_values[i, j]
                if value > 0:  # Only show values greater than zero
                    annotations.append(
                        dict(
                            text=f"{value:.1f}",
                            x=j,
                            y=i,
                            xref='x',
                            yref='y',
                            font=dict(
                                color='white' if value > 50 else 'black',
                                size=20,  # Increased font size for values
                                family="Arial, sans-serif"
                            ),
                            showarrow=False
                        )
                    )
    
    # Add color bars to identify models
    for i, y_label in enumerate(y_labels):
        model = row_metadata[y_label]['model']
        color = model_colors[model]
        
        # Add color rectangle to the left of the heatmap
        fig.add_shape(
            type="rect",
            x0=-0.95,
            y0=i - 0.5,
            x1=-0.5,
            y1=i + 0.5,
            xref='x',
            yref='y',
            fillcolor=color,
            line=dict(width=0),
            layer="below"
        )
    
    # Add dividing lines between clusters
    current_cluster = cluster_labels_reordered[0]
    for i in range(1, len(cluster_labels_reordered)):
        if cluster_labels_reordered[i] != current_cluster:
            # Add dividing line
            fig.add_shape(
                type="line",
                x0=-0.95,  # Extend line to include color bars
                y0=i - 0.5,
                x1=len(x_labels) - 0.5,
                xref='x',
                yref='y',
                line=dict(color="black", width=2, dash="dash"),
                layer="below"
            )
            current_cluster = cluster_labels_reordered[i]
    
    # Add cluster annotations
    for cluster_id in range(n_clusters):
        # Find first occurrence of this cluster in the reordered labels
        try:
            sample_idx = np.where(cluster_labels_reordered == cluster_id)[0][0]
            # Count rows in this cluster
            cluster_size = sum(cluster_labels_reordered == cluster_id)
            
            # Add annotation to the right of the heatmap
            annotations.append(
                dict(
                    x=len(x_labels) + 0.5,
                    y=sample_idx,
                    xref='x',
                    yref='y',
                    text=f"Cluster {cluster_id + 1}<br>({cluster_size} items)",
                    showarrow=True,
                    arrowhead=2,
                    ax=20,
                    font=dict(
                        size=18,  # Increased font size for cluster labels
                        family="Arial, sans-serif"
                    )
                )
            )
        except:
            print(f"Warning: Cluster {cluster_id} not found in reordered labels.")
    
    # Add legend traces to figure
    for trace in legend_traces:
        fig.add_trace(trace, row=1, col=1)
    
    # Add the annotations to the figure
    fig.update_layout(annotations=annotations)
    
    # Update layout with larger fonts throughout
    fig.update_layout(
        title=dict(
            text=f"Clustered Distribution of {score_titles.get(score_type, score_type)} Scores with Dendrogram",
            font=dict(size=28, family="Arial, sans-serif")  # Larger title font
        ),
        height=height,
        width=width,
        template="plotly_white",
        paper_bgcolor='white',
        plot_bgcolor='white',
        legend=dict(
            orientation="v",
            yanchor="top",
            y=1.0,
            xanchor="right",
            x=1.2,
            font=dict(size=20, family="Arial, sans-serif"),  # Larger legend font
            title=dict(
                text="Source",
                font=dict(size=24, family="Arial, sans-serif")
            )
        )
    )
    
    # Update heatmap axis properties
    fig.update_xaxes(
        title=dict(
            text="Score Value",
            font=dict(size=30, family="Arial, sans-serif")  # Larger axis title font
        ),
        tickmode='linear',
        dtick=1,
        tickvals=list(range(len(x_labels))),
        ticktext=x_labels,
        range=[-0.95, len(x_labels) - 0.5],  # Extend range to show color bars
        tickfont=dict(size=26, family="Arial, sans-serif"),  # Larger tick labels
        row=1, col=1
    )
    
    fig.update_yaxes(
        title=dict(
            text="Evaluator / Condition / Source",
            font=dict(size=30, family="Arial, sans-serif")  # Larger axis title font
        ),
        #autorange="reversed",
        tickvals=list(range(len(y_labels))),
        ticktext=y_labels,
        tickfont=dict(size=24, family="Arial, sans-serif"),  # Larger tick labels
        row=1, col=1
    )
    
    # Update dendrogram axis properties
    fig.update_xaxes(
        showticklabels=False,
        showgrid=False,
        zeroline=False,
        range=[0, 1.2],  # Set range to control the width of the dendrogram
        row=1, col=2
    )
    
    fig.update_yaxes(
        showticklabels=False,
        showgrid=False,
        zeroline=False,
        matches='y',  # Match the y-axis of the heatmap
        row=1, col=2
    )
    
    # Update subplot titles font
    for i in range(len(fig.layout.annotations)):
        if i < 2:  # Just the first two annotations are subplot titles
            fig.layout.annotations[i].font = dict(size=28, family="Arial, sans-serif")
    
    # Adjust margins to accommodate the dendrogram and heatmap
    fig.update_layout(
        margin=dict(l=100, r=100, t=100, b=100)
    )
    
    return fig

In [ ]:
fig = create_clustered_evaluation_heatmap(
    EVAL_COLLECTION_LIST, 
    EVAL_COLLECTION_LIST_BIASED, 
    EVAL_COLLECTION_LIST_SUPERVISED, 
    EVAL_NAMES,
    score_type='accuracy_score',
    idk = True,
)
fig.show()

In [ ]:
fig = create_clustered_evaluation_heatmap(
    EVAL_COLLECTION_LIST, 
    None, 
    None, 
    EVAL_NAMES,
    score_type='accuracy_score',
    idk = True,
    colorscale="Tempo",
)
fig.show()

In [ ]:
def create_correlation_matrices(
    eval_collections,
    eval_collections_biased,
    eval_collections_supervised,
    eval_names,
    score_type='accuracy_score',
    correlation_method="spearman",
    height=1200,
    width=1400,
    colorscale="RdBu_r",
    show_scores=True,
    idk=False
):
    """
    Crea matrices de correlación separadas por modelo, tipo de respuesta y LLM utilizando Plotly.
    
    Args:
        eval_collections: Lista de colecciones de datos normales
        eval_collections_biased: Lista de colecciones de datos sesgados
        eval_collections_supervised: Lista de colecciones de datos supervisados
        eval_names: Lista de nombres de los modelos
        score_type: Tipo de puntuación a visualizar (ej: 'accuracy_score')
        correlation_method: Método de correlación ('pearson', 'spearman' o 'kendall')
        height: Altura de la figura
        width: Ancho de la figura
        colorscale: Escala de colores para las matrices de correlación
        show_scores: Si es True, muestra los valores numéricos en las celdas
        idk: Si es False, filtra las respuestas "I don't know"
    
    Returns:
        Dictionary: Diccionario con las figuras de Plotly para cada tipo de visualización
    """
    import pandas as pd
    import plotly.graph_objects as go
    import numpy as np
    
    # Definir nombres
    llm_names = {0: 'Human', 1: 'Basic AI', 2: 'Advanced AI'}
    score_titles = {
        'accuracy_score': 'Accuracy',
        'clarity_score': 'Clarity',
        'completeness_score': 'Completeness',
    }
    eval_types = ["normal", "biased", "supervised"]
    
    # Función auxiliar para cargar datos con control de errores
    def safe_load_data(collection, model_name, eval_type, model_idx):
        try:
            if collection is None:
                print(f"Advertencia: Colección para {model_name} ({eval_type}) no está disponible.")
                return None
            
            # Cargar datos
            df = load_data(collection)
            
            # Preparar datos agregados
            df_agregado = df
            df_agregado['LLM_type'] = df_agregado['LLM'].map(llm_names)
            df_agregado['eval_type'] = eval_type
            df_agregado['model_name'] = model_name
            df_agregado['model_idx'] = model_idx
            
            # Agregar columna para identificar la fuente específica
            df_agregado['source'] = f"{model_name}_{eval_type}"
            
            # Filtrar idk si es necesario
            if not idk:
                try:
                    df_agregado['original_answer'] = df_agregado["response"]
                except Exception as e:
                    print(f"Error al asignar response: {str(e)}")

                df_agregado, _, _ = filter_and_analyze_responses(df_agregado)
            
            if len(df_agregado) > 0:
                return df_agregado
            else:
                print(f"Advertencia: No hay datos después del filtrado para {model_name} ({eval_type}).")
                return None
        except Exception as e:
            print(f"Error al cargar datos para {model_name} ({eval_type}): {str(e)}")
            return None
    
    # Procesar y cargar todos los datos
    all_data = []
    for eval_type, collections in [
        ("normal", eval_collections),
        ("biased", eval_collections_biased),
        ("supervised", eval_collections_supervised)
    ]:
        if collections is not None:
            for model_idx, (collection, name) in enumerate(zip(collections, eval_names)):
                df_agregado = safe_load_data(collection, name, eval_type, model_idx)
                if df_agregado is not None:
                    all_data.append(df_agregado)
    
    # Verificar si tenemos datos disponibles
    if not all_data:
        print("Error: No hay datos disponibles para visualizar.")
        return None
    
    # Combinar todos los datos
    combined_df = pd.concat(all_data)
    
    # Verificar si el score_type está disponible
    if score_type not in combined_df.columns:
        print(f"Error: El tipo de puntuación '{score_type}' no está disponible en los datos.")
        return None
    
    # Crear función para generar matriz de correlación con Plotly
    def create_correlation_heatmap(data, title, column_rename=None):
        if data.empty:
            print(f"No hay datos para: {title}")
            return None
        
        # Extraer solo las columnas de scores para correlación
        score_columns = [col for col in data.columns if score_type in col]
        
        if len(score_columns) < 2:
            print(f"No hay suficientes columnas de scores para: {title}")
            return None
        
        # Calcular matriz de correlación
        corr_matrix = data[score_columns].corr(method=correlation_method)
        print(f'tamaño de matriz = {len(data)}')
        # Renombrar columnas si se proporciona un diccionario de mapeo
        if column_rename:
            corr_matrix = corr_matrix.rename(columns=column_rename, index=column_rename)
        
        # Crear heatmap con Plotly
        fig = go.Figure(data=go.Heatmap(
            z=corr_matrix.values,
            x=corr_matrix.columns,
            y=corr_matrix.index,
            colorscale=colorscale,
            zmin=-1, zmax=1,
            colorbar=dict(title=f"{correlation_method.capitalize()} Correlation")
        ))
        
        # Añadir anotaciones con los valores de correlación
        if show_scores:
            annotations = []
            for i, row in enumerate(corr_matrix.values):
                for j, value in enumerate(row):
                    text_color = 'white' if abs(value) > 0.5 else 'black'
                    annotations.append(dict(
                        x=j, y=i,
                        text=f"{value:.2f}",
                        showarrow=False,
                        font=dict(color=text_color)
                    ))
            fig.update_layout(annotations=annotations)
        
        # Actualizar layout
        fig.update_layout(
            title=title,
            height=height,
            width=width,
            xaxis=dict(title=''),
            yaxis=dict(title=''),
            template="plotly_white"
        )
        
        return fig
    
    # Crear un diccionario para almacenar todas las figuras
    figures = {}
    
    # 1. Matrices por modelo
    for model_name in eval_names:
        model_data = combined_df[combined_df['model_name'] == model_name]
        if not model_data.empty:
            # Crear pivot para tener una columna por cada combinación de eval_type y LLM
            pivot_scores = []
            
            for eval_type in eval_types:
               
                    subset = model_data[(model_data['eval_type'] == eval_type)  
                                       ]
                    if not subset.empty:
                        subset_scores = subset[[score_type, 'original_answer_id']]
                        subset_scores = subset_scores.rename(columns={
                            score_type: f"{score_type}_{eval_type}"
                        })
                        pivot_scores.append(subset_scores)
            
            if pivot_scores:
                # Combinar todos los scores en un solo DataFrame
                merged_scores = pivot_scores[0]
                for df in pivot_scores[1:]:
                    merged_scores = pd.merge(merged_scores, df, on='original_answer_id', how='outer')
                
                # Crear matriz de correlación para este modelo
                title = f"Correlación de {score_titles.get(score_type, score_type)} - Modelo: {model_name}"
                fig = create_correlation_heatmap(merged_scores, title)
                if fig:
                    figures[f"model_{model_name}"] = fig
    
    # 2. Matrices por tipo de evaluación
    for eval_type in eval_types:
        eval_data = combined_df[combined_df['eval_type'] == eval_type]
        if not eval_data.empty:
            # Crear pivot para tener una columna por cada combinación de modelo y LLM
            pivot_scores = []
            
            for model_name in eval_names:
                
                    subset = eval_data[(eval_data['model_name'] == model_name)  
                                      ]
                    if not subset.empty:
                        subset_scores = subset[[score_type, 'original_answer_id']]
                        subset_scores = subset_scores.rename(columns={
                            score_type: f"{score_type}_{model_name}"
                        })
                        pivot_scores.append(subset_scores)
            
            if pivot_scores:
                # Combinar todos los scores en un solo DataFrame
                merged_scores = pivot_scores[0]
                for df in pivot_scores[1:]:
                    merged_scores = pd.merge(merged_scores, df, on='original_answer_id', how='outer')
                
                # Crear matriz de correlación para este tipo de evaluación
                title = f"Correlación de {score_titles.get(score_type, score_type)} - Tipo: {eval_type.capitalize()}"
                fig = create_correlation_heatmap(merged_scores, title)
                if fig:
                    figures[f"eval_type_{eval_type}"] = fig
    
    # 3. Matrices por LLM con cálculo por pares
    for llm_id, llm_name in llm_names.items():
        llm_data = combined_df[combined_df['LLM'] == llm_id]
        if not llm_data.empty:
            print(f"\nCalculando matriz de correlación para LLM: {llm_name}")
            
            # Crear una lista de todas las combinaciones de modelo y tipo de evaluación
            model_type_combinations = []
            
            for model_name in eval_names:
                for eval_type in eval_types:
                    combination_name = f"{model_name}_{eval_type}"
                    subset = llm_data[(llm_data['model_name'] == model_name) & 
                                    (llm_data['eval_type'] == eval_type)]
                    
                    if not subset.empty:
                        # Asegurarse de que estamos seleccionando solo los datos numéricos necesarios
                        column_name = f"{score_type}_{combination_name}"
                        temp_df = subset[['original_answer_id', score_type]].copy()
                        temp_df.rename(columns={score_type: column_name}, inplace=True)
                        
                        model_type_combinations.append({
                            'name': combination_name,
                            'data': temp_df
                        })
            
            # Verificar que tenemos suficientes combinaciones
            if len(model_type_combinations) >= 2:
                # Crear matriz vacía para almacenar resultados
                n_combinations = len(model_type_combinations)
                correlation_matrix = pd.DataFrame(
                    np.zeros((n_combinations, n_combinations)),
                    index=[combo['name'] for combo in model_type_combinations],
                    columns=[combo['name'] for combo in model_type_combinations]
                )
                
                # Calcular tamaños para referencia
                total_sizes = {}
                for combo in model_type_combinations:
                    total_sizes[combo['name']] = len(combo['data'])
                    print(f"Total de datos para {combo['name']}: {total_sizes[combo['name']]}")
                
                # Llenar matriz con correlaciones por pares
                for i, combo1 in enumerate(model_type_combinations):
                    for j, combo2 in enumerate(model_type_combinations):
                        if j < i:  # Aprovechamos la simetría
                            correlation_matrix.iloc[i, j] = correlation_matrix.iloc[j, i]
                            continue
                            
                        if i == j:  # Diagonal principal
                            correlation_matrix.iloc[i, j] = 1.0
                            continue
                        
                        try:
                            # Juntar DataFrames para este par específico
                            merged_pair = pd.merge(
                                combo1['data'], 
                                combo2['data'], 
                                on='original_answer_id', 
                                how='inner'
                            )
                            
                            if len(merged_pair) > 1:
                                # Identificar nombres de columnas correctas
                                col1_name = f"{score_type}_{combo1['name']}"
                                col2_name = f"{score_type}_{combo2['name']}"
                                
                                # Convertir a valores numéricos y eliminar NaNs
                                merged_pair[col1_name] = pd.to_numeric(merged_pair[col1_name], errors='coerce')
                                merged_pair[col2_name] = pd.to_numeric(merged_pair[col2_name], errors='coerce')
                                merged_pair = merged_pair.dropna(subset=[col1_name, col2_name])
                                
                                if len(merged_pair) > 1:
                                    # Calcular correlación
                                    pair_corr = merged_pair[col1_name].corr(
                                        merged_pair[col2_name], 
                                        method=correlation_method
                                    )
                                    
                                    # Guardar en matriz
                                    correlation_matrix.iloc[i, j] = pair_corr
                                    
                                    utilized_data = len(merged_pair)
                                    percent1 = utilized_data / total_sizes[combo1['name']] * 100
                                    percent2 = utilized_data / total_sizes[combo2['name']] * 100
                                    
                                    print(f"  Correlación {combo1['name']} - {combo2['name']}: {pair_corr:.2f} " +
                                        f"(n={utilized_data}, {percent1:.1f}% / {percent2:.1f}%)")
                                else:
                                    correlation_matrix.iloc[i, j] = np.nan
                                    print(f"  Insuficientes datos numéricos para {combo1['name']} - {combo2['name']}")
                            else:
                                correlation_matrix.iloc[i, j] = np.nan
                                print(f"  Insuficientes datos para {combo1['name']} - {combo2['name']}")
                        except Exception as e:
                            print(f"  Error calculando correlación {combo1['name']} - {combo2['name']}: {str(e)}")
                            correlation_matrix.iloc[i, j] = np.nan
                
                # Crear visualización para este LLM
                title = f"Correlación por pares de {score_titles.get(score_type, score_type)} - LLM: {llm_name}"
                
                # Crear heatmap
                fig = go.Figure(data=go.Heatmap(
                    z=correlation_matrix.values,
                    x=correlation_matrix.columns,
                    y=correlation_matrix.index,
                    colorscale=colorscale,
                    zmin=-1, zmax=1,
                    showscale=False, 
                ))
                
                # Añadir anotaciones con valores y tamaño de muestra
                if show_scores:
                    annotations = []
                    for i in range(len(correlation_matrix)):
                        for j in range(len(correlation_matrix.columns)):
                            value = correlation_matrix.iloc[i, j]
                            
                            if not np.isnan(value):
                                text_color = 'white' if abs(value) > 0.5 else 'black'
                                
                                if i != j:
                                    combo1 = model_type_combinations[i]['name']
                                    combo2 = model_type_combinations[j]['name']
                                    
                                    try:
                                        # Obtener tamaño de muestra para anotación
                                        merged_pair = pd.merge(
                                            model_type_combinations[i]['data'],
                                            model_type_combinations[j]['data'],
                                            on='original_answer_id',
                                            how='inner'
                                        )
                                        col1_name = f"{score_type}_{combo1}"
                                        col2_name = f"{score_type}_{combo2}"
                                        
                                        merged_pair[col1_name] = pd.to_numeric(merged_pair[col1_name], errors='coerce')
                                        merged_pair[col2_name] = pd.to_numeric(merged_pair[col2_name], errors='coerce')
                                        merged_pair = merged_pair.dropna(subset=[col1_name, col2_name])
                                        
                                        sample_size = len(merged_pair)
                                        text = f"{value:.2f}"
                                    except:
                                        text = f"{value:.2f}"
                                else:
                                    text = f"{value:.2f}"
                                
                                annotations.append(dict(
                                    x=j, y=i,
                                    text=text,
                                    showarrow=False,
                                    font=dict(color=text_color, size=14 if i != j else 16)
                                ))
                    
                    fig.update_layout(annotations=annotations)
                
                # Actualizar layout
                fig.update_layout(
                    title=dict(
                        text=title,
                        font=dict(size=22)  # Título más grande (tamaño 22)
                    ),
                    height=height,
                    width=width,
                    xaxis=dict(
                        title='',
                        tickfont=dict(size=16)  # Tamaño 16 para las etiquetas del eje X
                    ),
                    yaxis=dict(
                        title='',
                        tickfont=dict(size=16)  # Tamaño 16 para las etiquetas del eje Y
                    ),
                    template="plotly_white",
                )
                
                figures[f"llm_{llm_id}_pairwise"] = fig
            else:
                print(f"  No hay suficientes combinaciones para crear matriz de correlación para LLM: {llm_name}")



    # 4. Global pairwise correlation matrix (maximizing data for each pair)
    print("Calculating global pairwise correlation matrix...")

    # Create lists to store all model/type combinations
    all_combinations = []
    for model_name in eval_names:
        for eval_type in eval_types:
            combination_name = f"{model_name}_{eval_type}"
            
            # Skip combinations with "_biased" in the name
            if "_biased" in combination_name:
                print(f"Skipping biased sample: {combination_name}")
                continue
                
            subset = combined_df[(combined_df['model_name'] == model_name) & 
                                (combined_df['eval_type'] == eval_type)]
            
            if not subset.empty:
                # Ensure we're selecting only the necessary numeric data
                # and using specific names to avoid confusion
                column_name = f"{score_type}_{combination_name}"
                temp_df = subset[['original_answer_id', score_type]].copy()
                temp_df.rename(columns={score_type: column_name}, inplace=True)
                
                all_combinations.append({
                    'name': combination_name,
                    'data': temp_df
                })

    # Format the combination names for better display (remove underscores)
    display_names = {}
    for combo in all_combinations:
        # Replace underscores with spaces and capitalize words
        formatted_name = combo['name'].replace('_', ' ')
        # Capitalize each word
        formatted_name = ' '.join(word.capitalize() for word in formatted_name.split())
        display_names[combo['name']] = formatted_name

    # Check if we have enough combinations
    if len(all_combinations) >= 2:
        # Create an empty matrix to store correlation results
        n_combinations = len(all_combinations)
        correlation_matrix = pd.DataFrame(
            np.zeros((n_combinations, n_combinations)),
            index=[combo['name'] for combo in all_combinations],
            columns=[combo['name'] for combo in all_combinations]
        )
        
        # Calculate sizes for reference
        total_sizes = {}
        for combo in all_combinations:
            total_sizes[combo['name']] = len(combo['data'])
            print(f"Total data for {combo['name']}: {total_sizes[combo['name']]}")
        
        # Number of pairs to calculate
        total_pairs = (n_combinations * (n_combinations - 1)) // 2 + n_combinations
        print(f"Calculating {total_pairs} correlation pairs...")
        
        # Fill matrix with pairwise correlations
        for i, combo1 in enumerate(all_combinations):
            for j, combo2 in enumerate(all_combinations):
                if j < i:  # Use symmetry to avoid calculating twice
                    # We already calculated this pair before
                    correlation_matrix.iloc[i, j] = correlation_matrix.iloc[j, i]
                    continue
                    
                if i == j:  # Main diagonal
                    correlation_matrix.iloc[i, j] = 1.0
                    continue
                
                try:
                    # Join DataFrames for this specific pair
                    merged_pair = pd.merge(
                        combo1['data'], 
                        combo2['data'], 
                        on='original_answer_id', 
                        how='inner'
                    )
                    
                    if len(merged_pair) > 1:
                        # Extract specific column names for this pair
                        # Key change: explicitly identify columns by name
                        col1_name = f"{score_type}_{combo1['name']}"
                        col2_name = f"{score_type}_{combo2['name']}"
                        
                        # Ensure columns are numeric
                        merged_pair[col1_name] = pd.to_numeric(merged_pair[col1_name], errors='coerce')
                        merged_pair[col2_name] = pd.to_numeric(merged_pair[col2_name], errors='coerce')
                        
                        # Remove rows with NaN after conversion
                        merged_pair = merged_pair.dropna(subset=[col1_name, col2_name])
                        
                        if len(merged_pair) > 1:
                            # Calculate correlation using only this pair
                            pair_corr = merged_pair[col1_name].corr(
                                merged_pair[col2_name], 
                                method=correlation_method
                            )
                            
                            # Save to matrix
                            correlation_matrix.iloc[i, j] = pair_corr
                            
                            utilized_data = len(merged_pair)
                            percent1 = utilized_data / total_sizes[combo1['name']] * 100
                            percent2 = utilized_data / total_sizes[combo2['name']] * 100
                            
                            print(f"Correlation {combo1['name']} - {combo2['name']}: {pair_corr:.2f} " +
                                f"(using {utilized_data} data points, {percent1:.1f}% / {percent2:.1f}%)")
                        else:
                            # Not enough common data after NaN filtering
                            correlation_matrix.iloc[i, j] = np.nan
                            print(f"Insufficient numeric data for correlation {combo1['name']} - {combo2['name']}")
                    else:
                        # Not enough common data for this pair
                        correlation_matrix.iloc[i, j] = np.nan
                        print(f"Insufficient data for correlation {combo1['name']} - {combo2['name']}")
                except Exception as e:
                    # Error handling with more information
                    print(f"Error calculating correlation {combo1['name']} - {combo2['name']}: {str(e)}")
                    try:
                        # Try to show debug information if possible
                        if 'merged_pair' in locals() and 'col1_name' in locals() and 'col2_name' in locals():
                            print(f"Available columns: {merged_pair.columns.tolist()}")
                            print(f"Data types: {col1_name}: {merged_pair[col1_name].dtype}, {col2_name}: {merged_pair[col2_name].dtype}")
                            print(f"Data sample: \n{merged_pair[[col1_name, col2_name]].head()}")
                    except:
                        pass
                    correlation_matrix.iloc[i, j] = np.nan
        
        # Create visualization for the global matrix
        title = f"Pairwise Correlation of Evaluations for the Same Response"
        
        # Use formatted display names for visualization
        display_indices = [display_names[idx] for idx in correlation_matrix.index]
        display_columns = [display_names[col] for col in correlation_matrix.columns]
        
        # Create heatmap with calculated values
        fig = go.Figure(data=go.Heatmap(
            z=correlation_matrix.values,
            x=display_columns,
            y=display_indices,
            colorscale=colorscale,
            zmin=-1, zmax=1,
            showscale=False  
        ))
        
        # Add annotations with correlation values and sample size
        if show_scores:
            annotations = []
            for i in range(len(correlation_matrix)):
                for j in range(len(correlation_matrix.columns)):
                    value = correlation_matrix.iloc[i, j]
                    
                    # Only proceed if value is not NaN
                    if not np.isnan(value):
                        # Determine color based on value
                        text_color = 'white' if abs(value) > 0.5 else 'black'
                        
                        # Format the text
                        text = f"{value:.2f}"
                        
                        annotations.append(dict(
                            x=j, y=i,
                            text=text,
                            showarrow=False,
                            font=dict(
                                color=text_color, 
                                size=24 if i != j else 24,  # Much larger fonts
                                family="Arial, sans-serif"
                            )
                        ))
            
            fig.update_layout(annotations=annotations)
        
        # Update layout for publication quality
        fig.update_layout(
            title=dict(
                text=title,
                font=dict(size=36, family="Arial, sans-serif")  # Much larger title (size 36)
            ),
            height=height,
            width=width,
            xaxis=dict(
                title='',
                tickfont=dict(size=30, family="Arial, sans-serif"),  # Much larger x-axis labels (size 30)
                tickangle=45  # Angle labels for better readability
            ),
            yaxis=dict(
                title='',
                tickfont=dict(size=30, family="Arial, sans-serif")  # Much larger y-axis labels (size 30)
            ),
            template="plotly_white",
            margin=dict(l=150, r=150, t=100, b=150),  # Larger margins for better label display
            paper_bgcolor='white',
            plot_bgcolor='white'
        )
        
        figures["global_pairwise"] = fig
        
        # Also return the correlation matrix for inspection
        figures["correlation_matrix_data"] = correlation_matrix
    else:
        print("Not enough combinations to create a global correlation matrix.")

    # ===== CÓDIGO PARA CLUSTERING (AÑADIR DESPUÉS DE TU CÓDIGO ORIGINAL) =====
    # Importamos las bibliotecas necesarias para clustering
    from scipy.cluster import hierarchy
    from scipy.spatial import distance

    print("\n==== CREANDO VISUALIZACIÓN CON CLUSTERING ====")

    # PASO 1: OBTENER ORDEN DEL CLUSTERING SIN MODIFICAR LA MATRIZ ORIGINAL

    # Primero creamos una copia profunda de la matriz original para no alterarla
    correlation_matrix_copy = correlation_matrix.copy(deep=True)

    # Aseguramos que la copia es idéntica verificando algunos valores
    sample_i, sample_j = 0, 1  # Verificamos la primera correlación no diagonal
    original_value = correlation_matrix.iloc[sample_i, sample_j]
    copy_value = correlation_matrix_copy.iloc[sample_i, sample_j]
    print(f"Verification - Original matrix value at [{sample_i},{sample_j}]: {original_value}")
    print(f"Verification - Copy matrix value at [{sample_i},{sample_j}]: {copy_value}")
    assert original_value == copy_value, "Error: Los valores en la copia no coinciden con el original"

    # Para clustering necesitamos una matriz sin NaN, creamos una versión temporal solo para calcular el orden
    correlation_matrix_for_clustering = correlation_matrix_copy.fillna(0)

    # Convertimos la matriz de correlación en una matriz de distancia
    distance_matrix = 1 - np.abs(correlation_matrix_for_clustering)

    # Realizamos el clustering jerárquico
    linkage_matrix = hierarchy.linkage(distance.squareform(distance_matrix), method='ward')

    # Obtenemos el orden de filas/columnas basado en el clustering
    dendro_order = hierarchy.leaves_list(linkage_matrix)

    # PASO 2: REORDENAR LA MATRIZ ORIGINAL (EXACTAMENTE LA MISMA) SEGÚN EL CLUSTERING

    # Guardamos los nombres originales en el orden del clustering
    clustered_index = [correlation_matrix.index[i] for i in dendro_order]
    clustered_columns = [correlation_matrix.columns[i] for i in dendro_order]

    print(f"Original indices: {correlation_matrix.index.tolist()}")
    print(f"Clustered indices: {clustered_index}")

    # Creamos una nueva matriz con el mismo contenido pero con el orden según el clustering
    clustered_matrix = correlation_matrix.loc[clustered_index, clustered_columns]

    # Verificamos que los valores se mantienen idénticos
    original_values = []
    clustered_values = []

    # Tomamos algunos pares para comparar
    check_pairs = [(0, 1), (1, 2), (2, 3)]
    for orig_i, orig_j in check_pairs:
        if orig_i < len(correlation_matrix) and orig_j < len(correlation_matrix):
            # Obtenemos los nombres en la matriz original
            orig_index = correlation_matrix.index[orig_i]
            orig_column = correlation_matrix.columns[orig_j]
            
            # Encontramos dónde están esos nombres en la matriz clusterizada
            if orig_index in clustered_matrix.index and orig_column in clustered_matrix.columns:
                # Valor en la matriz original
                orig_value = correlation_matrix.loc[orig_index, orig_column]
                # El mismo valor pero en la matriz clusterizada
                clust_value = clustered_matrix.loc[orig_index, orig_column]
                
                original_values.append(orig_value)
                clustered_values.append(clust_value)
                
                print(f"Check pair [{orig_i},{orig_j}] ({orig_index},{orig_column}):")
                print(f"  Original matrix: {orig_value}")
                print(f"  Clustered matrix: {clust_value}")
                
                # Verificamos que sean idénticos
                if pd.isna(orig_value) and pd.isna(clust_value):
                    print("  ✓ Values match (both are NaN)")
                elif orig_value == clust_value:
                    print("  ✓ Values match")
                else:
                    print("  ✗ VALUES DO NOT MATCH!")

    # PASO 3: CREAR LA VISUALIZACIÓN CON LA MATRIZ REORDENADA

    # Aseguramos que tenemos los nombres de visualización
    if 'display_names' not in locals() or not isinstance(display_names, dict):
        print("Creating display_names dictionary")
        display_names = {}
        for idx in correlation_matrix.index:
            formatted_name = idx.replace('_', ' ')
            formatted_name = ' '.join(word.capitalize() for word in formatted_name.split())
            display_names[idx] = formatted_name

    # Obtenemos los nombres de visualización para la matriz clusterizada
    clustered_display_indices = []
    clustered_display_columns = []

    for idx in clustered_matrix.index:
        if idx in display_names:
            clustered_display_indices.append(display_names[idx])
        else:
            print(f"Warning: No display name for {idx}, using original")
            clustered_display_indices.append(str(idx))

    for col in clustered_matrix.columns:
        if col in display_names:
            clustered_display_columns.append(display_names[col])
        else:
            print(f"Warning: No display name for {col}, using original")
            clustered_display_columns.append(str(col))

    # Creamos el mapa de calor clusterizado con los MISMOS valores que el original
    fig_clustered = go.Figure(data=go.Heatmap(
        z=clustered_matrix.values,
        x=clustered_display_columns,
        y=clustered_display_indices,
        colorscale=colorscale,
        zmin=-1, zmax=1,
        showscale=False
    ))

    # Añadimos anotaciones con los valores de correlación
    if show_scores:
        annotations = []
        for i in range(len(clustered_matrix)):
            for j in range(len(clustered_matrix.columns)):
                value = clustered_matrix.iloc[i, j]
                
                # Solo procede si el valor no es NaN
                if not pd.isna(value):
                    # Determina el color basado en el valor
                    text_color = 'white' if abs(value) > 0.5 else 'black'
                    
                    # Formatea el texto exactamente igual que en la visualización original
                    text = f"{value:.2f}"
                    
                    annotations.append(dict(
                        x=j, y=i,
                        text=text,
                        showarrow=False,
                        font=dict(
                            color=text_color, 
                            size=24 if i != j else 24,
                            family="Arial, sans-serif"
                        )
                    ))
        
        fig_clustered.update_layout(annotations=annotations)

    # Actualizamos el diseño manteniendo el estilo original
    fig_clustered.update_layout(
        title=dict(
            text="Pairwise Correlation of Evaluations (Clustered)",
            font=dict(size=36, family="Arial, sans-serif")
        ),
        height=height,
        width=width,
        xaxis=dict(
            title='',
            tickfont=dict(size=30, family="Arial, sans-serif"),
            tickangle=45
        ),
        yaxis=dict(
            title='',
            tickfont=dict(size=30, family="Arial, sans-serif")
        ),
        template="plotly_white",
        margin=dict(l=150, r=150, t=100, b=150),
        paper_bgcolor='white',
        plot_bgcolor='white'
    )

    # PASO 4: CREAR UN DENDROGRAMA PARA VISUALIZAR LA JERARQUÍA

    # Creamos el dendrograma
    fig_dendrogram = go.Figure()
    fig_dendrogram.add_trace(go.Scatter(
        x=np.repeat(np.arange(len(linkage_matrix) + 1), 2)[1:],
        y=np.repeat(linkage_matrix[:, 2], 2),
        mode='lines',
    ))

    # Añadimos etiquetas al dendrograma usando los nombres de visualización
    dendro_leaves = hierarchy.leaves_list(linkage_matrix)
    dendro_labels = []

    for i in dendro_leaves:
        idx = correlation_matrix.index[i]
        if idx in display_names:
            dendro_labels.append(display_names[idx])
        else:
            dendro_labels.append(str(idx))

    fig_dendrogram.update_layout(
        title=dict(
            text="Hierarchical Clustering of Evaluation Methods",
            font=dict(size=36, family="Arial, sans-serif")
        ),
        height=height//2,
        width=width,
        xaxis=dict(
            ticktext=dendro_labels,
            tickvals=list(range(len(dendro_labels))),
            tickfont=dict(size=24, family="Arial, sans-serif"),
            tickangle=45
        ),
        template="plotly_white",
        margin=dict(l=100, r=100, t=100, b=150)
    )

    # PASO 5: ANÁLISIS DE CLUSTERS
    from scipy.cluster.hierarchy import fcluster
    max_clusters = min(8, len(correlation_matrix))  # Limitamos a 8 clusters o menos
    cluster_assignments = {}

    for num_clusters in range(2, max_clusters + 1):
        # Obtenemos las asignaciones de cluster
        clusters = fcluster(linkage_matrix, t=num_clusters, criterion='maxclust')
        
        # Organizamos los resultados por cluster
        cluster_groups = {}
        for i, cluster_id in enumerate(clusters):
            if cluster_id not in cluster_groups:
                cluster_groups[cluster_id] = []
            original_index = correlation_matrix.index[i]
            if original_index in display_names:
                cluster_groups[cluster_id].append(display_names[original_index])
            else:
                cluster_groups[cluster_id].append(str(original_index))
        
        cluster_assignments[num_clusters] = cluster_groups

    # Imprimimos los resultados para diferentes números de clusters
    print("\nCluster Assignments:")
    for num_clusters, groups in cluster_assignments.items():
        print(f"\nWith {num_clusters} clusters:")
        for cluster_id, members in groups.items():
            print(f"  Cluster {cluster_id}: {', '.join(members)}")

    # Guardamos las figuras en el diccionario
    figures["clustered_pairwise"] = fig_clustered
    figures["dendrogram"] = fig_dendrogram

    print("Clustering completed successfully")
            
    return figures

In [ ]:
figuras = create_correlation_matrices(
    EVAL_COLLECTION_LIST, 
    [], 
    EVAL_COLLECTION_LIST_SUPERVISED, 
    EVAL_NAMES,
    score_type='accuracy_score'
)

# Depuración
print(f"Tipo de retorno: {type(figuras)}")
if figuras is None:
    print("La función retornó None. Revisa los mensajes de error en la consola.")
elif isinstance(figuras, dict):
    print(f"Se generaron {len(figuras)} figuras.")
    print(f"Claves: {list(figuras.keys())}")
    
    # Verificar si alguna figura es None
    for key, fig in figuras.items():
        if fig is None:
            print(f"La figura '{key}' es None")
        else:
            try:
                # Intenta mostrar la figura
                fig.show()
                print(f"Figura '{key}' mostrada correctamente")
            except Exception as e:
                print(f"Error al mostrar la figura '{key}': {str(e)}")

In [ ]:
def visualize_confusion_matrices(
    eval_collections,
    eval_collections_biased,
    eval_collections_supervised,
    eval_names,
    organization='by_category',  # 'by_category', 'by_model'
    llm_filter=None,  # [0, 1, 2] o None para todos
    height_per_subplot=400,  # Altura de cada subplot
    width_per_subplot=400,  # Ancho de cada subplot
    colorscale="Blues",  # Escala de colores para las matrices
    normalize=True,  # Si es True, normaliza las matrices (porcentajes)
    include_metrics=True  # Si es True, incluye métricas de rendimiento
):
    """
    Función para visualizar matrices de confusión para la predicción de fuente (human/AI).
    
    Args:
        eval_collections: Lista de colecciones de datos normales.
        eval_collections_biased: Lista de colecciones de datos sesgados.
        eval_collections_supervised: Lista de colecciones de datos supervisados.
        eval_names: Lista de nombres de los modelos.
        organization: Organización de los subplots ('by_category', 'by_model').
        llm_filter: IDs de LLM a incluir, None para todos.
        height_per_subplot: Altura de cada subplot (por defecto 500).
        width_per_subplot: Ancho de cada subplot (por defecto 500).
        colorscale: Escala de colores para las matrices.
        normalize: Si es True, normaliza las matrices (porcentajes).
        include_metrics: Si es True, incluye métricas de rendimiento (accuracy, precision, recall, F1).
    """
    import pandas as pd
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    import numpy as np
    from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
    
    # Definir nombres
    llm_names = {0: 'Human', 1: 'Basic AI', 2: 'Advanced AI'}
    category_names = {"normal": "Normal", "biased": "Biased", "supervised": "Supervised"}
    
    if llm_filter is None:
        llm_filter = list(llm_names.keys())
    

    # Preparar datos procesados
    all_data = []
    available_models = []
    categories_with_data = set()
    available_llm_types = set()
    
    # Procesar todos los datos
    for eval_type, collections in [
        ("normal", eval_collections),
        ("biased", eval_collections_biased),
        ("supervised", eval_collections_supervised)
    ]:
        if collections is not None:
            for model_idx, (collection, name) in enumerate(zip(collections, eval_names)):
                try:
                    df = load_data(collection)
                    try:
                        df["source_guessed"] = df["response_source"]
                    except:
                        pass
                    if df is not None:
                        df['LLM_type'] = df['LLM'].map(llm_names)
                        df['eval_type'] = eval_type
                        df['model_name'] = name
                        df['model_idx'] = model_idx
                        df = df[df['LLM'].isin(llm_filter)]
                        
                        if 'source_guessed' not in df.columns:
                            print(f"Advertencia: 'source_guessed' no está en {name} ({eval_type}).")
                            continue
                        
                        if len(df) > 0:
                            available_models.append(name)
                            categories_with_data.add(eval_type)
                            available_llm_types.update(df['LLM'].unique())
                            all_data.append(df)
                except Exception as e:
                    print(f"Error al procesar {name} ({eval_type}): {str(e)}")
    
    if not all_data:
        print("Error: No hay datos disponibles para visualizar.")
        return
    
    # Combinar todos los datos
    combined_df = pd.concat(all_data)
    
    # Mapear LLM a etiquetas "human" o "ai"
    combined_df['true_label'] = combined_df['LLM'].apply(lambda x: "human" if x == 0 else "ai")
    
    # Limpiar source_guessed
    combined_df['source_guessed'] = combined_df['source_guessed'].str.lower()
    combined_df['source_guessed'] = combined_df['source_guessed'].apply(
        lambda x: "human" if x in ["human", "human?", "h"] else "ai" if x in ["ai", "ai?", "a"] else None
    )
    combined_df = combined_df.dropna(subset=['source_guessed'])
    
    # Configurar subplots según la organización
    if organization == 'by_category':
        categories = list(categories_with_data)
        n_rows = len(categories)
        n_cols = len(available_models)
        subplot_titles = [f"{category_names[cat]} - {model}" for cat in categories for model in available_models]
    elif organization == 'by_model':
        n_rows = len(available_models)
        n_cols = len(categories_with_data)
        subplot_titles = [f"{model} - {category_names[cat]}" for model in available_models for cat in categories_with_data]
    else:
        print("Error: Organización no válida.")
        return
    
    # Calcular el tamaño total de la figura
    total_height = height_per_subplot * n_rows
    total_width = width_per_subplot * n_cols
    
    # Crear figura
    fig = make_subplots(
        rows=n_rows, cols=n_cols,
        subplot_titles=subplot_titles,
        vertical_spacing=0.01,  # Espaciado vertical entre subplots
        horizontal_spacing=0.1   # Espaciado horizontal entre subplots
    )
    
    # Función para agregar heatmap y métricas
    def add_heatmap(fig, cm, cm_display, labels, row_idx, col_idx, include_metrics, y_true, y_pred):
        heatmap = go.Heatmap(
            z=cm_display,
            x=labels,
            y=labels,
            colorscale=colorscale,
            showscale=False,
            hoverinfo='text',
            text=[[f"True: {labels[i]}<br>Predicted: {labels[j]}<br>Count: {cm[i, j]}" + 
                   (f"<br>Percentage: {cm_display[i, j]:.1f}%" if normalize else "") 
                   for j in range(len(labels))] for i in range(len(labels))]
        )
        fig.add_trace(heatmap, row=row_idx, col=col_idx)
        
        # Añadir anotaciones
        for i in range(len(labels)):
            for j in range(len(labels)):
                text = f"{cm[i, j]}"
                if normalize:
                    text += f"<br>({cm_display[i, j]:.1f}%)"
                fig.add_annotation(
                    x=labels[j], 
                    y=labels[i],
                    text=text,
                    showarrow=False,
                    font=dict(color="white" if cm_display[i, j] > 50 else "black"),
                    row=row_idx, 
                    col=col_idx
                )
        
        # Añadir métricas
        if include_metrics:
            accuracy = accuracy_score(y_true, y_pred)
            precision = precision_score(y_true, y_pred, pos_label="ai", zero_division=0)
            recall = recall_score(y_true, y_pred, pos_label="ai", zero_division=0)
            f1 = f1_score(y_true, y_pred, pos_label="ai", zero_division=0)
            metrics_text = f"Acc: {accuracy:.2f}<br>Prec: {precision:.2f}<br>Rec: {recall:.2f}<br>F1: {f1:.2f}"
            fig.add_annotation(
                x=0.5, 
                y=-0.25,
                text=metrics_text,
                showarrow=False,
                xref=f"x{row_idx}{col_idx}",
                yref=f"y{row_idx}{col_idx}",
                xanchor="center",
                yanchor="top"
            )
    
    # Generar heatmaps
    labels = ["human", "ai"]
    for row_idx in range(1, n_rows + 1):
        for col_idx in range(1, n_cols + 1):
            if organization == 'by_category':
                category = categories[row_idx - 1]
                model_name = available_models[col_idx - 1]
                data = combined_df[(combined_df['eval_type'] == category) & (combined_df['model_name'] == model_name)]
            else:
                model_name = available_models[row_idx - 1]
                category = list(categories_with_data)[col_idx - 1]
                data = combined_df[(combined_df['model_name'] == model_name) & (combined_df['eval_type'] == category)]
            
            if not data.empty:
                y_true = data['true_label'].values
                y_pred = data['source_guessed'].values
                cm = confusion_matrix(y_true, y_pred, labels=labels)
                cm_display = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100 if normalize else cm
                add_heatmap(fig, cm, cm_display, labels, row_idx, col_idx, include_metrics, y_true, y_pred)
    
    # Actualizar layout
    normalize_suffix = " (Normalized)" if normalize else ""
    fig.update_layout(
        title=f"Confusion Matrices: True Source vs Predicted Source{normalize_suffix}",
        height=total_height,
        width=total_width,
        template="plotly_white",
        margin=dict(t=120, b=120, l=80, r=80)
    )
    
    fig.show()

In [ ]:
visualize_confusion_matrices(
    EVAL_COLLECTION_LIST, 
    None, 
    EVAL_COLLECTION_LIST_SUPERVISED, 
    EVAL_NAMES,
    organization='by_model',  # Alternativa: 'by_model'
    normalize=True,  # Muestra porcentajes en lugar de recuentos brutos
    include_metrics=False,
    
)

In [ ]:
# Función para calcular métricas a partir de una matriz de confusión
def calcular_metricas_confusion(confusion_matrix):
    """
    Calcula accuracy, recall, precision y f1_score a partir de una matriz de confusión.
    
    Args:
        confusion_matrix: Matriz de confusión 2x2 [TP, FN; FP, TN]
    
    Returns:
        dict: Diccionario con las métricas calculadas
    """
    # Extraer valores de la matriz
    tp = confusion_matrix[0, 0]  # Verdaderos positivos
    fn = confusion_matrix[0, 1]  # Falsos negativos
    fp = confusion_matrix[1, 0]  # Falsos positivos
    tn = confusion_matrix[1, 1]  # Verdaderos negativos
    
    # Calcular métricas
    accuracy = (tp + tn) / (tp + tn + fp + fn)
    
    # Manejo de casos donde el denominador es cero
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    
    # F1 Score: media armónica de precision y recall
    f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    return {
        'accuracy': accuracy,
        'recall': recall,
        'precision': precision,
        'f1_score': f1_score
    }

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

def create_graphical_abstract(
    df_eval_humans, 
    df_eval_ai,
    output_file='graphical_abstract.html',
    height=900,
    width=1200,
    title='Bias in Specialized Medical Response Evaluation',
    subtitle='Comparison between human evaluators and LLama 3.1 7B in cardiology and cellular biology',
    calculate_metrics=True
):
    """
    Creates a graphical abstract from two DataFrames.
    
    Args:
        df_eval_humans: DataFrame with human evaluations. 
                       Must have columns 'LLM_type' (with values 'Human' or 'AI')
                       and a column with scores (default 'score').
        df_eval_ai: DataFrame with AI evaluations.
                   Same format as df_eval_humans.
        output_file: Path to save the result (None to only display)
        height: Figure height
        width: Figure width
        title: Main title
        subtitle: Subtitle
        calculate_metrics: If True, calculates confusion and correlation matrices
                          from the data. If False, uses example values.
    
    Returns:
        fig: Plotly figure object
    """

    try:
        df_eval_humans['original_answer'] = df_eval_humans["response"]
    except Exception as e:
        print(e)

    df_eval_humans, summary, detailed_counts = filter_and_analyze_responses(df_eval_humans)

    try:
        df_eval_ai['original_answer'] = df_eval_ai["response"]
    except Exception as e:
        print(e)

    df_eval_ai, summary, detailed_counts = filter_and_analyze_responses(df_eval_ai)


    # Make sure DataFrames have the correct format
    for df, type_name in [(df_eval_humans, "humans"), (df_eval_ai, "AI")]:
        # Verify LLM_type column
        llm_names = {0: 'Human', 1: 'AI', 2: 'Advanced AI'}
        df['LLM_type'] = df['LLM'].map(llm_names)
        if 'LLM_type' not in df.columns:
            # Try to infer if there are alternative columns
            possible_cols = ['LLM']
            col_found = False
            for col in possible_cols:
                if col in df.columns:
                    print(f"Using column '{col}' as 'LLM_type' for {type_name}")
                    df['LLM_type'] = df[col]
                    col_found = True
                    break
            if not col_found:
                raise ValueError(f"The {type_name} DataFrame does not have the 'LLM_type' column or similar.")
        
        # Verify score column
        if 'score' not in df.columns:
            # Try to infer if there are alternative columns
            possible_cols = ['accuracy_score', 'score_total', 'evaluation', 'rating']
            col_found = False
            for col in possible_cols:
                if col in df.columns:
                    print(f"Using column '{col}' as 'score' for {type_name}")
                    df['score'] = df[col]
                    col_found = True
                    break
            if not col_found:
                raise ValueError(f"The {type_name} DataFrame does not have a score column.")
    
    # Color configuration
    human_color = '#004677'  # Color for human evaluators
    human_fill_color = '#5181a3'  # Color for human evaluators

    ai_color = '#FF7F0E'     # Color for AI evaluators
    ai_fill_color = '#fea14f'  # Color for AI evaluators
    
    # Extract data for violin plots
    human_resp_by_humans = df_eval_humans[df_eval_humans['LLM_type'] == 'Human']['score']
    ai_resp_by_humans = df_eval_humans[df_eval_humans['LLM_type'] == 'AI']['score']
    
    human_resp_by_ai = df_eval_ai[df_eval_ai['LLM_type'] == 'Human']['score']
    ai_resp_by_ai = df_eval_ai[df_eval_ai['LLM_type'] == 'AI']['score']
    
    # Calculate confusion and correlation matrices
    if calculate_metrics:
        # Calculate human confusion matrix
        print(df_eval_ai.columns)
        try:
            # Try different column names
            for pred_col in ['response_source']:
                for true_col in ['LLM_type']:
                    if pred_col in df_eval_humans.columns and true_col in df_eval_humans.columns:
                        # Count predictions
                        tp = sum((df_eval_humans[true_col] == 'Human') & (df_eval_humans[pred_col] == 'human'))
                        fp = sum((df_eval_humans[true_col] == 'AI') & (df_eval_humans[pred_col] == 'human'))
                        fn = sum((df_eval_humans[true_col] == 'Human') & (df_eval_humans[pred_col] == 'ai'))
                        tn = sum((df_eval_humans[true_col] == 'AI') & (df_eval_humans[pred_col] == 'ai'))
                        
                        total_h = tp + fn
                        total_ai = fp + tn
                        
                        # Normalized matrix
                        confusion_matrix_human = np.array([
                            [tp/total_h if total_h > 0 else 0, fn/total_h if total_h > 0 else 0],
                            [fp/total_ai if total_ai > 0 else 0, tn/total_ai if total_ai > 0 else 0]
                        ])
                        break
                else:
                    continue
                break
            else:
                # If we didn't find columns, use values close to random
                print("No columns found for human confusion matrix, using example values.")
                confusion_matrix_human = np.array([
                    [0.52, 0.48],  # Real: Human, Pred: [Human, AI]
                    [0.47, 0.53]   # Real: AI, Pred: [Human, AI]
                ])
        except Exception as e:
            print(f"Error calculating human confusion matrix: {e}")
            confusion_matrix_human = np.array([
                [0.52, 0.48],
                [0.47, 0.53]
            ])
        
        # Calculate AI confusion matrix
        try:
            # Similar for AI
          
            for pred_col in ['source_guessed']:
                for true_col in ['LLM_type']:
                    if pred_col in df_eval_ai.columns and true_col in df_eval_ai.columns:
                        # Count predictions
                       
                        tp = sum((df_eval_ai[true_col] == 'Human') & (df_eval_ai[pred_col] == 'human'))
                        fp = sum((df_eval_ai[true_col] == 'AI') & (df_eval_ai[pred_col] == 'human'))
                        fn = sum((df_eval_ai[true_col] == 'Human') & (df_eval_ai[pred_col] == 'ai'))
                        tn = sum((df_eval_ai[true_col] == 'AI') & (df_eval_ai[pred_col] == 'ai'))
                        
                        total_h = tp + fn
                        total_ai = fp + tn
                        
                        # Normalized matrix
                        confusion_matrix_ai = np.array([
                            [tp/total_h if total_h > 0 else 0, fn/total_h if total_h > 0 else 0],
                            [fp/total_ai if total_ai > 0 else 0, tn/total_ai if total_ai > 0 else 0]
                        ])
                        break
                else:
                    continue
                break
            else:
                # If we didn't find columns, use values close to random
                print("No columns found for AI confusion matrix, using example values.")
                confusion_matrix_ai = np.array([
                    [0.54, 0.46],
                    [0.49, 0.51]
                ])
        except Exception as e:
            print(f"Error calculating AI confusion matrix: {e}")
            confusion_matrix_ai = np.array([
                [0.54, 0.46],
                [0.49, 0.51]
            ])
            
        # Calculate correlation matrix using merge to combine data
        try:
            # ID column
            id_col = 'original_answer_id'
            
            if id_col in df_eval_humans.columns and id_col in df_eval_ai.columns:
               
                human_scores = df_eval_humans[[id_col, 'score']].copy()
                ai_scores = df_eval_ai[[id_col, 'score']].copy()
                
                # Rename columns to avoid conflicts
                human_scores.columns = [id_col, 'human_score']
                ai_scores.columns = [id_col, 'ai_score']
                
                # Combine data using merge with inner join
                combined_df = pd.merge(human_scores, ai_scores, on=id_col, how='outer')
                
                if len(combined_df) > 1:
                    # Calculate correlation if there's enough data
                    corr = combined_df['human_score'].corr(combined_df['ai_score'], method="spearman")
                    
                    correlation_matrix = np.array([
                        [1.00, corr],
                        [corr, 1.00]
                    ])
                    
                    print(f"Found {len(combined_df)} common responses. Correlation: {corr:.2f}")
                else:
                    print("Not enough common data to calculate correlation")
                    correlation_matrix = np.array([
                        [1.00, 0.28],
                        [0.28, 1.00]
                    ])
            else:
                print(f"Column {id_col} not found in both DataFrames")
                correlation_matrix = np.array([
                    [1.00, 0.28],
                    [0.28, 1.00]
                ])
        except Exception as e:
            print(f"Error calculating correlation matrix: {e}")
            correlation_matrix = np.array([
                [1.00, 0.28],
                [0.28, 1.00]
            ])
    else:
        # Example values if metrics shouldn't be calculated
        confusion_matrix_human = np.array([
            [0.52, 0.48],
            [0.47, 0.53]
        ])
        
        confusion_matrix_ai = np.array([
            [0.54, 0.46],
            [0.49, 0.51]
        ])
        
        correlation_matrix = np.array([
            [1.00, 0.28],
            [0.28, 1.00]
        ])
        
    # Create figure with subplots - 2 columns for confusion matrices
    fig = make_subplots(
        rows=5, cols=3,
        specs=[
            [{"colspan": 3, "rowspan": 1}, None, None],             # Title
            [{"rowspan": 2}, {"rowspan": 2}, None],                 # Violin plots
            [None, None, None],                                      # (continuation violin plots)
            [{"rowspan": 2}, {"rowspan": 2}, {"rowspan": 2}],       # Confusion and correlation matrices
            [None, None, None],                                      # (continuation matrices)
        ],
        subplot_titles=(
            "", 
            "Human Evaluators", "AI Evaluators", 
            "Human Conf. Matrix", "AI Conf. Matrix", "Correlation"
        ),
        vertical_spacing=0.15,
        horizontal_spacing=0.1,
        row_heights=[0.15, 0.3, 0.05, 0.3, 0.1]  # Adjust heights
    )
    
    # 1. Main title with HTML formatting
    fig.add_annotation(
        x=0.5, y=1,
        xref="paper", yref="paper",
        text=f"<b>{title}</b>",
        showarrow=False,
        font=dict(size=28, family="Arial, sans-serif"),  # Larger font
        align="center"
    )
    
    fig.add_annotation(
        x=0.5, y=0.95,
        xref="paper", yref="paper",
        text=subtitle,
        showarrow=False,
        font=dict(size=20, family="Arial, sans-serif", color="#555"),  # Larger font
        align="center"
    )
    
    # 2. Create violin plots for human evaluations
    fig.add_trace(
        go.Violin(
            y=human_resp_by_humans,
            name='Humans → Human Resp.',
            box_visible=True,
            meanline_visible=True,
            line_color=human_color,
           
            opacity=0.6,
            side='negative',
            showlegend=True,
            legendgroup='human_eval',
            x0=0
        ), row=2, col=1
    )
    
    fig.add_trace(
        go.Violin(
            y=ai_resp_by_humans,
            name='Humans → AI Resp.',
            box_visible=True,
            meanline_visible=True,
            line_color=human_color,
           
            opacity=0.3,
            side='positive',
            showlegend=True,
            legendgroup='human_eval',
            x0=0
        ), row=2, col=1
    )
    
    # 3. Create violin plots for AI evaluations
    fig.add_trace(
        go.Violin(
            y=human_resp_by_ai,
            name='AI → Human Resp.',
            box_visible=True,
            meanline_visible=True,
            line_color=ai_color,
           
            opacity=0.6,
            side='negative',
            showlegend=True,
            legendgroup='ai_eval',
            x0=0
        ), row=2, col=2
    )
    
    fig.add_trace(
        go.Violin(
            y=ai_resp_by_ai,
            name='AI → AI Resp.',
            box_visible=True,
            meanline_visible=True,
            line_color=ai_color,
            
            opacity=0.3,
            side='positive',
            showlegend=True,
            legendgroup='ai_eval',
            x0=0
        ), row=2, col=2
    )
    
    # 4. Visualization of confusion matrix for humans - NOW IN COL 1
    fig.add_trace(
        go.Heatmap(
            z=confusion_matrix_human,
            x=['Pred. Human', 'Pred. AI'],
            y=['Real Human', 'Real AI'],
            colorscale=[[0, 'rgb(255,255,255)'], [1, '#004677']],
            showscale=False,
            text=[[f"{val:.1%}" for val in row] for row in confusion_matrix_human],
            texttemplate="%{text}",
            textfont={"size": 18},  # Larger font
            hoverinfo='text',
            zmin=0, zmax=1,
            name='Confusion Matrix (Human)'
        ), row=4, col=1
    )
    
    # 5. Visualization of confusion matrix for AI - NOW IN COL 2
    fig.add_trace(
        go.Heatmap(
            z=confusion_matrix_ai,
            x=['Pred. Human', 'Pred. AI'],
            y=['Real Human', 'Real AI'],
            colorscale=[[0, 'rgb(255,255,255)'], [1, '#FF7F0E']],
            showscale=False,
            text=[[f"{val:.1%}" for val in row] for row in confusion_matrix_ai],
            texttemplate="%{text}",
            textfont={"size": 18},  # Larger font
            hoverinfo='text',
            zmin=0, zmax=1,
            name='Confusion Matrix (AI)'
        ), row=4, col=2
    )
    
    # 6. Correlation heatmap - NOW IN COL 3
    corr_colors = "RdBu_r"
    
    # Section subtitles
    fig.add_annotation(
        x=0.5, y=.88,
        xref="paper", yref="paper",
        text="<b>Score Distribution by Evaluator Type</b>",
        showarrow=False,
        font=dict(size=22, family="Arial, sans-serif"),  # Larger font
        align="center"
    )
    
    # Section subtitles
    fig.add_annotation(
        x=-0.05, y=.42,
        xref="paper", yref="paper",
        text="<b>Confusion Patterns in Authorship Identification: AI vs. Human</b>",
        showarrow=False,
        font=dict(size=22, family="Arial, sans-serif"),  # Larger font
        align="center"
    )
     # Section subtitles
    fig.add_annotation(
        x=1.0, y=.42,
        xref="paper", yref="paper",
        text="<b>Evaluation Correlations",
        showarrow=False,
        font=dict(size=22, family="Arial, sans-serif"),  # Larger font
        align="center"
    )
    
    # 7. Add correlation heatmap
    labels = ['Human', 'AI']
    print(correlation_matrix)
    fig.add_trace(
        go.Heatmap(
            z=correlation_matrix,
            x=labels,
            y=labels,
            colorscale=corr_colors,
            showscale=False,
            zmin=-1, zmax=1,
            text=[[f"{val:.2f}" for val in row] for row in correlation_matrix],
            texttemplate="%{text}",
            textfont={"size": 18},  # Larger font
            hoverinfo='text',
        ), row=4, col=3
    )
    
    # 8. Add annotations
    from scipy import stats

    # Annotation for AI bias: use t-test instead of fixed threshold
    mean_human_resp_by_ai = np.mean(human_resp_by_ai)
    mean_ai_resp_by_ai = np.mean(ai_resp_by_ai)
    # Perform t-test
    t_stat, p_value_ai = stats.ttest_ind(human_resp_by_ai, ai_resp_by_ai, equal_var=False)
    # Statistical significance (p < 0.05)
    ai_bias_significant = p_value_ai < 0.05 and mean_ai_resp_by_ai > mean_human_resp_by_ai

    if ai_bias_significant:
        fig.add_annotation(
            x=0.05, y=mean_ai_resp_by_ai-0.1,
            xref="x3", yref="y3",
            text=f"Bias towards<br>own responses<br>(p={p_value_ai:.3f})",
            showarrow=True,
            arrowhead=2,
            arrowsize=1,
            arrowwidth=2,
            arrowcolor=ai_color,
            ax=40, ay=30,
            font=dict(color=ai_color, size=16),  # Larger font
        )

    # Annotation for human impartiality: use t-test instead of fixed threshold
    mean_human_resp_by_humans = np.mean(human_resp_by_humans)
    mean_ai_resp_by_humans = np.mean(ai_resp_by_humans)
    # Perform t-test
    t_stat, p_value_humans = stats.ttest_ind(human_resp_by_humans, ai_resp_by_humans, equal_var=False)
    # Impartiality if there's no statistically significant difference (p > 0.05)
    print(p_value_humans)
    impartiality_significant = p_value_humans > 0.05

    if not impartiality_significant:
        fig.add_annotation(
            x=0, y=np.mean([mean_human_resp_by_humans, mean_ai_resp_by_humans]),
            xref="x2", yref="y2",
            text=f"Bias towards<br>own responses<br>(p={p_value_humans:.3f})",
            showarrow=True,
            arrowhead=2,
            arrowsize=1,
            arrowwidth=2,
            arrowcolor=human_color,
            ax=-40, ay=30,
            font=dict(color=human_color, size=16),  # Larger font
        )

    # Update conclusions to use t-test results
    conclusion = []

    if ai_bias_significant:
        conclusion.append(f"AI shows statistically significant bias towards AI responses (p={p_value_ai:.3f})")
    else:
        conclusion.append(f"AI does not show a statistically significant bias (p={p_value_ai:.3f})")
        
    if impartiality_significant:
        conclusion.append(f"human evaluators are impartial (p={p_value_humans:.3f})")
    else:
        conclusion.append(f"human evaluators show statistically significant bias (p={p_value_humans:.3f})")
        
    # Annotations for chance-level results
    confusion_accuracy_human = (confusion_matrix_human[0, 0] + confusion_matrix_human[1, 1]) / 2
    confusion_accuracy_ai = (confusion_matrix_ai[0, 0] + confusion_matrix_ai[1, 1]) / 2
    
    if 0.45 <= confusion_accuracy_human <= 0.55:
        fig.add_annotation(
            x=0.18, y=0.15,
            xref="paper", yref="paper",
            text="Accuracy level: Random (≈50%)",
            showarrow=False,
            font=dict(color="gray", size=16, style="italic"),  # Larger font
        )
    
    if 0.45 <= confusion_accuracy_ai <= 0.55:
        fig.add_annotation(
            x=0.5, y=0.15,
            xref="paper", yref="paper",
            text="Accuracy level: Random (≈50%)",
            showarrow=False,
            font=dict(color="gray", size=16, style="italic"),  # Larger font
        )
    
    # Annotation for correlation
    corr_value = correlation_matrix[0, 1]
    corr_text = "Low correlation" if corr_value < 0.4 else "Moderate correlation" if corr_value < 0.7 else "High correlation"
    
    fig.add_annotation(
        x=0.99, y=-0.1,
        xref="paper", yref="paper",
        text=f"{corr_text} between evaluations (r = {corr_value:.2f})",
        showarrow=False,
        font=dict(color="#e65100", size=16, style="italic"),  # Larger font
    )

    # Function to calculate metrics for confusion matrix
    def calculate_confusion_metrics(conf_matrix):
        """
        Calculate metrics from a confusion matrix.
        
        Args:
            conf_matrix: 2x2 numpy array with confusion matrix values
            
        Returns:
            Dictionary with accuracy, precision, recall, and f1_score
        """
        tp = conf_matrix[0, 0]
        fp = conf_matrix[1, 0]
        fn = conf_matrix[0, 1]
        tn = conf_matrix[1, 1]
        
        accuracy = (tp + tn) / (tp + fp + fn + tn)
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1_score = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
        
        return {
            "accuracy": accuracy,
            "precision": precision,
            "recall": recall,
            "f1_score": f1_score
        }
    
    # Calculate metrics for human confusion matrix
    metrics_human = calculate_confusion_metrics(confusion_matrix_human)

    # Add annotation with metrics for humans
    fig.add_annotation(
        x=0.10,  # x position in paper (adjust as needed)
        y=-0.16,  # y position in paper (adjust as needed)
        xref="paper",
        yref="paper",
        text=(f"<b>Metrics:</b><br>"
            f"Accuracy: {metrics_human['accuracy']:.2f}<br>"
            f"Recall: {metrics_human['recall']:.2f}<br>"
            f"F1-Score: {metrics_human['f1_score']:.2f}"),
        showarrow=False,
        font=dict(color="#004677", size=14),  # Larger font
        align="left",
        bgcolor="rgba(255,255,255,0.8)",
        bordercolor="#004677",
        borderwidth=1,
        borderpad=4,
    )

    # Calculate metrics for AI confusion matrix
    metrics_ai = calculate_confusion_metrics(confusion_matrix_ai)

    # Add annotation with metrics for AI
    fig.add_annotation(
        x=0.5,  # x position in paper (adjust as needed)
        y=-0.16,  # y position in paper (adjust as needed)
        xref="paper",
        yref="paper",
        text=(f"<b>Metrics:</b><br>"
            f"Accuracy: {metrics_ai['accuracy']:.2f}<br>"
            f"Recall: {metrics_ai['recall']:.2f}<br>"
            f"F1-Score: {metrics_ai['f1_score']:.2f}"),
        showarrow=False,
        font=dict(color="#FF7F0E", size=14),  # Larger font
        align="left",
        bgcolor="rgba(255,255,255,0.8)",
        bordercolor="#FF7F0E",
        borderwidth=1,
        borderpad=4,
    )
    
    # 10. Update layout
    fig.update_layout(
        height=height,
        width=width,
        template="plotly_white",
        showlegend=True,
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=0.65,
            xanchor="right",
            x=1.2,
            font=dict(size=18)  # Larger legend font
        ),
        margin=dict(t=20, b=150)
    )
    
    # Update subplot title font sizes
    for i in range(len(fig.layout.annotations)):
        if i >= 0 and i <= 5:  # Subplot titles
            fig.layout.annotations[i].font.size = 22  # Larger font for subplot titles
            fig.layout.annotations[i].font.family = "Arial, sans-serif"  # Add the font family as well
        
    # 11. Update axes
    # Calculate range for Y axes (show all data)
    all_scores = np.concatenate([
        human_resp_by_humans, 
        ai_resp_by_humans, 
        human_resp_by_ai, 
        ai_resp_by_ai
    ])
    
    y_min = max(0, np.min(all_scores) - 0.5)  # Never less than 0
    y_max = min(10, np.max(all_scores) + 0.5)  # Never more than 10
    
    fig.update_yaxes(
        title=dict(text="Score", font=dict(size=18)),  # Larger font
        range=[y_min, y_max], 
        row=2, col=1, 
        tickfont=dict(size=16)  # Larger tick font
    )
    
    fig.update_yaxes(
        title=dict(text="Score", font=dict(size=18)),  # Larger font
        range=[y_min, y_max], 
        row=2, col=2, 
        tickfont=dict(size=16)  # Larger tick font
    )
    
    # Remove axis labels for violins
    fig.update_xaxes(showticklabels=False, row=2, col=1)
    fig.update_xaxes(showticklabels=False, row=2, col=2)
    
    # Update confusion matrix axes with larger fonts
    for row_idx, col_idx in [(4, 1), (4, 2), (4, 3)]:
        fig.update_xaxes(
            tickfont=dict(size=16),  # Larger tick font
            row=row_idx, col=col_idx
        )
        fig.update_yaxes(
            tickfont=dict(size=16),  # Larger tick font
            row=row_idx, col=col_idx
        )
    
    # Save figure if output file is specified
    if output_file:
        try:
            if output_file.endswith('.html'):
                pio.write_html(fig, output_file)
            elif output_file.endswith(('.png', '.jpg', '.jpeg', '.svg', '.pdf')):
                pio.write_image(fig, output_file)
            else:
                print(f"Unsupported format: {output_file}")
                print("Supported formats: .html, .png, .jpg, .jpeg, .svg, .pdf")
                print("Saving as HTML instead.")
                pio.write_html(fig, output_file + '.html')
            print(f"Graphical abstract saved at: {output_file}")
        except Exception as e:
            print(f"Error saving: {e}")
            print("The figure will be displayed but not saved.")
    
    return fig

In [ ]:
fig = create_graphical_abstract(
        df_eval_human=load_data("eval"),
        df_eval_ai=pd.concat([load_data("ia_eval_2"), load_data("ia_eval")]),
        calcular_metricas=True
    )
fig.show()

In [ ]:
fig = create_graphical_abstract(
        df_eval_humans=load_data("eval"),
        df_eval_ai=load_data("ia_eval"),
        calculate_metrics=True
    )
fig.show()

In [ ]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import ttest_ind, mannwhitneyu
import numpy as np


def crear_violin_guessed_vs_real(df_eval, score_column='score', output_file=None, title="", width=1000, height=600):
    """
    Creates a violin plot with scientific style for academic publications.
    
    Args:
        df_eval: DataFrame with evaluations
        score_column: Name of the score column
        output_file: Path to save the output
        title: Short title for the plot
        width: Figure width in pixels
        height: Figure height in pixels
    
    Returns:
        fig: Plotly figure object
    """
    # Preprocess data
    try:
        df_eval['original_answer'] = df_eval["response"]
    except:
        pass
   
    try:
        df_eval, _, _ = filter_and_analyze_responses(df_eval)
    except:
        pass
    
    # Filter only entries with LLM 0, 1 or 2
    df_eval = df_eval[df_eval['LLM'].isin([0, 1, 2])].copy()
    
    # Create real source column
    df_eval['real_source'] = df_eval['LLM'].apply(
        lambda x: 'Human' if x == 0 else ('AI' if x == 1 else 'AI-CoT')
    )
    
    # Normalize predicted source column
    try:
        df_eval['predicted_source'] = df_eval['response_source'].str.strip().str.upper().replace({'AI': 'AI', 'HUMAN': 'Human'})
    except:
        try:
            df_eval['predicted_source'] = df_eval['source_guessed'].str.strip().str.capitalize()
        except:
            try:
                df_eval['predicted_source'] = df_eval['guessed_source'].str.strip().str.capitalize()
            except:
                raise ValueError("Predicted source column not found")
    
    # Ensure consistent formatting
    df_eval['predicted_source'] = df_eval['predicted_source'].replace({
        'Ai': 'AI',
        'ai': 'AI',
        'human': 'Human',
        'HUMAN': 'Human'
    })
    
    # Create combined categories
    df_eval['source_category'] = df_eval.apply(
        lambda row: f"Real: {row['real_source']}<br>Predicted: {row['predicted_source']}", 
        axis=1
    )
    
    # Define specific order for categories
    category_order = [
        "Real: Human<br>Predicted: Human",
        "Real: Human<br>Predicted: AI",
        "Real: AI<br>Predicted: Human",
        "Real: AI<br>Predicted: AI",
        "Real: AI-CoT<br>Predicted: Human",
        "Real: AI-CoT<br>Predicted: AI"
    ]
    
    # Color configuration
    color_map = {
        "Real: Human<br>Predicted: Human": '#1b9e77',
        "Real: Human<br>Predicted: AI": '#d95f02',
        "Real: AI<br>Predicted: Human": '#7570b3',
        "Real: AI<br>Predicted: AI": '#e7298a',
        "Real: AI-CoT<br>Predicted: Human": '#66a61e',
        "Real: AI-CoT<br>Predicted: AI": '#e6ab02'
    }
    
    # Calculate statistics and prepare data
    category_stats = {}
    category_data = {}
    
    for category in category_order:
        subset = df_eval[df_eval['source_category'] == category]
        if len(subset) > 0:
            scores = subset[score_column].dropna().values
            if len(scores) > 0:
                category_data[category] = scores
                category_stats[category] = {
                    'mean': np.mean(scores),
                    'max': np.max(scores),
                    'count': len(scores),
                    'pct': 100 * len(scores) / len(df_eval)
                }
    
    # Calculate t-tests
    def calculate_pvalue(data1, data2):
        """Calculate p-value using independent t-test"""
        if len(data1) < 2 or len(data2) < 2:
            return np.nan
        try:
            statistic, pvalue = ttest_ind(data1, data2)
            return pvalue
        except:
            return np.nan
    
    def format_pvalue(p):
        """Format p-value for display"""
        if np.isnan(p):
            return "N/A"
        elif p < 0.001:
            return "***"
        elif p < 0.01:
            return "**"
        elif p < 0.05:
            return "*"
        else:
            return f"ns"
    
    def format_pvalue_full(p):
        """Format full p-value for display"""
        if np.isnan(p):
            return "N/A"
        elif p < 0.001:
            return "p<0.001"
        elif p < 0.01:
            return f"p={p:.3f}"
        elif p < 0.05:
            return f"p={p:.3f}"
        else:
            return f"p={p:.3f}"
    
    # Define comparisons with their x positions
    comparisons = [
        (0, 1, "Real: Human<br>Predicted: Human", "Real: Human<br>Predicted: AI"),
        (2, 3, "Real: AI<br>Predicted: Human", "Real: AI<br>Predicted: AI"),
        (4, 5, "Real: AI-CoT<br>Predicted: Human", "Real: AI-CoT<br>Predicted: AI"),
    ]
    
    pvalue_comparisons = []
    for x1, x2, cat1, cat2 in comparisons:
        if cat1 in category_data and cat2 in category_data:
            pval = calculate_pvalue(category_data[cat1], category_data[cat2])
            pvalue_comparisons.append({
                'x1': x1,
                'x2': x2,
                'pvalue': pval,
                'formatted': format_pvalue(pval),
                'formatted_full': format_pvalue_full(pval)
            })
    
    # Create figure
    fig = go.Figure()
    
    # Add violin plots
    for category in category_order:
        if category in category_stats:
            subset = df_eval[df_eval['source_category'] == category]
            cat_stats = category_stats[category]
            
            label = f"{category}<br>n={cat_stats['count']} ({cat_stats['pct']:.1f}%)"
            
            fig.add_trace(
                go.Violin(
                    y=subset[score_column].dropna(),
                    name=label,
                    box_visible=True,
                    meanline_visible=True,
                    line_color=color_map[category],
                    opacity=0.7,
                    width=0.9,
                    points=False,
                    jitter=0
                )
            )
    
    # Calculate y position for significance brackets
    # Find the maximum value across all data
    all_max = max([category_stats[cat]['max'] for cat in category_stats])
    y_base = all_max + 0.3
    y_increment = 0.4
    
    # Update layout
    fig.update_layout(
        title={
            'text': title,
            'font': {'size': 26, 'family': 'Arial, sans-serif'},
            'x': 0.5,
            'xanchor': 'center'
        },
        xaxis={
            'title': None,
            'tickfont': {'size': 16, 'family': 'Arial, sans-serif'},
            'tickangle': 0
        },
        yaxis={
            'title': {
                'text': score_column.replace('_', ' ').title(),
                'font': {'size': 22, 'family': 'Arial, sans-serif'}
            },
            'tickfont': {'size': 18, 'family': 'Arial, sans-serif'},
            'gridcolor': 'lightgray',
            'gridwidth': 0.5,
            'range': [0, all_max + 1.5]  # Extra space for brackets
        },
        template="plotly_white",
        height=height,
        width=width,
        violinmode='group',
        showlegend=False,
        paper_bgcolor='white',
        plot_bgcolor='white',
        margin={'t': 80, 'b': 80, 'l': 80, 'r': 30}
    )
    
    # Add mean annotations and significance brackets
    annotations = []
    
    # Add mean values
    for i, category in enumerate(category_order):
        if category in category_stats:
            mean_val = category_stats[category]['mean']
            
            annotations.append(dict(
                x=i, 
                y=mean_val,
                text=f"{mean_val:.2f}",
                showarrow=True,
                arrowhead=0,
                arrowsize=1,
                arrowwidth=1,
                arrowcolor='black',
                ax=25,
                ay=0,
                font=dict(color='black', size=16, family='Arial, sans-serif'),
                bordercolor='black',
                borderwidth=1,
                bgcolor='white',
                opacity=0.8
            ))
    
    # Add significance brackets and p-values
    for comp_idx, comp in enumerate(pvalue_comparisons):
        x1, x2 = comp['x1'], comp['x2']
        y_pos = y_base + comp_idx * 0  # All at same height for cleaner look
        
        # Add bracket line (horizontal)
        fig.add_shape(
            type="line",
            x0=x1, x1=x2,
            y0=y_pos, y1=y_pos,
            line=dict(color="black", width=2)
        )
        
        # Add vertical ticks at ends
        fig.add_shape(
            type="line",
            x0=x1, x1=x1,
            y0=y_pos - 0.1, y1=y_pos,
            line=dict(color="black", width=2)
        )
        
        fig.add_shape(
            type="line",
            x0=x2, x1=x2,
            y0=y_pos - 0.1, y1=y_pos,
            line=dict(color="black", width=2)
        )
        
        # Add p-value text above bracket
        annotations.append(dict(
            x=(x1 + x2) / 2,
            y=y_pos + 0.15,
            text=f"{comp['formatted_full']}<br>{comp['formatted']}",
            showarrow=False,
            font=dict(color='black', size=14, family='Arial, sans-serif', weight='bold'),
            xanchor='center',
            yanchor='bottom'
        ))
    
    fig.update_layout(annotations=annotations)
    
    # Save figure
    if output_file:
        if output_file.endswith('.html'):
            fig.write_html(output_file)
        elif output_file.endswith(('.png', '.jpg', '.jpeg', '.svg', '.pdf')):
            fig.write_image(output_file, scale=3)
        else:
            fig.write_image(f"{output_file}.pdf", scale=3)
    
    return fig

In [ ]:
fig = crear_violin_guessed_vs_real(load_data("eval"), score_column='accuracy_score', title="Score Real source vs Predicted source.<br> Evaluator: Human", output_file="human_guessed_2.png")
fig.show()

In [ ]:
fig = crear_violin_guessed_vs_real(load_data("ia_eval"), score_column='accuracy_score', title= "Scores Real source vs Predicted source.<br> Evaluator: Llama 3.1:7b", output_file="ia_guessed_2.png")
fig.show()

In [ ]:
fig = crear_violin_guessed_vs_real(load_data("ia_deepseek_eval"), score_column='accuracy_score', title= "Puntuaciones Fuente Real vs Fuente Adivinada.<br> Evaluador: Deepseek-r1:7b", output_file="ia_guessed_ej.png")
fig.show()

In [ ]:
fig = crear_violin_guessed_vs_real(load_data("ia_phi_4_eval"), score_column='accuracy_score', title= "Puntuaciones Fuente Real vs Fuente Adivinada.<br> Evaluador: Phi4", output_file="ia_guessed_ej.png")
fig.show()

In [ ]:
fig = crear_violin_guessed_vs_real(load_data("ia_deepseek-r1_70b_eval"), score_column='accuracy_score', title= "Puntuaciones Fuente Real vs Fuente Adivinada.<br> Evaluador: Deepseek-r1:70b", output_file="ia_guessed_ej.png")
fig.show()

In [ ]:
fig = crear_violin_guessed_vs_real(load_data("ia_qwen_2_5_eval"), score_column='accuracy_score', title= "Puntuaciones Fuente Real vs Fuente Adivinada.<br> Evaluador: qwen2.5", output_file="ia_guessed_ej.png")
fig.show()

In [ ]:
import pandas as pd
import numpy as np
import spacy
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA
from textstat import flesch_reading_ease, syllable_count, lexicon_count, text_standard, automated_readability_index
import re

# Cargar los modelos de spaCy (asegúrate de tener instalado 'es_core_news_md' o 'en_core_web_md')
# !python -m spacy download es_core_news_md  # Descomentar si necesitas español
!python -m spacy download en_core_web_md   # Descomentar si necesitas inglés
nlp = spacy.load('en_core_web_md')  # Cambia a 'en_core_web_md' para inglés

# Asumiendo que df_human y df_ai son tus dataframes con las evaluaciones
# df_human = pd.read_csv('evaluaciones_humanas.csv')
# df_ai = pd.read_csv('evaluaciones_ia.csv')

def extract_linguistic_features(text):
    """Extrae características lingüísticas de un texto."""
    if pd.isna(text) or text == "":
        return {
            'word_count': 0,
            'sentence_count': 0,
            'avg_word_length': 0,
            'avg_sentence_length': 0,
            'unique_words_ratio': 0,
            'lexical_diversity': 0,
            'reading_ease': 0,
            "grade_level": 0,
            'verb_ratio': 0,
            'noun_ratio': 0,
            'adj_ratio': 0,
            'adv_ratio': 0,
            'pronoun_ratio': 0,
            'named_entity_count': 0,
            'dependency_distance': 0,
            'connector_count': 0
        }
    
    doc = nlp(text)
   
    # Conteos básicos
    words = [token.text for token in doc if not token.is_punct and not token.is_space]
    
    word_count = len(words)
    sentence_count = len(list(doc.sents))
    
    if word_count == 0 or sentence_count == 0:
        return {
            'word_count': 0,
            'sentence_count': 0,
            'avg_word_length': 0,
            'avg_sentence_length': 0,
            'unique_words_ratio': 0,
            'lexical_diversity': 0,
            'reading_ease': 0,
            "grade_level": 0,
            'readind_ease': 0, 
            'verb_ratio': 0,
            'noun_ratio': 0,
            'adj_ratio': 0,
            'adv_ratio': 0,
            'pronoun_ratio': 0,
            'named_entity_count': 0,
            'dependency_distance': 0,
            'connector_count': 0
        }
    
    # Promedios y ratios
    avg_word_length = sum(len(word) for word in words) / word_count
    avg_sentence_length = word_count / sentence_count
    unique_words = set(word.lower() for word in words)
    unique_words_ratio = len(unique_words) / word_count
    
    # Tipo de palabras
    pos_counts = Counter([token.pos_ for token in doc])
    verb_count = pos_counts.get('VERB', 0)
    noun_count = pos_counts.get('NOUN', 0)
    adj_count = pos_counts.get('ADJ', 0)
    adv_count = pos_counts.get('ADV', 0)
    pronoun_count = pos_counts.get('PRON', 0)
    
    # Entidades nombradas
    named_entity_count = len(doc.ents)
    
    # Distancia de dependencia (promedio de la distancia entre tokens relacionados)
    dependency_distances = [abs(token.i - token.head.i) for token in doc if token.head is not token]
    avg_dependency_distance = sum(dependency_distances) / len(dependency_distances) if dependency_distances else 0
    
    # Conteo de conectores (ajustar según el idioma)
    connectors = ['however', 'therefore', 'furthermore', 'moreover', 'on the other hand', 
                  'consequently', 'although', 'because', 'since', 'thus', 'nevertheless', 
                  'in addition', 'as a result', 'despite', 'in contrast', 'for example',
                  'in fact', 'specifically', 'in conclusion', 'in summary']
    connector_count = sum(1 for conn in connectors if conn in text.lower())
    
    # Métricas de legibilidad
    reading_ease = flesch_reading_ease(text)  # Nota: esto funciona mejor en inglés
    grade = automated_readability_index(text)
    return {
        'word_count': word_count,
        'sentence_count': sentence_count,
        'avg_word_length': avg_word_length,
        'avg_sentence_length': avg_sentence_length,
        'unique_words_ratio': unique_words_ratio,
        'lexical_diversity': len(unique_words) / (word_count ** 0.5),  # MTLD normalizado
        'reading_ease': reading_ease,
        "grade_level": grade,
        'verb_ratio': verb_count / word_count,
        'noun_ratio': noun_count / word_count,
        'adj_ratio': adj_count / word_count,
        'adv_ratio': adv_count / word_count,
        'pronoun_ratio': pronoun_count / word_count,
        'named_entity_count': named_entity_count,
        'dependency_distance': avg_dependency_distance,
        'connector_count': connector_count
    }

def analyze_dataframe(df):
    """Analiza características lingüísticas para un dataframe."""
    # Añadir columnas para cada característica lingüística
    linguistic_features = df['response'].apply(extract_linguistic_features)
    
    # Convertir la lista de diccionarios en columnas
    for feature in linguistic_features.iloc[0].keys():
        df[feature] = linguistic_features.apply(lambda x: x[feature])
    
    # Agrupar por fuente (LLM) y calcular medias
    feature_means = df.groupby('LLM')[list(linguistic_features.iloc[0].keys())].mean()
    
    # Analizar correlación entre características y puntuación
    correlations = {}
    for feature in linguistic_features.iloc[0].keys():
        correlations[feature] = df[[feature, 'accuracy_score']].corr().iloc[0, 1]
    
    return feature_means, correlations

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

def plot_feature_comparison(feature_means):
    """Generate comparative plots for linguistic features using Plotly with each feature in a separate subplot."""
    
    # Create response type labels mapping
    label_mapping = {0: 'Human', 1: 'AI', 2: 'AI CoT'}
    
    # Color palette
    colors = {
        'Human': 'rgb(31, 119, 180)',
        'AI': 'rgb(255, 127, 14)',
        'AI CoT': 'rgb(44, 160, 44)'
    }
    
    # Map the index to readable labels
    feature_means_labeled = feature_means.copy()
    feature_means_labeled.index = [label_mapping.get(idx, f'Type {idx}') for idx in feature_means.index]
    
    # Get features and clean their names for display
    features = list(feature_means.columns)
    feature_labels = [feature.replace('_', ' ').title() for feature in features]
    
    # Calculate subplot layout
    n_features = len(features)
    cols = 4
    rows = (n_features + cols - 1) // cols
    
    # Create subplots
    fig = make_subplots(
        rows=rows, cols=cols,
        subplot_titles=feature_labels,
        vertical_spacing=0.08,
        horizontal_spacing=0.08
    )
    
    # Add bars for each feature
    for i, feature in enumerate(features):
        row = i // cols + 1
        col = i % cols + 1
        
        # Add bars for each response type
        for j, response_type in enumerate(feature_means_labeled.index):
            fig.add_trace(
                go.Bar(
                    x=[response_type],
                    y=[feature_means_labeled.loc[response_type, feature]],
                    name=response_type,
                    marker_color=colors[response_type],
                    text=[f'{feature_means_labeled.loc[response_type, feature]:.3f}'],
                    textposition='outside',
                    textfont=dict(size=18, color='black'),
                    opacity=0.8,
                    showlegend=(i == 0),  # Only show legend for first subplot
                    legendgroup=response_type
                ),
                row=row, col=col
            )
        
        # Calculate y-axis range with extra space for text labels
        max_val = feature_means_labeled[feature].max()
        min_val = feature_means_labeled[feature].min()
        if min_val >= 0:
            y_range = [0, max_val * 1.15]  # Add 15% extra space at the top
        else:
            y_range = [min_val * 1.3, max_val * 1.15]  # Add 15% extra space at the top
        # Update axes for each subplot
        fig.update_xaxes(
            tickfont=dict(size=18, color='black', family='Arial'),
            showgrid=False,
            gridcolor='lightgray',
            gridwidth=0,
            showline=True,
            linewidth=2,
            linecolor='black',
            showticklabels=False,
            row=row, col=col
        )
        fig.update_yaxes(
            range=y_range,
            tickfont=dict(size=18, color='black', family='Arial'),
            showgrid=False,
            gridcolor='lightgray',
            gridwidth=0,
            showline=True,
            linewidth=2,
            linecolor='black',
            showticklabels=False,
            row=row, col=col
        )
    
    # Update layout for paper-like appearance
    fig.update_layout(
        title=dict(
            text='<b>Linguistic Feature Comparison Across Response Types</b>',
            x=0.5,
            y=1,
            font=dict(size=30, color='black', family='Arial')
        ),
        legend=dict(
            font=dict(size=22, color='black', family='Arial'),
            bgcolor='rgba(255,255,255,0.9)',
            bordercolor='black',
            borderwidth=1,
            x=1.02,
            y=1.02,
        ),
        plot_bgcolor='white',
        paper_bgcolor='white',
        font=dict(family='Arial', size=20, color='black'),
        width=1000,
        height=250 * rows,
        margin=dict(l=80, r=150, t=100, b=80)
    )
    
    # Update subplot titles font and position
    for annotation in fig['layout']['annotations']:
        annotation['font'] = dict(size=20, color='black', family='Arial')
        annotation['y'] = annotation['y'] + 0.01  # Move titles up
    
    fig.show()
    return fig

def analyze_text_complexity(df):
    """Analiza y compara la complejidad del texto entre diferentes fuentes."""
    # Combinar ambos dataframes
    df_combined = df
    
    # Extraer características para cada respuesta
    feature_means, correlations = analyze_dataframe(df_combined)
    
    # Visualizar diferencias
    fig = plot_feature_comparison(feature_means)
    
    # Analizar correlaciones entre características y puntuaciones
    sorted_correlations = sorted(correlations.items(), key=lambda x: abs(x[1]), reverse=True)
    
    # Análisis de TF-IDF para términos distintivos
    tfidf = TfidfVectorizer(max_features=100, stop_words='english') 
    
    results = {
        'feature_means': feature_means,
        'correlations': sorted_correlations,
        'visualization': fig
    }
    
    # Para cada tipo de fuente, identificar términos característicos
    source_terms = {}
    for source in df_combined['LLM'].unique():
        source_texts = df_combined[df_combined['LLM'] == source]['response'].dropna()
        if len(source_texts) > 0:
            try:
                source_matrix = tfidf.fit_transform(source_texts)
                feature_names = tfidf.get_feature_names_out()
                
                # Obtener los términos más importantes para esta fuente
                if source_matrix.shape[0] > 0:
                    importance = np.asarray(source_matrix.mean(axis=0)).flatten()
                    top_indices = importance.argsort()[-10:][::-1]
                    top_terms = [(feature_names[i], importance[i]) for i in top_indices]
                    source_terms[source] = top_terms
            except:
                source_terms[source] = []
    
    results['distinctive_terms'] = source_terms
    
    return results

# Función principal para ejecutar el análisis
def run_linguistic_analysis(df):
    print("Iniciando análisis lingüístico comparativo...")
    results = analyze_text_complexity(df)
    
    print("\nCaracterísticas lingüísticas promedio por fuente:")
    print(results['feature_means'])
    
    print("\nCorrelaciones entre características lingüísticas y puntuaciones:")
    for feature, corr in results['correlations']:
        print(f"{feature}: {corr:.4f}")
    
    print("\nTérminos distintivos por fuente:")
    for source, terms in results['distinctive_terms'].items():
        print(f"\nFuente {source}:")
        for term, importance in terms:
            print(f"  - {term}: {importance:.4f}")
    
    return results

In [ ]:
def plot_feature_comparison(feature_means):
    """Generate radar chart for linguistic features comparison across response types."""
    
    # Create response type labels mapping
    label_mapping = {0: 'Human', 1: 'AI', 2: 'AI CoT'}
    
    # Color palette
    colors = {
        'Human': 'rgb(31, 119, 180)',
        'AI': 'rgb(255, 127, 14)',
        'AI CoT': 'rgb(44, 160, 44)'
    }
    
    # Map the index to readable labels
    feature_means_labeled = feature_means.copy()
    feature_means_labeled.index = [label_mapping.get(idx, f'Type {idx}') for idx in feature_means.index]
    
    # Get features and clean their names for display
    features = list(feature_means.columns)
    
    # Group features by category for better visual organization
    feature_groups = {
        'Basic Metrics': ['word_count', 'sentence_count', 'avg_word_length', 'avg_sentence_length'],
        'Lexical Features': ['unique_words_ratio', 'lexical_diversity'],
        'Readability': ['reading_ease', 'grade_level'],
        'Part-of-Speech': ['verb_ratio', 'noun_ratio', 'adj_ratio', 'adv_ratio', 'pronoun_ratio'],
        'Complexity': ['named_entity_count', 'dependency_distance', 'connector_count']
    }
    
    # Reorder features by groups for better visual flow
    ordered_features = []
    for group in feature_groups.values():
        for feature in group:
            if feature in features:
                ordered_features.append(feature)
    
    # Add any remaining features not in groups
    for feature in features:
        if feature not in ordered_features:
            ordered_features.append(feature)
    
    # Create ordered feature labels
    feature_labels = [feature.replace('_', ' ').title() for feature in ordered_features]
    
    # Normalize the data to 0-1 scale for better radar visualization
    # Apply MinMaxScaler separately for each feature with min=0
    feature_means_normalized = feature_means_labeled.copy()
    for feature in ordered_features:
        max_val = feature_means_labeled[feature].max()
        min_val = 0  # Always use 0 as minimum
        feature_means_normalized[feature] = (feature_means_labeled[feature] - min_val) / (max_val - min_val)
    
    # Create the radar chart
    fig = go.Figure()
    
    # Add trace for each response type
    for response_type in feature_means_normalized.index:
        values = [feature_means_normalized.loc[response_type, feature] for feature in ordered_features]
        original_values = [feature_means_labeled.loc[response_type, feature] for feature in ordered_features]
        # Close the radar chart by adding the first value at the end
        values += [values[0]]
        original_values += [original_values[0]]
        theta_labels = feature_labels + [feature_labels[0]]
        
        fig.add_trace(go.Scatterpolar(
            r=values,
            theta=theta_labels,
            fill='toself',
            name=response_type,
            line=dict(color=colors[response_type], width=3),
            fillcolor=colors[response_type].replace('rgb', 'rgba').replace(')', ', 0.1)'),
            marker=dict(
                size=8,
                color=colors[response_type],
                line=dict(color='white', width=2)
            ),
            hovertemplate=(
                f'<b>{response_type}</b><br>' +
                'Feature: %{theta}<br>' +
                'Normalized Value: %{r:.3f}<br>' +
                'Original Value: %{customdata:.3f}<extra></extra>'
            ),
            customdata=original_values
        ))
    
    # Update layout for paper-like appearance
    fig.update_layout(
        title=dict(
            text='<b>Linguistic Features Profile Comparison</b>',
            x=0.5,
            font=dict(size=24, color='black', family='Arial')
        ),
        polar=dict(
            radialaxis=dict(
                visible=True,
                range=[0, 1],
                tickfont=dict(size=14, color='black', family='Arial'),
                gridcolor='lightgray',
                gridwidth=1,
                linecolor='black',
                linewidth=1
            ),
            angularaxis=dict(
                tickfont=dict(size=14, color='black', family='Arial'),
                linecolor='black',
                linewidth=1,
                gridcolor='lightgray',
                gridwidth=1
            ),
            bgcolor='white'
        ),
        legend=dict(
            font=dict(size=16, color='black', family='Arial'),
            bgcolor='rgba(255,255,255,0.9)',
            bordercolor='black',
            borderwidth=1,
            x=1.05,
            y=1
        ),
        font=dict(family='Arial', size=14, color='black'),
        width=900,
        height=800,
        margin=dict(l=80, r=150, t=100, b=80),
        plot_bgcolor='white',
        paper_bgcolor='white'
    )
    
    # Add annotation explaining normalization
    fig.add_annotation(
        text="<i>Note: Values normalized to 0-1 scale for comparison</i>",
        xref="paper", yref="paper",
        x=0.5, y=-0.05,
        showarrow=False,
        font=dict(size=12, color='gray', family='Arial'),
        xanchor='center'
    )
    fig.show()
    return fig

In [ ]:
# Cargar los datos
answers = load_data("answers")
evaluations = load_data("ia_eval")

# Convertir los IDs a string para asegurar compatibilidad
answers["_id"] = answers["_id"].astype(str)
evaluations["original_answer_id"] = evaluations["original_answer_id"].astype(str)

# Método 1: Usando merge con todas las columnas necesarias
evaluations_with_answers = evaluations.merge(
    answers[['_id', 'answer', 'bibliography_sources', 'confidence_level']], 
    left_on='original_answer_id',
    right_on='_id',
    how='left'
)

# Función para combinar respuesta y bibliografía cuando esté disponible
def combine_answer_and_bibliography(row):
    if pd.notna(row['bibliography_sources']):
        return row['answer'] + '\n\nReferences: ' + row['bibliography_sources']
    else:
        return row['answer']

# Crear la columna original_answer con la combinación de respuesta y bibliografía
evaluations_with_answers['original_answer'] = evaluations_with_answers.apply(combine_answer_and_bibliography, axis=1)


# Verificar el resultado
print("Primeras filas del DataFrame final:")
print(evaluations_with_answers.head())
evaluations_with_answers["response"] = evaluations_with_answers["original_answer"]
evaluations_with_answers_ia, _ , _ = filter_and_analyze_responses(evaluations_with_answers)


In [ ]:

results = run_linguistic_analysis(evaluations_with_answers_ia)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from sklearn.preprocessing import MinMaxScaler

def create_correlation_heatmap_scaled(df):
    """
    Crea un heatmap de correlaciones con características escaladas entre 0 y 1.
    Categorías: Conteo, Superficie y Estructura.
    
    Args:
        df: DataFrame con características lingüísticas y puntuaciones
        
    Returns:
        Dict con resultados y visualizaciones
    """
    # Identificar todas las columnas de puntuación
    score_columns = ["accuracy_score", "clarity_score", "completeness_score"]
    
    # Definir las categorías de características
    count_features = ['word_count', 'sentence_count']
    
    surface_features = [
        'avg_word_length', 'avg_sentence_length', 'reading_ease', 'grade_level',
        'connector_count'
    ]
    
    structure_features = [
        'unique_words_ratio', 'lexical_diversity',
        'verb_ratio', 'noun_ratio', 'adj_ratio', 'adv_ratio', 'pronoun_ratio',
        'named_entity_count', 'dependency_distance'
    ]
    
    # Todas las características
    all_features = count_features + surface_features + structure_features
    
    # Verificar qué características están disponibles
    available_features = [feature for feature in all_features if feature in df.columns]
    
    if not available_features:
        print("No se encontraron características lingüísticas en el dataframe.")
        return None
    
    if not score_columns:
        print("No se encontraron columnas de puntuación (_score) en el dataframe.")
        return None
    
    # Crear una copia del dataframe para no modificar el original
    df_scaled = df.copy()
    
    # Escalar todas las características lingüísticas entre 0 y 1
    print("Escalando características lingüísticas entre 0 y 1...")
    scaler = MinMaxScaler()
    if len(available_features) > 0:
        df_scaled[available_features] = scaler.fit_transform(df[available_features])
    
    # Calcular matriz de correlación con valores escalados
    correlation_matrix = df_scaled[available_features + score_columns].corr()
    
    # Ordenar características por categoría para visualización
    ordered_features = []
    for feature_group in [count_features, surface_features, structure_features]:
        ordered_features.extend([f for f in feature_group if f in available_features])
    
    # Extraer subconjunto de la matriz de correlación
    correlation_subset = correlation_matrix.loc[ordered_features, score_columns]
    
    # Crear figura para el heatmap
    plt.figure(figsize=(12, len(available_features) / 2))
    
    # Crear heatmap con anotaciones
    heatmap = sns.heatmap(
        correlation_subset, 
        annot=True,
        cmap='coolwarm',
        vmin=-1,
        vmax=1,
        center=0,
        fmt='.2f',
        linewidths=.5
    )
    
    # Añadir líneas para separar categorías
    count_end = sum(1 for feature in count_features if feature in ordered_features)
    surface_end = count_end + sum(1 for feature in surface_features if feature in ordered_features)
    
    plt.axhline(y=count_end, color='black', linewidth=2)
    plt.axhline(y=surface_end, color='black', linewidth=2)
    
    # Añadir etiquetas de categoría
    plt.text(-2.5, count_end / 2, 'CONTEO', verticalalignment='center', fontsize=12, fontweight='bold')
    plt.text(-2.5, count_end + (surface_end - count_end) / 2, 'SUPERFICIE', verticalalignment='center', fontsize=12, fontweight='bold')
    plt.text(-2.5, surface_end + (len(ordered_features) - surface_end) / 2, 'ESTRUCTURA', verticalalignment='center', fontsize=12, fontweight='bold')
    
    plt.title('Correlaciones entre Características Lingüísticas Escaladas y Puntuaciones', fontsize=16)
    plt.tight_layout()
    
    # Calcular correlaciones promedio por grupo
    avg_correlations = {
        'Conteo': pd.DataFrame({
            score: [abs(correlation_subset.loc[feature, score]) for feature in count_features if feature in available_features]
            for score in score_columns
        }).mean(),
        
        'Superficie': pd.DataFrame({
            score: [abs(correlation_subset.loc[feature, score]) for feature in surface_features if feature in available_features]
            for score in score_columns
        }).mean(),
        
        'Estructura': pd.DataFrame({
            score: [abs(correlation_subset.loc[feature, score]) for feature in structure_features if feature in available_features]
            for score in score_columns
        }).mean()
    }
    
    # Crear figura para el heatmap de correlaciones promedio
    fig_avg, ax_avg = plt.subplots(figsize=(10, 6))
    avg_df = pd.DataFrame(avg_correlations)
    
    sns.heatmap(avg_df.T, annot=True, cmap='coolwarm', fmt='.3f', linewidths=.5, ax=ax_avg, vmin=-1, vmax=1)
    ax_avg.set_title('Correlación Promedio por Categoría de Características Escaladas', fontsize=14)
    
    # Crear gráfico de barras para comparación visual
    fig_bar, ax_bar = plt.subplots(figsize=(12, 6))
    avg_df.plot(kind='bar', ax=ax_bar)
    ax_bar.set_title('Correlación Promedio por Tipo de Puntuación (Características Escaladas)', fontsize=14)
    ax_bar.set_ylabel('Correlación Promedio (valor absoluto)')
    ax_bar.set_xlabel('Tipo de Puntuación')
    plt.xticks(rotation=45)
    plt.tight_layout()
    
    # Crear un análisis explicativo de los resultados
    explanations = []
    for score in score_columns:
        category_correlations = {cat: val for cat, val in zip(avg_df.columns, avg_df.loc[score])}
        top_category = max(category_correlations, key=category_correlations.get)
        
        explanation = f"Para {score}, la categoría con mayor correlación es {top_category} ({category_correlations[top_category]:.3f}). "
        
        # Información sobre las características específicas con mayor correlación
        top_features = correlation_subset[score].abs().sort_values(ascending=False).head(3)
        explanation += f"Las características más correlacionadas son: "
        
        for i, (feature, corr_val) in enumerate(zip(top_features.index, correlation_subset.loc[top_features.index, score])):
            # Determinar a qué categoría pertenece esta característica
            if feature in count_features:
                cat = "Conteo"
            elif feature in surface_features:
                cat = "Superficie"
            else:
                cat = "Estructura"
                
            explanation += f"{feature} ({cat}, {corr_val:.3f})"
            if i < len(top_features) - 1:
                explanation += ", "
        
        explanations.append(explanation)
    
    # Análisis de dominancia entre categorías
    dominance_analysis = ""
    all_scores_avg = avg_df.mean()
    dominant_category = all_scores_avg.idxmax()
    
    dominance_analysis += f"\nEn promedio para todas las puntuaciones, la categoría {dominant_category} "
    dominance_analysis += f"tiene la mayor correlación ({all_scores_avg[dominant_category]:.3f}), "
    
    for cat in all_scores_avg.index:
        if cat != dominant_category:
            diff = all_scores_avg[dominant_category] - all_scores_avg[cat]
            dominance_analysis += f"superando a {cat} por {diff:.3f} ({all_scores_avg[cat]:.3f}). "
    
    return {
        'heatmap_fig': plt.gcf(),
        'avg_heatmap_fig': fig_avg,
        'bar_fig': fig_bar,
        'correlation_matrix': correlation_subset,
        'avg_correlations': avg_df,
        'explanations': explanations,
        'dominance_analysis': dominance_analysis,
        'df_scaled': df_scaled
    }
# Función para imprimir un resumen comprensible de los resultados
def print_correlation_analysis_summary(results):
    """
    Imprime un resumen legible de los resultados del análisis de correlación.
    """
    print("\n===== RESUMEN DEL ANÁLISIS DE CORRELACIÓN =====\n")
    
    print("CORRELACIONES PROMEDIO POR CATEGORÍA:")
    print(results['avg_correlations'])
    
    print("\nANÁLISIS POR TIPO DE PUNTUACIÓN:")
    for explanation in results['explanations']:
        print(f"\n• {explanation}")
    
    print("\nANÁLISIS DE DOMINANCIA GENERAL:")
    print(results['dominance_analysis'])
    
    print("\nINTERPRETACIÓN:")
    if 'Superficie' in results['avg_correlations'].columns and results['avg_correlations'].mean()['Superficie'] > results['avg_correlations'].mean()['Estructura']:
        print("Los resultados sugieren que las evaluaciones están más influenciadas por características superficiales")
        print("(longitud de palabras, legibilidad, uso de conectores) que por la estructura lingüística más profunda.")
        print("Esto apoya la hipótesis de que la forma puede tener más peso que el contenido en las evaluaciones.")
    else:
        print("Los resultados sugieren que las características estructurales del lenguaje tienen")
        print("mayor influencia en las evaluaciones que las características superficiales.")
        print("Esto no apoya completamente la hipótesis de que la forma domina sobre el contenido.")
    
    print("\nNOTA IMPORTANTE:")
    print("Este análisis se limita a características lingüísticas cuantificables y no mide")
    print("aspectos semánticos profundos como precisión factual, coherencia lógica o relevancia.")
    print("Un análisis completo de 'forma vs. contenido' requeriría métricas adicionales de calidad de contenido.")



def prepare_dataframe_for_analysis(df, text_column='response'):
    """
    Prepara el dataframe extrayendo todas las características lingüísticas necesarias.
    
    Args:
        df: DataFrame con las respuestas y puntuaciones
        text_column: Nombre de la columna que contiene el texto a analizar
        
    Returns:
        DataFrame con características lingüísticas añadidas
    """
    import pandas as pd
    import numpy as np
    from tqdm import tqdm  # Para mostrar una barra de progreso
    
    print(f"Extrayendo características lingüísticas de {len(df)} respuestas...")
    
    # Crear una copia para no modificar el original
    processed_df = df.copy()
    
    # Extraer características lingüísticas con barra de progreso
    features_list = []
    for i, text in tqdm(enumerate(processed_df[text_column]), total=len(processed_df)):
        try:
            features = extract_linguistic_features(text)
            features_list.append(features)
        except Exception as e:
            print(f"Error procesando fila {i}: {e}")
            # Proporcionar un diccionario de características vacío en caso de error
            features_list.append({
                'word_count': 0, 'sentence_count': 0, 'avg_word_length': 0,
                'avg_sentence_length': 0, 'unique_words_ratio': 0, 'lexical_diversity': 0,
                'reading_ease': 0, 'grade_level': 0, 'verb_ratio': 0, 'noun_ratio': 0,
                'adj_ratio': 0, 'adv_ratio': 0, 'pronoun_ratio': 0, 'named_entity_count': 0,
                'dependency_distance': 0, 'connector_count': 0
            })
    
    # Convertir la lista de diccionarios en columnas
    for feature in features_list[0].keys():
        processed_df[feature] = [features.get(feature, 0) for features in features_list]
    
    print("Extracción de características completada.")
    return processed_df

def run_complete_correlation_analysis(df, text_column='response', save_figures=True):
    """
    Ejecuta un análisis completo de correlación entre características lingüísticas escaladas y puntuaciones.
    
    Args:
        df: DataFrame con las respuestas y puntuaciones
        text_column: Nombre de la columna que contiene el texto a analizar
        save_figures: Si se deben guardar las figuras en archivos
        
    Returns:
        Dict con resultados y figuras
    """
    # 1. Preparar el dataframe añadiendo características lingüísticas
    print("Paso 1: Extrayendo características lingüísticas...")
    df_with_features = prepare_dataframe_for_analysis(df, text_column=text_column)
    
    # 2. Crear heatmap y análisis de correlación con valores escalados
    print("Paso 2: Escalando características y generando matrices de correlación...")
    correlation_results = create_correlation_heatmap_scaled(df_with_features)
    
    if correlation_results is None:
        print("Error al generar el análisis de correlación.")
        return None
    
    # 3. Identificar las características más predictivas para cada puntuación
    print("Paso 3: Analizando características más predictivas por tipo de puntuación...")
    score_columns = [col for col in df.columns if col.endswith('_score')]
    
    # 4. Imprimir resumen de hallazgos
    print("\n===== RESUMEN DEL ANÁLISIS DE CORRELACIÓN (CARACTERÍSTICAS ESCALADAS) =====\n")
    
    # Mostrar la tabla de correlaciones promedio
    print("CORRELACIONES PROMEDIO POR CATEGORÍA:")
    print(correlation_results['avg_correlations'])
    
    # Mostrar análisis por tipo de puntuación
    print("\nANÁLISIS POR TIPO DE PUNTUACIÓN:")
    for explanation in correlation_results['explanations']:
        print(f"\n• {explanation}")
    
    # Mostrar análisis de dominancia general
    print("\nANÁLISIS DE DOMINANCIA GENERAL:")
    print(correlation_results['dominance_analysis'])
    
    # Interpretación de los resultados
    print("\nINTERPRETACIÓN:")
    avg_corrs = correlation_results['avg_correlations'].mean()
    
    if 'Superficie' in avg_corrs.index and 'Estructura' in avg_corrs.index:
        if avg_corrs['Superficie'] > avg_corrs['Estructura']:
            print("Los resultados con características escaladas sugieren que las evaluaciones están más")
            print("influenciadas por características superficiales (longitud de palabras, legibilidad,")
            print("uso de conectores) que por la estructura lingüística más profunda.")
            print("Esto apoya la hipótesis de que la forma puede tener más peso que el contenido.")
        else:
            print("Los resultados con características escaladas sugieren que las características")
            print("estructurales del lenguaje tienen mayor influencia en las evaluaciones que las")
            print("características superficiales. Esto no apoya completamente la hipótesis de que")
            print("la forma domina sobre el contenido.")
    
    print("\nNOTA SOBRE EL ESCALADO:")
    print("Al escalar todas las características entre 0 y 1, hemos eliminado las diferencias en")
    print("magnitud y unidades entre las distintas características, lo que permite una comparación")
    print("más equitativa de la importancia relativa de cada característica en las puntuaciones.")
    
    # 5. Guardar figuras si se solicitó
    if save_figures:
        correlation_results['heatmap_fig'].savefig('correlation_heatmap_scaled.png', dpi=300, bbox_inches='tight')
        correlation_results['avg_heatmap_fig'].savefig('avg_correlation_by_category_scaled.png', dpi=300, bbox_inches='tight')
        correlation_results['bar_fig'].savefig('correlation_by_score_category_scaled.png', dpi=300, bbox_inches='tight')
        print("\nFiguras guardadas como:")
        print("- correlation_heatmap_scaled.png")
        print("- avg_correlation_by_category_scaled.png")
        print("- correlation_by_score_category_scaled.png")
    
    return {
        'df_with_features': df_with_features,
        'df_scaled': correlation_results['df_scaled'],
        'correlation_results': correlation_results,
        'avg_correlations': correlation_results['avg_correlations']
    }

In [ ]:
results = run_complete_correlation_analysis(evaluations_with_answers_ia, text_column='response')

# Mostrar los gráficos interactivamente
plt.show()

In [ ]:
# Cargar los datos
answers = load_data("answers")
evaluations = load_data("eval")

# Convertir los IDs a string para asegurar compatibilidad
answers["_id"] = answers["_id"].astype(str)
evaluations["response"] = evaluations["original_answer"]
evaluations["original_answer_id"] = evaluations["original_answer_id"].astype(str)

# Método 1: Usando merge con todas las columnas necesarias
evaluations_with_answers = evaluations.merge(
    answers[['_id', 'answer', 'bibliography_sources', 'confidence_level']], 
    left_on='original_answer_id',
    right_on='_id',
    how='left'
)

# Función para combinar respuesta y bibliografía cuando esté disponible
def combine_answer_and_bibliography(row):
    if pd.notna(row['bibliography_sources']):
        return row['answer'] + '\n\nReferences: ' + row['bibliography_sources']
    else:
        return row['answer']

# Crear la columna original_answer con la combinación de respuesta y bibliografía
evaluations_with_answers['original_answer'] = evaluations_with_answers.apply(combine_answer_and_bibliography, axis=1)


# Verificar el resultado
print("Primeras filas del DataFrame final:")
print(evaluations_with_answers.head())
evaluations_with_answers["response"] = evaluations_with_answers["original_answer"]
evaluations_with_answers, _ , _ = filter_and_analyze_responses(evaluations_with_answers)

In [ ]:
results = run_linguistic_analysis(evaluations_with_answers)

In [ ]:
results = run_complete_correlation_analysis(evaluations_with_answers, text_column='response')

# Mostrar los gráficos interactivamente
plt.show()

In [ ]:
# Cargar los datos
answers = load_data("answers")
evaluations = load_data("ia_deepseek_eval")

# Convertir los IDs a string para asegurar compatibilidad
answers["_id"] = answers["_id"].astype(str)
evaluations["original_answer_id"] = evaluations["original_answer_id"].astype(str)

# Método 1: Usando merge con todas las columnas necesarias
evaluations_with_answers = evaluations.merge(
    answers[['_id', 'answer', 'bibliography_sources']], 
    left_on='original_answer_id',
    right_on='_id',
    how='left'
)

# Función para combinar respuesta y bibliografía cuando esté disponible
def combine_answer_and_bibliography(row):
    if pd.notna(row['bibliography_sources']):
        return row['answer'] + '\n\nReferences: ' + row['bibliography_sources']
    else:
        return row['answer']

# Crear la columna original_answer con la combinación de respuesta y bibliografía
evaluations_with_answers['original_answer'] = evaluations_with_answers.apply(combine_answer_and_bibliography, axis=1)


# Verificar el resultado
print("Primeras filas del DataFrame final:")
print(evaluations_with_answers.head())
evaluations_with_answers["response"] = evaluations_with_answers["original_answer"]
evaluations_with_answers, _ , _ = filter_and_analyze_responses(evaluations_with_answers)

In [ ]:
results = run_complete_correlation_analysis(evaluations_with_answers, text_column='response')

# Mostrar los gráficos interactivamente
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from sklearn.preprocessing import MinMaxScaler

def create_correlation_heatmap_multi_df(dataframes, df_names, text_column='response'):
    """
    Crea un heatmap de correlaciones con múltiples dataframes, analizando cada uno por separado.
    
    Args:
        dataframes: Lista de DataFrames con respuestas y puntuaciones
        df_names: Lista de nombres para identificar cada dataframe
        text_column: Nombre de la columna que contiene el texto a analizar
        
    Returns:
        Dict con resultados y visualizaciones
    """
    # Verificar que los parámetros sean correctos
    if len(dataframes) != len(df_names):
        raise ValueError("El número de dataframes debe ser igual al número de nombres proporcionados")
    
    # Preparar cada dataframe y extraer características
    processed_dfs = []
    for i, df in enumerate(dataframes):
        print(f"Procesando dataframe {df_names[i]}...")
        processed_df = prepare_dataframe_for_analysis(df, text_column=text_column)
        processed_dfs.append(processed_df)
    
    # Definir las categorías de características
    count_features = ['word_count', 'sentence_count']
    
    surface_features = [
        'avg_word_length', 'avg_sentence_length', 'reading_ease', 'grade_level',
        'connector_count'
    ]
    
    structure_features = [
        'unique_words_ratio', 'lexical_diversity',
        'verb_ratio', 'noun_ratio', 'adj_ratio', 'adv_ratio', 'pronoun_ratio',
        'named_entity_count', 'dependency_distance'
    ]
    
    # Todas las características
    all_features = count_features + surface_features + structure_features
    
    # Analizar cada dataframe por separado
    all_correlations = []
    prefixed_score_columns = []
    
    for i, df in enumerate(processed_dfs):
        prefix = df_names[i]
        
        # Verificar qué características están disponibles
        available_features = [f for f in all_features if f in df.columns]
        
        # Verificar qué puntuaciones están disponibles
        score_columns = ["accuracy_score", "clarity_score", "completeness_score"]
        available_scores = [s for s in score_columns if s in df.columns]
        
        if not available_features or not available_scores:
            print(f"No se encontraron suficientes datos en {prefix}. Omitiendo...")
            continue
        
        # Escalar características
        df_scaled = df.copy()
        scaler = MinMaxScaler()
        df_scaled[available_features] = scaler.fit_transform(df[available_features])
        
        # Calcular correlaciones
        corr_matrix = df_scaled[available_features + available_scores].corr()
        feature_score_corr = corr_matrix.loc[available_features, available_scores]
        
        # Guardar con prefijos para cada score
        for score in available_scores:
            prefixed_score = f"{prefix}_{score}"
            prefixed_score_columns.append(prefixed_score)
            
            # Guardar correlaciones para esta puntuación
            all_correlations.append(
                pd.DataFrame(
                    feature_score_corr[score].values, 
                    index=available_features,
                    columns=[prefixed_score]
                )
            )
    
    # Combinar todas las correlaciones
    if not all_correlations:
        print("No se pudieron calcular correlaciones para ningún dataframe.")
        return None
    
    combined_corr = pd.concat(all_correlations, axis=1)
    
    # Verificar características disponibles en el resultado combinado
    available_features = combined_corr.index.tolist()
    
    # Ordenar características por categoría para visualización
    ordered_features = []
    for feature_group in [count_features, surface_features, structure_features]:
        ordered_features.extend([f for f in feature_group if f in available_features])
    
    # Crear figura para el heatmap principal
    plt.figure(figsize=(len(prefixed_score_columns) * 1.5, len(ordered_features) * 0.5))
    
    # Reordenar filas para agrupar por categoría
    sorted_corr = combined_corr.loc[ordered_features, :]
    
    # Crear heatmap con anotaciones
    heatmap = sns.heatmap(
        sorted_corr, 
        annot=True,
        cmap='coolwarm',
        vmin=-1,
        vmax=1,
        center=0,
        fmt='.2f',
        linewidths=.5,
    )
    
    # Añadir líneas para separar categorías
    count_end = sum(1 for feature in count_features if feature in ordered_features)
    surface_end = count_end + sum(1 for feature in surface_features if feature in ordered_features)
    
    plt.axhline(y=count_end, color='black', linewidth=2)
    plt.axhline(y=surface_end, color='black', linewidth=2)
    
    # Añadir etiquetas de categoría
    plt.text(-2.5, count_end / 2, 'CONTEO', verticalalignment='center', fontsize=12, fontweight='bold')
    plt.text(-2.5, count_end + (surface_end - count_end) / 2, 'SUPERFICIE', verticalalignment='center', fontsize=12, fontweight='bold')
    plt.text(-2.5, surface_end + (len(ordered_features) - surface_end) / 2, 'ESTRUCTURA', verticalalignment='center', fontsize=12, fontweight='bold')
    
    plt.title('Correlaciones entre Características Lingüísticas y Puntuaciones por Fuente', fontsize=16)
    plt.tight_layout()
 
    
    # Calcular correlaciones promedio por categoría y fuente
    avg_correlations = {}
    
    for prefix in df_names:
        prefix_scores = [col for col in prefixed_score_columns if col.startswith(prefix)]
        if not prefix_scores:
            continue
            
        # Calcular promedios por categoría
        avg_by_category = {}
        
        # Conteo
        count_features_available = [f for f in count_features if f in available_features]
        if count_features_available:
            count_correlations = sorted_corr.loc[count_features_available, prefix_scores].abs().mean().mean()
            avg_by_category['Conteo'] = count_correlations
        
        # Superficie
        surface_features_available = [f for f in surface_features if f in available_features]
        if surface_features_available:
            surface_correlations = sorted_corr.loc[surface_features_available, prefix_scores].abs().mean().mean()
            avg_by_category['Superficie'] = surface_correlations
        
        # Estructura
        structure_features_available = [f for f in structure_features if f in available_features]
        if structure_features_available:
            structure_correlations = sorted_corr.loc[structure_features_available, prefix_scores].abs().mean().mean()
            avg_by_category['Estructura'] = structure_correlations
        
        avg_correlations[prefix] = avg_by_category
    
    # Crear dataframe para visualización comparativa
    comparison_data = []
    for source, categories in avg_correlations.items():
        for category, value in categories.items():
            comparison_data.append({
                'Fuente': source,
                'Categoría': category,
                'Correlación Promedio': value
            })
    
    comparison_df = pd.DataFrame(comparison_data)
    
    # Crear heatmap de comparación
    if len(comparison_df) > 0:
        fig_comparison, ax_comparison = plt.subplots(figsize=(8, 6))
        
        # Crear pivot para el heatmap
        pivot_df = comparison_df.pivot(index='Categoría', columns='Fuente', values='Correlación Promedio')
        
        sns.heatmap(pivot_df, annot=True, cmap='coolwarm', vmin=-1, vmax=1, fmt='.3f', linewidths=.5, ax=ax_comparison)
        ax_comparison.set_title('Comparación de Correlaciones Promedio por Fuente', fontsize=14)
    else:
        fig_comparison = None
    
    # Crear gráfico de barras para comparación
    if len(comparison_df) > 0:
        fig_bar, ax_bar = plt.subplots(figsize=(10, 6))
        
        sns.barplot(x='Categoría', y='Correlación Promedio', hue='Fuente', data=comparison_df, ax=ax_bar)
        ax_bar.set_title('Correlación Promedio por Categoría y Fuente', fontsize=14)
        ax_bar.set_ylabel('Correlación Promedio (valor absoluto)')
        ax_bar.set_xlabel('Categoría de Características')
        plt.legend(title='Fuente')
        plt.tight_layout()
    else:
        fig_bar = None
    
    # Crear análisis comparativo
    comparative_analysis = []
    
    for category in ['Conteo', 'Superficie', 'Estructura']:
        category_data = comparison_df[comparison_df['Categoría'] == category]
        if len(category_data) < 2:
            continue
            
        sources = category_data['Fuente'].tolist()
        values = category_data['Correlación Promedio'].tolist()
        
        if len(sources) >= 2:
            max_idx = values.index(max(values))
            min_idx = values.index(min(values))
            
            analysis = f"Para la categoría {category}, {sources[max_idx]} muestra la mayor correlación "
            analysis += f"({values[max_idx]:.3f}), mientras que {sources[min_idx]} muestra la menor "
            analysis += f"({values[min_idx]:.3f}). "
            
            diff_pct = ((values[max_idx] / values[min_idx]) - 1) * 100 if values[min_idx] > 0 else 0
            if diff_pct > 10:
                analysis += f"Esto representa una diferencia del {diff_pct:.1f}%, lo que sugiere una "
                analysis += f"sensibilidad considerablemente distinta a las características de {category} "
                analysis += f"entre estas fuentes de evaluación."
            
            comparative_analysis.append(analysis)
    
    # Análisis general
    general_analysis = "Análisis general por fuente:\n"
    
    for source in df_names:
        if source in avg_correlations:
            categories = avg_correlations[source]
            if categories:
                max_category = max(categories.items(), key=lambda x: x[1])
                
                general_analysis += f"- {source}: La categoría dominante es {max_category[0]} ({max_category[1]:.3f}), "
                
                other_categories = {k: v for k, v in categories.items() if k != max_category[0]}
                if other_categories:
                    other_items = list(other_categories.items())
                    
                    for i, (cat, val) in enumerate(other_items):
                        diff = max_category[1] - val
                        general_analysis += f"superando a {cat} por {diff:.3f} ({val:.3f})"
                        
                        if i < len(other_items) - 1:
                            general_analysis += " y "
                
                general_analysis += ".\n"
    
    return {
        'heatmap_fig': plt.gcf(),
        'comparison_fig': fig_comparison,
        'bar_fig': fig_bar,
        'correlation_matrix': sorted_corr,
        'avg_correlations': avg_correlations,
        'comparison_df': comparison_df,
        'comparative_analysis': comparative_analysis,
        'general_analysis': general_analysis
    }

def run_multi_df_correlation_analysis(dataframes, df_names, text_column='response', save_figures=True):
    """
    Ejecuta un análisis completo de correlación comparando múltiples dataframes.
    
    Args:
        dataframes: Lista de DataFrames con las respuestas y puntuaciones
        df_names: Lista de nombres para identificar cada dataframe
        text_column: Nombre de la columna que contiene el texto a analizar
        save_figures: Si se deben guardar las figuras en archivos
        
    Returns:
        Dict con resultados y figuras
    """
    print(f"Iniciando análisis comparativo entre {len(dataframes)} fuentes de evaluación...")
    
    # Ejecutar el análisis comparativo
    results = create_correlation_heatmap_multi_df(dataframes, df_names, text_column)
    
    if results is None:
        print("Error al generar el análisis de correlación.")
        return None
    
    # Imprimir resumen de hallazgos
    print("\n===== RESUMEN DEL ANÁLISIS COMPARATIVO =====\n")
    
    # Mostrar comparación general entre fuentes
    if 'comparison_df' in results and len(results['comparison_df']) > 0:
        pivot_df = results['comparison_df'].pivot(index='Categoría', columns='Fuente', values='Correlación Promedio')
        print("CORRELACIONES PROMEDIO POR CATEGORÍA Y FUENTE:")
        print(pivot_df)
    
    # Análisis comparativo entre fuentes
    if 'comparative_analysis' in results and results['comparative_analysis']:
        print("\nANÁLISIS COMPARATIVO ENTRE FUENTES:")
        for analysis in results['comparative_analysis']:
            print(f"\n• {analysis}")
    
    # Análisis general
    if 'general_analysis' in results:
        print("\nANÁLISIS GENERAL POR FUENTE:")
        print(results['general_analysis'])
    
    # Guardar figuras si se solicitó
    if save_figures:
        if 'heatmap_fig' in results and results['heatmap_fig'] is not None:
            results['heatmap_fig'].savefig('multi_correlation_heatmap.png', dpi=300, bbox_inches='tight')
            print("\nFigura guardada como: multi_correlation_heatmap.png")
            
        if 'comparison_fig' in results and results['comparison_fig'] is not None:
            results['comparison_fig'].savefig('comparison_heatmap.png', dpi=300, bbox_inches='tight')
            print("Figura guardada como: comparison_heatmap.png")
            
        if 'bar_fig' in results and results['bar_fig'] is not None:
            results['bar_fig'].savefig('comparison_barplot.png', dpi=300, bbox_inches='tight')
            print("Figura guardada como: comparison_barplot.png")
    
    return results

# Ejemplo de uso:
"""
# Para ejecutar el análisis comparativo:
results = run_multi_df_correlation_analysis(
    dataframes=[df_ai, df_human],
    df_names=['AI', 'Human'],
    text_column='response'
)

# Para mostrar los gráficos:
import matplotlib.pyplot as plt
plt.show()
"""

In [ ]:
# Cargar los datos
answers = load_data("answers")
evaluations = load_data("ia_eval")

# Convertir los IDs a string para asegurar compatibilidad
answers["_id"] = answers["_id"].astype(str)
evaluations["original_answer_id"] = evaluations["original_answer_id"].astype(str)

# Método 1: Usando merge con todas las columnas necesarias
evaluations_with_answers_ia = evaluations.merge(
    answers[['_id', 'answer', 'bibliography_sources', 'confidence_level']], 
    left_on='original_answer_id',
    right_on='_id',
    how='left'
)

# Función para combinar respuesta y bibliografía cuando esté disponible
def combine_answer_and_bibliography(row):
    if pd.notna(row['bibliography_sources']):
        return row['answer'] + '\n\nReferences: ' + row['bibliography_sources']
    else:
        return row['answer']

# Crear la columna original_answer con la combinación de respuesta y bibliografía
evaluations_with_answers_ia['original_answer'] = evaluations_with_answers_ia.apply(combine_answer_and_bibliography, axis=1)


# Verificar el resultado
print("Primeras filas del DataFrame final:")
print(evaluations_with_answers_ia.head())
evaluations_with_answers_ia["response"] = evaluations_with_answers_ia["original_answer"]
evaluations_with_answers_ia, _ , _ = filter_and_analyze_responses(evaluations_with_answers_ia)


# Cargar los datos
answers = load_data("answers")
evaluations = load_data("eval")

# Convertir los IDs a string para asegurar compatibilidad
answers["_id"] = answers["_id"].astype(str)
evaluations["response"] = evaluations["original_answer"]
evaluations["original_answer_id"] = evaluations["original_answer_id"].astype(str)

# Método 1: Usando merge con todas las columnas necesarias
evaluations_with_answers = evaluations.merge(
    answers[['_id', 'answer', 'bibliography_sources', 'confidence_level']], 
    left_on='original_answer_id',
    right_on='_id',
    how='left'
)

# Función para combinar respuesta y bibliografía cuando esté disponible
def combine_answer_and_bibliography(row):
    if pd.notna(row['bibliography_sources']):
        return row['answer'] + '\n\nReferences: ' + row['bibliography_sources']
    else:
        return row['answer']

# Crear la columna original_answer con la combinación de respuesta y bibliografía
evaluations_with_answers['original_answer'] = evaluations_with_answers.apply(combine_answer_and_bibliography, axis=1)


# Verificar el resultado
print("Primeras filas del DataFrame final:")
print(evaluations_with_answers.head())
evaluations_with_answers["response"] = evaluations_with_answers["original_answer"]
evaluations_with_answers, _ , _ = filter_and_analyze_responses(evaluations_with_answers)

In [ ]:
# Para ejecutar el análisis comparativo:
df_human = evaluations_with_answers
df_ai = evaluations_with_answers_ia

results = run_multi_df_correlation_analysis(
    dataframes=[df_ai, df_human],
    df_names=['AI', 'human'],
    text_column='response'
)

# Para mostrar los gráficos:
import matplotlib.pyplot as plt
plt.show()

In [ ]:
def create_correlation_heatmap_by_llm(dataframes, df_names, text_column='response', llm_column='LLM'):
    """
    Crea heatmaps de correlaciones separados por tipo de LLM (0=humano, 1=IA, 2=IA CoT)
    para cada conjunto de datos en dataframes.
    
    Args:
        dataframes: Lista de DataFrames con respuestas y puntuaciones
        df_names: Lista de nombres para identificar cada dataframe
        text_column: Nombre de la columna que contiene el texto a analizar
        llm_column: Nombre de la columna que contiene el tipo de LLM
        
    Returns:
        Dict con resultados y visualizaciones separados por tipo de LLM
    """
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns
    from tqdm import tqdm
    from sklearn.preprocessing import MinMaxScaler
    
    # Verificar que los parámetros sean correctos
    if len(dataframes) != len(df_names):
        raise ValueError("El número de dataframes debe ser igual al número de nombres proporcionados")
    
    # Preparar cada dataframe y extraer características
    processed_dfs = []
    for i, df in enumerate(dataframes):
        print(f"Procesando dataframe {df_names[i]}...")
        processed_df = prepare_dataframe_for_analysis(df, text_column=text_column)
        processed_dfs.append(processed_df)
    
    # Definir las categorías de características
    count_features = ['word_count', 'sentence_count']
    
    surface_features = [
        'avg_word_length', 'avg_sentence_length', 'reading_ease', 'grade_level',
        'connector_count'
    ]
    
    structure_features = [
        'unique_words_ratio', 'lexical_diversity',
        'verb_ratio', 'noun_ratio', 'adj_ratio', 'adv_ratio', 'pronoun_ratio',
        'named_entity_count', 'dependency_distance'
    ]
    
    # Todas las características
    all_features = count_features + surface_features + structure_features
    
    # Definir tipos de LLM
    llm_types = {0: "Humano", 1: "IA", 2: "IA CoT"}
    
    # Resultados para cada tipo de LLM
    results_by_llm = {}
    
    # Analizar cada tipo de LLM por separado
    for llm_value, llm_name in llm_types.items():
        print(f"\nAnalizando correlaciones para respuestas de {llm_name} (LLM={llm_value})...")
        
        # Filtrar dataframes por tipo de LLM
        filtered_dfs = []
        filtered_df_names = []
        
        for i, df in enumerate(processed_dfs):
            if llm_column in df.columns:
                filtered_df = df[df[llm_column] == llm_value].copy()
                
                if len(filtered_df) > 0:
                    filtered_dfs.append(filtered_df)
                    filtered_df_names.append(df_names[i])
                    print(f"  - {df_names[i]}: {len(filtered_df)} evaluaciones encontradas")
                else:
                    print(f"  - {df_names[i]}: No hay evaluaciones para LLM={llm_value}")
            else:
                print(f"  - {df_names[i]}: No contiene la columna '{llm_column}'")
        
        if not filtered_dfs:
            print(f"No se encontraron datos para LLM={llm_value}")
            continue
        
        # Analizar correlaciones para este tipo de LLM
        all_correlations = []
        prefixed_score_columns = []
        
        for i, df in enumerate(filtered_dfs):
            prefix = filtered_df_names[i]
            
            # Verificar qué características están disponibles
            available_features = [f for f in all_features if f in df.columns]
            
            # Verificar qué puntuaciones están disponibles
            score_columns = ["accuracy_score", "clarity_score", "completeness_score"]
            available_scores = [s for s in score_columns if s in df.columns]
            
            if not available_features or not available_scores:
                print(f"No se encontraron suficientes datos en {prefix}. Omitiendo...")
                continue
            
            # Escalar características
            df_scaled = df.copy()
            scaler = MinMaxScaler()
            df_scaled[available_features] = scaler.fit_transform(df[available_features])
            
            # Calcular correlaciones
            corr_matrix = df_scaled[available_features + available_scores].corr()
            feature_score_corr = corr_matrix.loc[available_features, available_scores]
            
            # Guardar con prefijos para cada score
            for score in available_scores:
                prefixed_score = f"{prefix}_{score}"
                prefixed_score_columns.append(prefixed_score)
                
                # Guardar correlaciones para esta puntuación
                all_correlations.append(
                    pd.DataFrame(
                        feature_score_corr[score].values, 
                        index=available_features,
                        columns=[prefixed_score]
                    )
                )
        
        # Combinar todas las correlaciones
        if not all_correlations:
            print(f"No se pudieron calcular correlaciones para LLM={llm_value}")
            continue
        
        combined_corr = pd.concat(all_correlations, axis=1)
        
        # Verificar características disponibles en el resultado combinado
        available_features = combined_corr.index.tolist()
        
        # Ordenar características por categoría para visualización
        ordered_features = []
        for feature_group in [count_features, surface_features, structure_features]:
            ordered_features.extend([f for f in feature_group if f in available_features])
        
        # Crear figura para el heatmap principal
        plt.figure(figsize=(20, len(ordered_features) * 0.5))  # Ancho aumentado
        
        # Reordenar filas para agrupar por categoría
        sorted_corr = combined_corr.loc[ordered_features, :]
        
        # Crear heatmap con anotaciones
        heatmap = sns.heatmap(
            sorted_corr, 
            annot=True,
            cmap='coolwarm',
            vmin=-1,
            vmax=1,
            center=0,
            fmt='.2f',
            linewidths=.5,
        )
        
        # Añadir líneas para separar categorías
        count_end = sum(1 for feature in count_features if feature in ordered_features)
        surface_end = count_end + sum(1 for feature in surface_features if feature in ordered_features)
        
        plt.axhline(y=count_end, color='black', linewidth=2)
        plt.axhline(y=surface_end, color='black', linewidth=2)
        
        # Añadir etiquetas de categoría - ajustadas para un gráfico más ancho
        plt.text(-1.0, count_end / 2, 'CONTEO', verticalalignment='center', fontsize=12, fontweight='bold')
        plt.text(-1.0, count_end + (surface_end - count_end) / 2, 'SUPERFICIE', verticalalignment='center', fontsize=12, fontweight='bold')
        plt.text(-1.0, surface_end + (len(ordered_features) - surface_end) / 2, 'ESTRUCTURA', verticalalignment='center', fontsize=12, fontweight='bold')
        
        plt.title(f'Correlaciones para Respuestas de {llm_name} (LLM={llm_value})', fontsize=16)
        plt.tight_layout()
     
        
        # Calcular correlaciones promedio por categoría y fuente
        avg_correlations = {}
        
        for prefix in filtered_df_names:
            prefix_scores = [col for col in prefixed_score_columns if col.startswith(prefix)]
            if not prefix_scores:
                continue
                
            # Calcular promedios por categoría
            avg_by_category = {}
            
            # Conteo
            count_features_available = [f for f in count_features if f in available_features]
            if count_features_available:
                count_correlations = sorted_corr.loc[count_features_available, prefix_scores].abs().mean().mean()
                avg_by_category['Conteo'] = count_correlations
            
            # Superficie
            surface_features_available = [f for f in surface_features if f in available_features]
            if surface_features_available:
                surface_correlations = sorted_corr.loc[surface_features_available, prefix_scores].abs().mean().mean()
                avg_by_category['Superficie'] = surface_correlations
            
            # Estructura
            structure_features_available = [f for f in structure_features if f in available_features]
            if structure_features_available:
                structure_correlations = sorted_corr.loc[structure_features_available, prefix_scores].abs().mean().mean()
                avg_by_category['Estructura'] = structure_correlations
            
            avg_correlations[prefix] = avg_by_category
        
        # Crear dataframe para visualización comparativa
        comparison_data = []
        for source, categories in avg_correlations.items():
            for category, value in categories.items():
                comparison_data.append({
                    'Fuente': source,
                    'Categoría': category,
                    'Correlación Promedio': value
                })
        
        comparison_df = pd.DataFrame(comparison_data)
        
        # Crear heatmap de comparación
        if len(comparison_df) > 0:
            fig_comparison, ax_comparison = plt.subplots(figsize=(8, 6))
            
            # Crear pivot para el heatmap
            pivot_df = comparison_df.pivot(index='Categoría', columns='Fuente', values='Correlación Promedio')
            
            sns.heatmap(pivot_df, annot=True, cmap='coolwarm', vmin=-1, vmax=1, fmt='.3f', linewidths=.5, ax=ax_comparison)
            ax_comparison.set_title(f'Comparación para {llm_name} (LLM={llm_value})', fontsize=14)
        else:
            fig_comparison = None
        
        # Crear gráfico de barras para comparación
        if len(comparison_df) > 0:
            fig_bar, ax_bar = plt.subplots(figsize=(10, 6))
            
            sns.barplot(x='Categoría', y='Correlación Promedio', hue='Fuente', data=comparison_df, ax=ax_bar)
            ax_bar.set_title(f'Correlación Promedio para {llm_name} (LLM={llm_value})', fontsize=14)
            ax_bar.set_ylabel('Correlación Promedio (valor absoluto)')
            ax_bar.set_xlabel('Categoría de Características')
            plt.legend(title='Fuente')
            plt.tight_layout()
        else:
            fig_bar = None
        
        # Guardar resultados para este tipo de LLM
        results_by_llm[llm_value] = {
            'llm_name': llm_name,
            'heatmap_fig': plt.gcf(),
            'comparison_fig': fig_comparison,
            'bar_fig': fig_bar,
            'correlation_matrix': sorted_corr,
            'avg_correlations': avg_correlations,
            'comparison_df': comparison_df
        }
    
    return results_by_llm

def run_llm_type_correlation_analysis(dataframes, df_names, text_column='response', llm_column='LLM', save_figures=True):
    """
    Ejecuta un análisis completo de correlación separado por tipo de LLM (0=humano, 1=IA, 2=IA CoT).
    
    Args:
        dataframes: Lista de DataFrames con las respuestas y puntuaciones
        df_names: Lista de nombres para identificar cada dataframe
        text_column: Nombre de la columna que contiene el texto a analizar
        llm_column: Nombre de la columna que identifica el tipo de LLM
        save_figures: Si se deben guardar las figuras en archivos
        
    Returns:
        Dict con resultados y figuras para cada tipo de LLM
    """
    print(f"Iniciando análisis por tipo de LLM entre {len(dataframes)} fuentes de evaluación...")
    
    # Ejecutar el análisis comparativo
    results_by_llm = create_correlation_heatmap_by_llm(dataframes, df_names, text_column, llm_column)
    
    if not results_by_llm:
        print("Error al generar el análisis de correlación por tipo de LLM.")
        return None
    
    # Imprimir resumen de hallazgos para cada tipo de LLM
    for llm_value, results in results_by_llm.items():
        llm_name = results['llm_name']
        print(f"\n===== RESUMEN DEL ANÁLISIS PARA {llm_name.upper()} (LLM={llm_value}) =====\n")
        
        # Mostrar comparación general entre fuentes
        if 'comparison_df' in results and len(results['comparison_df']) > 0:
            pivot_df = results['comparison_df'].pivot(index='Categoría', columns='Fuente', values='Correlación Promedio')
            print("CORRELACIONES PROMEDIO POR CATEGORÍA Y FUENTE:")
            print(pivot_df)
        
        # Guardar figuras si se solicitó
        if save_figures:
            if 'heatmap_fig' in results and results['heatmap_fig'] is not None:
                filename = f'heatmap_llm{llm_value}_{results["llm_name"]}.png'
                results['heatmap_fig'].savefig(filename, dpi=300, bbox_inches='tight')
                print(f"Figura guardada como: {filename}")
                
            if 'comparison_fig' in results and results['comparison_fig'] is not None:
                filename = f'comparison_heatmap_llm{llm_value}_{results["llm_name"]}.png'
                results['comparison_fig'].savefig(filename, dpi=300, bbox_inches='tight')
                print(f"Figura guardada como: {filename}")
                
            if 'bar_fig' in results and results['bar_fig'] is not None:
                filename = f'barplot_llm{llm_value}_{results["llm_name"]}.png'
                results['bar_fig'].savefig(filename, dpi=300, bbox_inches='tight')
                print(f"Figura guardada como: {filename}")
    
    # Crear un análisis comparativo entre tipos de LLM
    print("\n===== COMPARACIÓN ENTRE TIPOS DE LLM =====\n")
    
    # Comparar categorías dominantes por fuente y tipo de LLM
    all_dominants = {}
    
    for llm_value, results in results_by_llm.items():
        llm_name = results['llm_name']
        for source, categories in results['avg_correlations'].items():
            if not categories:
                continue
                
            dominant_category = max(categories.items(), key=lambda x: x[1])
            
            if source not in all_dominants:
                all_dominants[source] = {}
            
            all_dominants[source][llm_name] = {
                'dominant_category': dominant_category[0],
                'correlation': dominant_category[1]
            }
    
    # Imprimir comparación
    for source, llm_data in all_dominants.items():
        print(f"\nFuente: {source}")
        for llm_name, data in llm_data.items():
            print(f"  - {llm_name}: {data['dominant_category']} ({data['correlation']:.3f})")
    
    return results_by_llm



In [ ]:
# Para ejecutar el análisis por tipo de LLM:
results_by_llm = run_llm_type_correlation_analysis(
    dataframes=[df_ai, df_human],
    df_names=['AI', 'Human'],
    text_column='response',
    llm_column='LLM'
)

# Para mostrar los gráficos:
import matplotlib.pyplot as plt
plt.show()

In [ ]:
def create_comparative_correlation_heatmap_with_bootstrap(df_ai, df_human, text_column='response', llm_column='LLM', n_bootstrap=1000, n_samples=50, confidence=0.95):
    """
    Creates a correlation heatmap with confidence intervals calculated through bootstrapping using Plotly.
    
    Args:
        df_ai: DataFrame with AI evaluations
        df_human: DataFrame with human evaluations
        text_column: Name of the column containing the text to analyze
        llm_column: Name of the column containing the LLM type
        n_bootstrap: Number of bootstrap samples for confidence intervals
        confidence: Confidence level for intervals (default 0.95)
        
    Returns:
        Plotly figure with comparative heatmap and DataFrame with correlations and intervals
    """
    import pandas as pd
    import numpy as np
    import plotly.graph_objects as go
    from scipy.stats import pearsonr
    from tqdm import tqdm
    
    def bootstrap_correlation(data, features, target, sample_size=50, n_iterations=1000, conf_level=0.95):
        """
        Calculates correlations using bootstrap resampling technique.
        Takes 'sample_size' random samples, repeats 'n_iterations' times and averages the results.
        
        Args:
            data: DataFrame with data
            features: List of features to correlate
            target: Target variable (e.g., 'accuracy_score')
            sample_size: Size of each subsample (default 50)
            n_iterations: Number of iterations (default 1000)
            conf_level: Confidence level for intervals (default 0.95)
            
        Returns:
            Dict with correlations and their confidence intervals
        """
        import numpy as np
        import pandas as pd
        from scipy.stats import pearsonr
        
        # Verify that sample_size is not larger than available data
        n = len(data)
        if sample_size > n:
            print(f"WARNING: Sample size ({sample_size}) larger than available data ({n}). Adjusting to {n}.")
            sample_size = n
        
        correlations = []
        
        # Perform n_iterations of subsampling
        for _ in range(n_iterations):
            # Select sample_size random samples (with replacement)
            indices = np.random.choice(range(n), size=sample_size, replace=True)
            bootstrap_sample = data.iloc[indices]
            
            # Calculate correlations for this sample
            corr_sample = {}
            for feature in features:
                # Verify if there's enough valid data
                valid_data = ~(bootstrap_sample[feature].isna() | bootstrap_sample[target].isna())
                if valid_data.sum() < 3:  # Need at least 3 points for correlation
                    corr_sample[feature] = np.nan
                    continue
                    
                # Use scipy.stats.pearsonr to get only the coefficient
                corr, _ = pearsonr(bootstrap_sample.loc[valid_data, feature], 
                                bootstrap_sample.loc[valid_data, target])
                corr_sample[feature] = corr
            
            correlations.append(corr_sample)
        
        # Convert to DataFrame for easier calculation
        corr_df = pd.DataFrame(correlations)
        
        # Calculate point correlation and confidence intervals
        alpha = 1 - conf_level
        result = {}
        for feature in features:
            # Filter NaN values
            feature_corrs = corr_df[feature].dropna()
            
            if len(feature_corrs) == 0:
                # No valid correlations for this feature
                result[feature] = {
                    'correlation': np.nan,
                    'lower_ci': np.nan,
                    'upper_ci': np.nan,
                    'sample_count': 0
                }
                continue
            
            # Point estimate (bootstrap average)
            point_estimate = feature_corrs.mean()
            
            # Confidence intervals
            if len(feature_corrs) >= 3:  # Need at least 3 samples for CI
                lower_bound = np.percentile(feature_corrs, alpha/2 * 100)
                upper_bound = np.percentile(feature_corrs, (1 - alpha/2) * 100)
            else:
                # Not enough samples to calculate CIs
                lower_bound = np.nan
                upper_bound = np.nan
            
            result[feature] = {
                'correlation': point_estimate,
                'lower_ci': lower_bound,
                'upper_ci': upper_bound,
                'sample_count': len(feature_corrs)
            }
        
        return result
    
    # Prepare the dataframes
    print("Processing AI dataframe...")
    df_ai_processed = prepare_dataframe_for_analysis(df_ai, text_column=text_column)
    
    print("Processing Human dataframe...")
    df_human_processed = prepare_dataframe_for_analysis(df_human, text_column=text_column)
    
    # Define feature categories
    count_features = ['word_count', 'sentence_count']
    
    surface_features = [
        'avg_word_length', 'avg_sentence_length', 'reading_ease', 'grade_level',
        'connector_count'
    ]
    
    structure_features = [
        'unique_words_ratio', 'lexical_diversity',
        'verb_ratio', 'noun_ratio', 'adj_ratio', 'adv_ratio', 'pronoun_ratio',
        'named_entity_count', 'dependency_distance'
    ]
    
    # All features
    all_features = count_features + surface_features + structure_features
    
    # Define LLM types
    llm_types = {0: "Human", 1: "AI", 2: "CoT"}
    
    # Check which features are available in both dataframes
    available_features = [f for f in all_features 
                         if f in df_ai_processed.columns and f in df_human_processed.columns]
    
    # Check if accuracy_score is available in both dataframes
    if "accuracy_score" not in df_ai_processed.columns:
        raise ValueError("'accuracy_score' not found in AI dataframe")
    if "accuracy_score" not in df_human_processed.columns:
        raise ValueError("'accuracy_score' not found in Human dataframe")
    
    # Create dataframes to store correlations and intervals
    point_correlations = pd.DataFrame(index=available_features)
    lower_ci = pd.DataFrame(index=available_features)
    upper_ci = pd.DataFrame(index=available_features)
    
    # 1. Calculate correlations with bootstrapping for TOTAL (AI vs Human)
    print("\nCalculating correlations with bootstrapping for AI (Total)...")
    ai_bootstrap = bootstrap_correlation(
        df_ai_processed, 
        available_features, 
        "accuracy_score", 
        sample_size=n_samples,
        n_iterations=n_bootstrap,
        conf_level=confidence
    )
    
    # Save results
    point_correlations["AI_Total"] = [ai_bootstrap[feature]['correlation'] for feature in available_features]
    lower_ci["AI_Total"] = [ai_bootstrap[feature]['lower_ci'] for feature in available_features]
    upper_ci["AI_Total"] = [ai_bootstrap[feature]['upper_ci'] for feature in available_features]
    
    print("Calculating correlations with bootstrapping for Human (Total)...")
    human_bootstrap = bootstrap_correlation(
        df_human_processed, 
        available_features, 
        "accuracy_score", 
        sample_size=n_samples,
        n_iterations=n_bootstrap,
        conf_level=confidence
    )
    
    # Save results
    point_correlations["Human_Total"] = [human_bootstrap[feature]['correlation'] for feature in available_features]
    lower_ci["Human_Total"] = [human_bootstrap[feature]['lower_ci'] for feature in available_features]
    upper_ci["Human_Total"] = [human_bootstrap[feature]['upper_ci'] for feature in available_features]
    
    # 2. Now calculate correlations for each LLM type
    for llm_value in [0, 1, 2]:
        llm_name = llm_types[llm_value]
        print(f"\nProcessing correlations for {llm_name} (LLM={llm_value})...")
        
        # Filter AI dataframe by LLM type
        if llm_column in df_ai_processed.columns:
            df_ai_filtered = df_ai_processed[df_ai_processed[llm_column] == llm_value].copy()
            if len(df_ai_filtered) > 10:  # Ensure enough data for bootstrap
                print(f"  - AI: {len(df_ai_filtered)} evaluations found")
                
                # Calculate correlations with bootstrap
                ai_llm_bootstrap = bootstrap_correlation(
                    df_ai_filtered, 
                    available_features, 
                    "accuracy_score", 
                    sample_size=n_samples,
                    n_iterations=n_bootstrap,
                    conf_level=confidence
                )
                
                # Save results - FIXED: Corrected upper_ci assignment
                point_correlations[f"AI_LLM{llm_value}"] = [ai_llm_bootstrap[feature]['correlation'] for feature in available_features]
                lower_ci[f"AI_LLM{llm_value}"] = [ai_llm_bootstrap[feature]['lower_ci'] for feature in available_features]
                upper_ci[f"AI_LLM{llm_value}"] = [ai_llm_bootstrap[feature]['upper_ci'] for feature in available_features]
            else:
                print(f"  - AI: Not enough evaluations for LLM={llm_value} (minimum 10)")
                point_correlations[f"AI_LLM{llm_value}"] = np.nan
                lower_ci[f"AI_LLM{llm_value}"] = np.nan
                upper_ci[f"AI_LLM{llm_value}"] = np.nan
        
        # Filter Human dataframe by LLM type
        if llm_column in df_human_processed.columns:
            df_human_filtered = df_human_processed[df_human_processed[llm_column] == llm_value].copy()
            if len(df_human_filtered) > 10:  # Ensure enough data for bootstrap
                print(f"  - Human: {len(df_human_filtered)} evaluations found")
                
                # Calculate correlations with bootstrap
                human_llm_bootstrap = bootstrap_correlation(
                    df_human_filtered, 
                    available_features, 
                    "accuracy_score", 
                    sample_size=n_samples,
                    n_iterations=n_bootstrap,
                    conf_level=confidence
                )
                
                # Save results
                point_correlations[f"Human_LLM{llm_value}"] = [human_llm_bootstrap[feature]['correlation'] for feature in available_features]
                lower_ci[f"Human_LLM{llm_value}"] = [human_llm_bootstrap[feature]['lower_ci'] for feature in available_features]
                upper_ci[f"Human_LLM{llm_value}"] = [human_llm_bootstrap[feature]['upper_ci'] for feature in available_features]
            else:
                print(f"  - Human: Not enough evaluations for LLM={llm_value} (minimum 10)")
                point_correlations[f"Human_LLM{llm_value}"] = np.nan
                lower_ci[f"Human_LLM{llm_value}"] = np.nan
                upper_ci[f"Human_LLM{llm_value}"] = np.nan
    
    # Sort features by category for visualization
    ordered_features = []
    for feature_group in [count_features, surface_features, structure_features]:
        ordered_features.extend([f for f in feature_group if f in available_features])
    
    # Reorder dataframe rows
    sorted_corr = point_correlations.loc[ordered_features, :]
    sorted_lower = lower_ci.loc[ordered_features, :]
    sorted_upper = upper_ci.loc[ordered_features, :]
    
    # Create a DataFrame with complete results - FIXED: Using concat properly with pandas 
    result_df = pd.DataFrame()
    for col in sorted_corr.columns:
        for feature in sorted_corr.index:
            corr = sorted_corr.loc[feature, col]
            lower = sorted_lower.loc[feature, col]
            upper = sorted_upper.loc[feature, col]
            
            # Determine if correlation is significant (CI does not include zero)
            significant = False
            if not (np.isnan(lower) or np.isnan(upper)):
                significant = (lower > 0 and upper > 0) or (lower < 0 and upper < 0)
                
            new_row = pd.DataFrame({
                'Feature': [feature],
                'Group': [col],
                'Correlation': [corr],
                'Lower_CI': [lower],
                'Upper_CI': [upper],
                'Significant': [significant]
            })
            result_df = pd.concat([result_df, new_row], ignore_index=True)
    
    # Create Plotly heatmap
    fig = create_plotly_correlation_heatmap(
        sorted_corr, 
        sorted_lower, 
        sorted_upper, 
        confidence=confidence
    )
    
    # Save the image and HTML
    fig.write_image('bootstrap_correlation_heatmap.png', scale=2)
    fig.write_html('bootstrap_correlation_heatmap.html')
    
    # Save detailed results
    result_df.to_csv('bootstrap_correlation_results.csv', index=False)
    
    return fig, result_df
def create_plotly_correlation_heatmap(point_correlations, lower_ci, upper_ci, confidence=0.95):
    """
    Creates a publication-quality correlation heatmap with confidence intervals using Plotly.
    Optimized for small sizes (400x400) and paper publication.
    
    Args:
        point_correlations: DataFrame with point correlations
        lower_ci: DataFrame with lower bounds of confidence intervals
        upper_ci: DataFrame with upper bounds of confidence intervals
        confidence: Confidence level for intervals (default 0.95)
        
    Returns:
        Plotly figure object
    """
    import plotly.graph_objects as go
    import numpy as np
    import pandas as pd
    
    # Rename columns for better clarity
    column_mapping = {
        'AI_Total': 'AI (All)',
        'Human_Total': 'Human (All)',
        'AI_LLM0': 'AI (Human)',
        'Human_LLM0': 'Human (Human)',
        'AI_LLM1': 'AI (AI)',
        'Human_LLM1': 'Human (AI)',
        'AI_LLM2': 'AI (CoT)',
        'Human_LLM2': 'Human (CoT)'
    }
    
    # Create copies to avoid modifying the originals
    sorted_corr = point_correlations.copy()
    sorted_lower = lower_ci.copy()
    sorted_upper = upper_ci.copy()
    
    # Rename columns
    sorted_corr = sorted_corr.rename(columns=column_mapping)
    sorted_lower = sorted_lower.rename(columns=column_mapping)
    sorted_upper = sorted_upper.rename(columns=column_mapping)
    
    # Define feature categories
    count_features = ['word_count', 'sentence_count']
    surface_features = [
        'avg_word_length', 'avg_sentence_length', 'reading_ease', 'grade_level',
        'connector_count'
    ]
    structure_features = [
        'unique_words_ratio', 'lexical_diversity',
        'verb_ratio', 'noun_ratio', 'adj_ratio', 'adv_ratio', 'pronoun_ratio',
        'named_entity_count', 'dependency_distance'
    ]
    
    # Format feature names for better readability with shorter names for small plots
    feature_name_mapping = {
        'word_count': 'Word Count',
        'sentence_count': 'Sentence Count',
        'avg_word_length': 'Word Length',
        'avg_sentence_length': 'Sentence Length',
        'reading_ease': 'Readability',
        'grade_level': 'Grade Level',
        'connector_count': 'Connectors',
        'unique_words_ratio': 'Unique Words',
        'lexical_diversity': 'Lexical Diversity',
        'verb_ratio': 'Verbs',
        'noun_ratio': 'Nouns',
        'adj_ratio': 'Adjectives',
        'adv_ratio': 'Adverbs',
        'pronoun_ratio': 'Pronouns',
        'named_entity_count': 'Named Entities',
        'dependency_distance': 'Dependency Dist.'
    }
    
    # Apply the mapping to create a new index
    sorted_corr.index = [feature_name_mapping.get(feature, feature) for feature in sorted_corr.index]
    sorted_lower.index = [feature_name_mapping.get(feature, feature) for feature in sorted_lower.index]
    sorted_upper.index = [feature_name_mapping.get(feature, feature) for feature in sorted_upper.index]
    
    # Get available features in order (ensuring we use the formatted names)
    ordered_features = []
    for feature_group in [count_features, surface_features, structure_features]:
        for feature in feature_group:
            formatted_name = feature_name_mapping.get(feature, feature)
            if formatted_name in sorted_corr.index:
                ordered_features.append(formatted_name)
    
    # Reorder dataframe rows
    sorted_corr = sorted_corr.loc[ordered_features, :]
    sorted_lower = sorted_lower.loc[ordered_features, :]
    sorted_upper = sorted_upper.loc[ordered_features, :]
    
    # Determine significance (CI does not include zero)
    significant = ((sorted_lower > 0) & (sorted_upper > 0)) | ((sorted_lower < 0) & (sorted_upper < 0))
    
    # Define category boundaries in the formatted index
    count_formatted = [feature_name_mapping.get(f, f) for f in count_features 
                      if feature_name_mapping.get(f, f) in sorted_corr.index]
    surface_formatted = [feature_name_mapping.get(f, f) for f in surface_features 
                        if feature_name_mapping.get(f, f) in sorted_corr.index]
    
    count_end = len(count_formatted)
    surface_end = count_end + len(surface_formatted)
    
    # Prepare hover text
    hover_texts = []
    for i, feature in enumerate(sorted_corr.index):
        feature_texts = []
        for j, column in enumerate(sorted_corr.columns):
            corr_value = sorted_corr.loc[feature, column]
            lower = sorted_lower.loc[feature, column]
            upper = sorted_upper.loc[feature, column]
            is_sig = significant.loc[feature, column]
            
            if not np.isnan(corr_value):
                text = f"Feature: {feature}<br>"
                text += f"Group: {column}<br>"
                text += f"Correlation: {corr_value:.3f}<br>"
                text += f"CI ({confidence*100:.0f}%): [{lower:.3f}, {upper:.3f}]<br>"
                text += f"Statistically significant: {'Yes' if is_sig else 'No'}"
                feature_texts.append(text)
            else:
                feature_texts.append("")
        hover_texts.append(feature_texts)
    
    # Create base figure
    fig = go.Figure()
    
    # Add heatmap with regular y-axis labels (no embedded category headers)
    fig.add_trace(go.Heatmap(
        z=sorted_corr.values,
        x=sorted_corr.columns,
        y=sorted_corr.index,
        colorscale='RdBu_r',
        zmid=0,
        zmin=-1,
        zmax=1,
        hoverinfo='text',
        hovertext=hover_texts,
        showscale=False  # Remove the color scale/legend
    ))
    
    # Add annotations for correlation values with optimized size
    annotations = []
    for i, feature in enumerate(sorted_corr.index):
        for j, column in enumerate(sorted_corr.columns):
            corr_value = sorted_corr.loc[feature, column]
            is_sig = significant.loc[feature, column]
            
            if not np.isnan(corr_value):
                # Format with only 2 decimal places to save space
                text = f"{corr_value:.2f}"
                if is_sig:
                    text += "*"
                
                # Text color based on value for better readability
                text_color = 'white' if abs(corr_value) > 0.5 else 'black'
                
                annotations.append(dict(
                    x=j,
                    y=i,
                    text=text,
                    font=dict(
                        color=text_color, 
                        size=18,  # Even larger font size for correlation values
                        family="Arial Black"
                    ),
                    showarrow=False,
                    xref='x',
                    yref='y'
                ))
    
    # Add horizontal lines to separate categories
    shapes = [
        # Horizontal line after count features
        dict(
            type="line",
            x0=-0.5,
            y0=count_end - 0.5,
            x1=len(sorted_corr.columns) - 0.5,
            y1=count_end - 0.5,
            line=dict(color="black", width=1)
        ),
        # Horizontal line after surface features
        dict(
            type="line",
            x0=-0.5,
            y0=surface_end - 0.5,
            x1=len(sorted_corr.columns) - 0.5,
            y1=surface_end - 0.5,
            line=dict(color="black", width=1)
        )
    ]

    
    # Add all annotations together
    all_annotations = annotations 
    
    # Add note about statistical significance
    all_annotations.append(dict(
        x=-0.3,
        y=-0.32,
        xref="paper",
        yref="paper",
        text="* Indicates statistically significant correlation (CI does not include zero)",
        showarrow=False,
        font=dict(size=16, family="Arial"),
        align="left",
        xanchor="left"
    ))
    
    # Update layout optimized for small size
    fig.update_layout(
        title=dict(
            text=f"Feature Correlations with Accuracy Score",
            font=dict(size=24, family="Arial"),
            x=0.5,
            xanchor="center",
            y=0.98
        ),
        height=600,  # Fixed small size
        width=600,   # Fixed small size
        xaxis=dict(
            title='',
            tickfont=dict(size=18, family="Arial"),
            tickangle=45
        ),
        yaxis=dict(
            title='',
            tickfont=dict(size=18, family="Arial"),
            autorange="reversed"
        ),
        template="plotly_white",
        shapes=shapes,
        annotations=all_annotations,
        margin=dict(l=70, r=5, t=50, b=90),  # Increased left margin for category labels
        paper_bgcolor='white',
        plot_bgcolor='white'
    )
    
    return fig

def run_bootstrap_correlation_analysis(df_ai, df_human, text_column='response', llm_column='LLM', n_bootstrap=1000, n_samples=50, confidence=0.95):
    """
    Runs comparative analysis between AI and Human with bootstrapping.
    
    Args:
        df_ai: DataFrame with AI evaluations
        df_human: DataFrame with human evaluations
        text_column: Name of the column containing the text to analyze
        llm_column: Name of the column identifying the LLM type
        n_bootstrap: Number of bootstrap samples
        confidence: Confidence level (0-1)
        
    Returns:
        Tuple with (figure, results dataframe)
    """
    print(f"Starting comparative analysis with bootstrapping ({n_bootstrap} iterations with {n_samples} samples, {confidence*100:.0f}% confidence)...")
    
    # Run the comparative analysis
    fig, results_df = create_comparative_correlation_heatmap_with_bootstrap(
        df_ai, df_human, text_column, llm_column, n_bootstrap, n_samples, confidence
    )
    
    # Print a summary of findings
    print("\n===== BOOTSTRAP ANALYSIS SUMMARY =====\n")
    
    # Print available groups
    groups = results_df['Group'].unique()
    print(f"Analyzed groups: {', '.join(groups)}")
    
    # Define feature categories
    count_features = ['word_count', 'sentence_count']
    surface_features = [
        'avg_word_length', 'avg_sentence_length', 'reading_ease', 'grade_level',
        'connector_count'
    ]
    structure_features = [
        'unique_words_ratio', 'lexical_diversity',
        'verb_ratio', 'noun_ratio', 'adj_ratio', 'adv_ratio', 'pronoun_ratio',
        'named_entity_count', 'dependency_distance'
    ]
    
    # Find significant correlations
    significant_results = results_df[results_df['Significant'] == True]
    print(f"\nFound {len(significant_results)} statistically significant correlations")
    
    # Summary by group
    for group in groups:
        group_results = results_df[results_df['Group'] == group]
        sig_count = sum(group_results['Significant'])
        print(f"\n{group}: {sig_count} significant correlations out of {len(group_results)}")
        
        # Top positive significant correlations
        pos_sig = group_results[(group_results['Significant']) & (group_results['Correlation'] > 0)]
        if len(pos_sig) > 0:
            top_pos = pos_sig.sort_values('Correlation', ascending=False).head(3)
            print("  Top positive significant correlations:")
            for _, row in top_pos.iterrows():
                print(f"    - {row['Feature']}: {row['Correlation']:.3f} (CI: {row['Lower_CI']:.3f} to {row['Upper_CI']:.3f})")
                
        # Top negative significant correlations
        neg_sig = group_results[(group_results['Significant']) & (group_results['Correlation'] < 0)]
        if len(neg_sig) > 0:
            top_neg = neg_sig.sort_values('Correlation', ascending=True).head(3)
            print("  Top negative significant correlations:")
            for _, row in top_neg.iterrows():
                print(f"    - {row['Feature']}: {row['Correlation']:.3f} (CI: {row['Lower_CI']:.3f} to {row['Upper_CI']:.3f})")
    
    # Analysis by feature category
    print("\n===== ANALYSIS BY FEATURE CATEGORY =====")
    
    for group in groups:
        print(f"\n{group}:")
        
        # Count significant by category
        for category_name, category_features in [
            ("COUNT", count_features), 
            ("SURFACE", surface_features), 
            ("STRUCTURE", structure_features)
        ]:
            category_results = results_df[
                (results_df['Group'] == group) & 
                (results_df['Feature'].isin(category_features))
            ]
            
            if len(category_results) > 0:
                sig_count = sum(category_results['Significant'])
                sig_percent = sig_count / len(category_results) * 100
                
                # Calculate average correlation (absolute)
                abs_corr = category_results['Correlation'].abs().mean()
                
                print(f"  {category_name}: {sig_count}/{len(category_results)} significant ({sig_percent:.1f}%), |r| average = {abs_corr:.3f}")
    
    # Save detailed results
    results_df.to_csv('bootstrap_correlation_results.csv', index=False)
    print("\nDetailed results saved to 'bootstrap_correlation_results.csv'")
    
    return fig, results_df

In [ ]:
# Para ejecutar el análisis comparativo entre AI y Human
fig, corr_df = create_comparative_correlation_heatmap_with_bootstrap(
    df_ai=df_ai,  # Tu dataframe de evaluaciones de IA
    df_human=df_human,  # Tu dataframe de evaluaciones de humanos
    text_column='response',  # Columna con el texto a analizar
    llm_column='LLM'  # Columna que identifica el tipo de LLM
)
fig.show()

In [ ]:
def create_simplified_correlation_heatmap_with_bootstrap(df_ai, df_human, text_column='response', n_bootstrap=1000, n_samples=50, confidence=0.95):
    """
    Creates a simplified correlation heatmap with confidence intervals (only AI vs Human totals).
    Paper-ready visualization with clean academic styling.
    
    Args:
        df_ai: DataFrame with AI evaluations
        df_human: DataFrame with human evaluations
        text_column: Name of the column containing the text to analyze
        n_bootstrap: Number of bootstrap samples for confidence intervals
        n_samples: Sample size for each bootstrap iteration
        confidence: Confidence level for intervals (default 0.95)
        
    Returns:
        Plotly figure with comparative heatmap and DataFrame with correlations and intervals
    """
    import pandas as pd
    import numpy as np
    import plotly.graph_objects as go
    from scipy.stats import pearsonr
    from tqdm import tqdm
    
    def bootstrap_correlation(data, features, target, sample_size=50, n_iterations=1000, conf_level=0.95):
        """
        Calculates correlations using bootstrap resampling technique.
        """
        import numpy as np
        import pandas as pd
        from scipy.stats import pearsonr
        
        # Verify that sample_size is not larger than available data
        n = len(data)
        if sample_size > n:
            print(f"WARNING: Sample size ({sample_size}) larger than available data ({n}). Adjusting to {n}.")
            sample_size = n
        
        correlations = []
        
        # Perform n_iterations of subsampling
        for _ in range(n_iterations):
            # Select sample_size random samples (with replacement)
            indices = np.random.choice(range(n), size=sample_size, replace=True)
            bootstrap_sample = data.iloc[indices]
            
            # Calculate correlations for this sample
            corr_sample = {}
            for feature in features:
                # Verify if there's enough valid data
                valid_data = ~(bootstrap_sample[feature].isna() | bootstrap_sample[target].isna())
                if valid_data.sum() < 3:  # Need at least 3 points for correlation
                    corr_sample[feature] = np.nan
                    continue
                    
                # Use scipy.stats.pearsonr to get only the coefficient
                corr, _ = pearsonr(bootstrap_sample.loc[valid_data, feature], 
                                bootstrap_sample.loc[valid_data, target])
                corr_sample[feature] = corr
            
            correlations.append(corr_sample)
        
        # Convert to DataFrame for easier calculation
        corr_df = pd.DataFrame(correlations)
        
        # Calculate point correlation and confidence intervals
        alpha = 1 - conf_level
        result = {}
        for feature in features:
            # Filter NaN values
            feature_corrs = corr_df[feature].dropna()
            
            if len(feature_corrs) == 0:
                result[feature] = {
                    'correlation': np.nan,
                    'lower_ci': np.nan,
                    'upper_ci': np.nan,
                    'sample_count': 0
                }
                continue
            
            # Point estimate (bootstrap average)
            point_estimate = feature_corrs.mean()
            
            # Confidence intervals
            if len(feature_corrs) >= 3:
                lower_bound = np.percentile(feature_corrs, alpha/2 * 100)
                upper_bound = np.percentile(feature_corrs, (1 - alpha/2) * 100)
            else:
                lower_bound = np.nan
                upper_bound = np.nan
            
            result[feature] = {
                'correlation': point_estimate,
                'lower_ci': lower_bound,
                'upper_ci': upper_bound,
                'sample_count': len(feature_corrs)
            }
        
        return result
    
    # Prepare the dataframes
    print("Processing AI dataframe...")
    df_ai_processed = prepare_dataframe_for_analysis(df_ai, text_column=text_column)
    
    print("Processing Human dataframe...")
    df_human_processed = prepare_dataframe_for_analysis(df_human, text_column=text_column)
    
    # Define feature categories
    count_features = ['word_count', 'sentence_count']
    
    surface_features = [
        'avg_word_length', 'avg_sentence_length', 'reading_ease', 'grade_level',
        'connector_count'
    ]
    
    structure_features = [
        'unique_words_ratio', 'lexical_diversity',
        'verb_ratio', 'noun_ratio', 'adj_ratio', 'adv_ratio', 'pronoun_ratio',
        'named_entity_count', 'dependency_distance'
    ]
    
    # All features
    all_features = count_features + surface_features + structure_features
    
    # Check which features are available in both dataframes
    available_features = [f for f in all_features 
                         if f in df_ai_processed.columns and f in df_human_processed.columns]
    
    # Check if accuracy_score is available in both dataframes
    if "accuracy_score" not in df_ai_processed.columns:
        raise ValueError("'accuracy_score' not found in AI dataframe")
    if "accuracy_score" not in df_human_processed.columns:
        raise ValueError("'accuracy_score' not found in Human dataframe")
    
    # Create dataframes to store correlations and intervals
    point_correlations = pd.DataFrame(index=available_features)
    lower_ci = pd.DataFrame(index=available_features)
    upper_ci = pd.DataFrame(index=available_features)
    
    # Calculate correlations with bootstrapping for AI (Total)
    print("\nCalculating correlations with bootstrapping for AI evaluators...")
    ai_bootstrap = bootstrap_correlation(
        df_ai_processed, 
        available_features, 
        "accuracy_score", 
        sample_size=n_samples,
        n_iterations=n_bootstrap,
        conf_level=confidence
    )
    
    # Save AI results
    point_correlations["AI"] = [ai_bootstrap[feature]['correlation'] for feature in available_features]
    lower_ci["AI"] = [ai_bootstrap[feature]['lower_ci'] for feature in available_features]
    upper_ci["AI"] = [ai_bootstrap[feature]['upper_ci'] for feature in available_features]
    
    # Calculate correlations with bootstrapping for Human (Total)
    print("Calculating correlations with bootstrapping for Human evaluators...")
    human_bootstrap = bootstrap_correlation(
        df_human_processed, 
        available_features, 
        "accuracy_score", 
        sample_size=n_samples,
        n_iterations=n_bootstrap,
        conf_level=confidence
    )
    
    # Save Human results
    point_correlations["Human"] = [human_bootstrap[feature]['correlation'] for feature in available_features]
    lower_ci["Human"] = [human_bootstrap[feature]['lower_ci'] for feature in available_features]
    upper_ci["Human"] = [human_bootstrap[feature]['upper_ci'] for feature in available_features]
    
    # Sort features by category for visualization
    ordered_features = []
    for feature_group in [count_features, surface_features, structure_features]:
        ordered_features.extend([f for f in feature_group if f in available_features])
    
    # Reorder dataframe rows
    sorted_corr = point_correlations.loc[ordered_features, :]
    sorted_lower = lower_ci.loc[ordered_features, :]
    sorted_upper = upper_ci.loc[ordered_features, :]
    
    # Create a DataFrame with complete results
    result_df = pd.DataFrame()
    for col in sorted_corr.columns:
        for feature in sorted_corr.index:
            corr = sorted_corr.loc[feature, col]
            lower = sorted_lower.loc[feature, col]
            upper = sorted_upper.loc[feature, col]
            
            # Determine if correlation is significant (CI does not include zero)
            significant = False
            if not (np.isnan(lower) or np.isnan(upper)):
                significant = (lower > 0 and upper > 0) or (lower < 0 and upper < 0)
                
            new_row = pd.DataFrame({
                'Feature': [feature],
                'Evaluator': [col],
                'Correlation': [corr],
                'Lower_CI': [lower],
                'Upper_CI': [upper],
                'Significant': [significant]
            })
            result_df = pd.concat([result_df, new_row], ignore_index=True)
    
    # Create paper-ready Plotly heatmap
    fig = create_paper_ready_heatmap(
        sorted_corr, 
        sorted_lower, 
        sorted_upper, 
        confidence=confidence
    )
    
    return fig, result_df


def create_paper_ready_heatmap(point_correlations, lower_ci, upper_ci, confidence=0.95):
    """
    Creates a clean publication-ready correlation heatmap optimized for academic papers.
    
    Args:
        point_correlations: DataFrame with point correlations
        lower_ci: DataFrame with lower bounds of confidence intervals
        upper_ci: DataFrame with upper bounds of confidence intervals
        confidence: Confidence level for intervals (default 0.95)
        
    Returns:
        Plotly figure object optimized for academic publications
    """
    import plotly.graph_objects as go
    import numpy as np
    import pandas as pd
    
    # Define feature categories for better organization
    count_features = ['word_count', 'sentence_count']
    surface_features = [
        'avg_word_length', 'avg_sentence_length', 'reading_ease', 'grade_level',
        'connector_count'
    ]
    structure_features = [
        'unique_words_ratio', 'lexical_diversity',
        'verb_ratio', 'noun_ratio', 'adj_ratio', 'adv_ratio', 'pronoun_ratio',
        'named_entity_count', 'dependency_distance'
    ]
    
    # Format feature names for academic presentation
    feature_name_mapping = {
        'word_count': 'Word Count',
        'sentence_count': 'Sentence Count',
        'avg_word_length': 'Average Word Length',
        'avg_sentence_length': 'Average Sentence Length',
        'reading_ease': 'Reading Ease',
        'grade_level': 'Grade Level',
        'connector_count': 'Connector Count',
        'unique_words_ratio': 'Unique Words Ratio',
        'lexical_diversity': 'Lexical Diversity',
        'verb_ratio': 'Verb Ratio',
        'noun_ratio': 'Noun Ratio',
        'adj_ratio': 'Adjective Ratio',
        'adv_ratio': 'Adverb Ratio',
        'pronoun_ratio': 'Pronoun Ratio',
        'named_entity_count': 'Named Entity Count',
        'dependency_distance': 'Dependency Distance'
    }
    
    # Apply the mapping to create formatted labels
    sorted_corr = point_correlations.copy()
    sorted_lower = lower_ci.copy()
    sorted_upper = upper_ci.copy()
    
    sorted_corr.index = [feature_name_mapping.get(feature, feature) for feature in sorted_corr.index]
    sorted_lower.index = [feature_name_mapping.get(feature, feature) for feature in sorted_lower.index]
    sorted_upper.index = [feature_name_mapping.get(feature, feature) for feature in sorted_upper.index]
    
    # Determine significance (CI does not include zero)
    significant = ((sorted_lower > 0) & (sorted_upper > 0)) | ((sorted_lower < 0) & (sorted_upper < 0))
    
    # Prepare hover text with detailed information
    hover_texts = []
    for i, feature in enumerate(sorted_corr.index):
        feature_texts = []
        for j, column in enumerate(sorted_corr.columns):
            corr_value = sorted_corr.loc[feature, column]
            lower = sorted_lower.loc[feature, column]
            upper = sorted_upper.loc[feature, column]
            is_sig = significant.loc[feature, column]
            
            if not np.isnan(corr_value):
                text = f"<b>{feature}</b><br>"
                text += f"<b>Evaluator:</b> {column}<br>"
                text += f"<b>Correlation:</b> {corr_value:.3f}<br>"
                text += f"<b>{confidence*100:.0f}% CI:</b> [{lower:.3f}, {upper:.3f}]<br>"
                text += f"<b>Significant:</b> {'Yes' if is_sig else 'No'}"
                feature_texts.append(text)
            else:
                feature_texts.append("")
        hover_texts.append(feature_texts)
    
    # Create base figure
    fig = go.Figure()
    
    # Add heatmap without colorbar (cleaner for papers)
    fig.add_trace(go.Heatmap(
        z=sorted_corr.values,
        x=sorted_corr.columns,
        y=sorted_corr.index,
        colorscale='RdBu_r',
        zmid=0,
        zmin=-0.8,
        zmax=0.8,
        hoverinfo='text',
        hovertext=hover_texts,
        showscale=False,  # No colorbar
        colorbar=None     # Ensure no colorbar
    ))
    
    # Add annotations for correlation values with much larger fonts
    annotations = []
    for i, feature in enumerate(sorted_corr.index):
        for j, column in enumerate(sorted_corr.columns):
            corr_value = sorted_corr.loc[feature, column]
            is_sig = significant.loc[feature, column]
            
            if not np.isnan(corr_value):
                # Format correlation value
                text = f"{corr_value:.3f}"
                if is_sig:
                    text = f"<b>{text}*</b>"
                
                # Text color for readability
                text_color = 'white' if abs(corr_value) > 0.4 else 'black'
                
                annotations.append(dict(
                    x=j,
                    y=i,
                    text=text,
                    font=dict(
                        color=text_color, 
                        size=28,  # Much larger font size for correlation values
                        family="Arial Black"
                    ),
                    showarrow=False,
                    xref='x',
                    yref='y'
                ))
    
    # Calculate category boundaries for visual separation
    count_formatted = [feature_name_mapping.get(f, f) for f in count_features 
                      if feature_name_mapping.get(f, f) in sorted_corr.index]
    surface_formatted = [feature_name_mapping.get(f, f) for f in surface_features 
                        if feature_name_mapping.get(f, f) in sorted_corr.index]
    
    count_end = len(count_formatted)
    surface_end = count_end + len(surface_formatted)
    
    # Add category separation lines (thicker and more prominent)
    shapes = []
    if count_end > 0 and count_end < len(sorted_corr.index):
        shapes.append(dict(
            type="line",
            x0=-0.5,
            y0=count_end - 0.5,
            x1=len(sorted_corr.columns) - 0.5,
            y1=count_end - 0.5,
            line=dict(color="black", width=2)
        ))
    
    if surface_end > count_end and surface_end < len(sorted_corr.index):
        shapes.append(dict(
            type="line",
            x0=-0.5,
            y0=surface_end - 0.5,
            x1=len(sorted_corr.columns) - 0.5,
            y1=surface_end - 0.5,
            line=dict(color="black", width=2)
        ))
    
    # Statistical significance note (larger font and positioned better)
    sig_note = dict(
        x=0.5,
        y=-0.03,
        xref="paper",
        yref="paper",
        text=f"* <i>p</i> < {1-confidence:.2f}",
        showarrow=False,
        font=dict(size=20, family="Arial", weight="bold")  # Much larger font
      
    )
    
    # Only use correlation value annotations + significance note
    all_annotations = annotations 
    
    # Update layout with much larger fonts and better spacing
    fig.update_layout(
        height=1000,  # Increased height for better spacing
        width=600,    # Keep width reasonable
        xaxis=dict(
            title=dict(
                text="<b>Evaluator</b>",
                font=dict(size=26, family="Arial", weight="bold")  # Much larger title
            ),
            tickfont=dict(size=24, family="Arial", weight="bold"),  # Much larger tick labels
            side="bottom"
        ),
        yaxis=dict(
            title="",
            tickfont=dict(size=20, family="Arial", weight="bold"),  # Much larger feature names
            autorange="reversed"
        ),
        template="plotly_white",
        shapes=shapes,
        annotations=all_annotations,
        margin=dict(l=250, r=50, t=50, b=100),  # Much larger left margin so labels don't overlap plot
        paper_bgcolor='white',
        plot_bgcolor='white',
        font=dict(family="Arial", size=20),
        showlegend=False  # Explicitly no legend
    )
    
    return fig


def run_simplified_bootstrap_analysis(df_ai, df_human, text_column='response', n_bootstrap=1000, n_samples=50, confidence=0.95):
    """
    Runs simplified comparative analysis between AI and Human evaluators with bootstrapping.
    
    Args:
        df_ai: DataFrame with AI evaluations
        df_human: DataFrame with human evaluations
        text_column: Name of the column containing the text to analyze
        n_bootstrap: Number of bootstrap samples
        n_samples: Sample size for each bootstrap iteration
        confidence: Confidence level (0-1)
        
    Returns:
        Tuple with (figure, results dataframe)
    """
    print(f"Starting simplified comparative analysis...")
    print(f"Bootstrap parameters: {n_bootstrap} iterations, {n_samples} samples per iteration, {confidence*100:.0f}% confidence")
    
    # Run the comparative analysis
    fig, results_df = create_simplified_correlation_heatmap_with_bootstrap(
        df_ai, df_human, text_column, n_bootstrap, n_samples, confidence
    )
    
    # Print summary of findings
    print("\n===== SIMPLIFIED BOOTSTRAP ANALYSIS SUMMARY =====\n")
    
    # Count significant correlations by evaluator
    for evaluator in ['AI', 'Human']:
        evaluator_results = results_df[results_df['Evaluator'] == evaluator]
        sig_count = sum(evaluator_results['Significant'])
        total_features = len(evaluator_results)
        sig_percentage = (sig_count / total_features) * 100
        
        print(f"{evaluator} Evaluators:")
        print(f"  - Significant correlations: {sig_count}/{total_features} ({sig_percentage:.1f}%)")
        
        # Top positive correlations
        pos_corr = evaluator_results[evaluator_results['Correlation'] > 0].sort_values('Correlation', ascending=False).head(3)
        if len(pos_corr) > 0:
            print("  - Top positive correlations:")
            for _, row in pos_corr.iterrows():
                sig_marker = "*" if row['Significant'] else ""
                print(f"    • {row['Feature']}: r = {row['Correlation']:.3f} [{row['Lower_CI']:.3f}, {row['Upper_CI']:.3f}]{sig_marker}")
        
        # Top negative correlations
        neg_corr = evaluator_results[evaluator_results['Correlation'] < 0].sort_values('Correlation', ascending=True).head(3)
        if len(neg_corr) > 0:
            print("  - Top negative correlations:")
            for _, row in neg_corr.iterrows():
                sig_marker = "*" if row['Significant'] else ""
                print(f"    • {row['Feature']}: r = {row['Correlation']:.3f} [{row['Lower_CI']:.3f}, {row['Upper_CI']:.3f}]{sig_marker}")
        print()
    
    # Compare correlations between AI and Human
    print("===== AI vs HUMAN COMPARISON =====")
    ai_results = results_df[results_df['Evaluator'] == 'AI'].set_index('Feature')
    human_results = results_df[results_df['Evaluator'] == 'Human'].set_index('Feature')
    
    comparison_df = pd.DataFrame({
        'Feature': ai_results.index,
        'AI_Correlation': ai_results['Correlation'].values,
        'Human_Correlation': human_results['Correlation'].values,
        'AI_Significant': ai_results['Significant'].values,
        'Human_Significant': human_results['Significant'].values
    })
    
    comparison_df['Difference'] = comparison_df['AI_Correlation'] - comparison_df['Human_Correlation']
    comparison_df['Both_Significant'] = comparison_df['AI_Significant'] & comparison_df['Human_Significant']
    
    # Features with largest differences
    print("Features with largest correlation differences (AI - Human):")
    top_diff = comparison_df.sort_values('Difference', key=abs, ascending=False).head(5)
    for _, row in top_diff.iterrows():
        print(f"  • {row['Feature']}: Δr = {row['Difference']:+.3f} (AI: {row['AI_Correlation']:.3f}, Human: {row['Human_Correlation']:.3f})")
    
    # Save results
    results_df.to_csv('simplified_bootstrap_correlation_results.csv', index=False)
    comparison_df.to_csv('ai_human_correlation_comparison.csv', index=False)
    
    # Save figure with updated dimensions
    fig.write_image('simplified_correlation_heatmap.png', scale=2, width=600, height=1000)
    fig.write_html('simplified_correlation_heatmap.html')
    
    print(f"\nResults saved:")
    print("- simplified_bootstrap_correlation_results.csv")
    print("- ai_human_correlation_comparison.csv") 
    print("- simplified_correlation_heatmap.png")
    print("- simplified_correlation_heatmap.html")
    
    return fig, results_df

In [ ]:
# Para ejecutar el análisis comparativo entre AI y Human
fig, corr_df = create_simplified_correlation_heatmap_with_bootstrap(
    df_ai=df_ai,  # Tu dataframe de evaluaciones de IA
    df_human=df_human,  # Tu dataframe de evaluaciones de humanos
    text_column='response',  # Columna con el texto a analizar
    
)
fig.show()

In [ ]:
def create_clustered_correlation_heatmap(point_correlations, lower_ci, upper_ci, confidence=0.95):
    """
    Creates a clustered correlation heatmap with dendrogram and confidence intervals using Plotly.
    
    Args:
        point_correlations: DataFrame with point correlations
        lower_ci: DataFrame with lower bounds of confidence intervals
        upper_ci: DataFrame with upper bounds of confidence intervals
        confidence: Confidence level for intervals (default 0.95)
        
    Returns:
        Plotly figure object with clustered heatmap and dendrogram
    """
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    import numpy as np
    import pandas as pd
    from scipy.cluster.hierarchy import linkage, dendrogram
    from scipy.spatial.distance import pdist
    
    # Rename columns for better clarity
    column_mapping = {
        'AI_Total': 'AI (All)',
        'Human_Total': 'Human (All)',
        'AI_LLM0': 'AI (Human Answers)',
        'Human_LLM0': 'Human (Human Answers)',
        'AI_LLM1': 'AI (AI Answers)',
        'Human_LLM1': 'Human (AI Answers)',
        'AI_LLM2': 'AI (CoT Answers)',
        'Human_LLM2': 'Human (CoT Answers)'
    }
    
    # Create copies to avoid modifying the originals
    sorted_corr = point_correlations.copy()
    sorted_lower = lower_ci.copy()
    sorted_upper = upper_ci.copy()
    
    # Rename columns
    sorted_corr = sorted_corr.rename(columns=column_mapping)
    sorted_lower = sorted_lower.rename(columns=column_mapping)
    sorted_upper = sorted_upper.rename(columns=column_mapping)
    
    # Define feature categories
    count_features = ['word_count', 'sentence_count']
    surface_features = [
        'avg_word_length', 'avg_sentence_length', 'reading_ease', 'grade_level',
        'connector_count'
    ]
    structure_features = [
        'unique_words_ratio', 'lexical_diversity',
        'verb_ratio', 'noun_ratio', 'adj_ratio', 'adv_ratio', 'pronoun_ratio',
        'named_entity_count', 'dependency_distance'
    ]
    
    # Format feature names for better readability (remove underscores and capitalize)
    feature_name_mapping = {}
    for feature in sorted_corr.index:
        formatted_name = ' '.join(word.capitalize() for word in feature.split('_'))
        feature_name_mapping[feature] = formatted_name
    
    # Apply the mapping to create a new index
    sorted_corr.index = [feature_name_mapping.get(feature, feature) for feature in sorted_corr.index]
    sorted_lower.index = [feature_name_mapping.get(feature, feature) for feature in sorted_lower.index]
    sorted_upper.index = [feature_name_mapping.get(feature, feature) for feature in sorted_upper.index]
    
    # Sort features by category - don't cluster rows, just keep logical categories
    ordered_features = []
    for feature_group in [count_features, surface_features, structure_features]:
        for feature in feature_group:
            formatted_name = feature_name_mapping.get(feature, feature)
            if formatted_name in sorted_corr.index:
                ordered_features.append(formatted_name)
    
    # Reorder dataframe rows based on predefined categories
    sorted_corr = sorted_corr.loc[ordered_features, :]
    sorted_lower = sorted_lower.loc[ordered_features, :]
    sorted_upper = sorted_upper.loc[ordered_features, :]
    
    # Determine significance (CI does not include zero)
    significant = ((sorted_lower > 0) & (sorted_upper > 0)) | ((sorted_lower < 0) & (sorted_upper < 0))
    
    # ==== CLUSTERING APPROACH ====
    # For clustering columns, convert to numeric and handle NaNs
    numeric_corr = sorted_corr.astype(float).fillna(0)
    
    # Use the transposed correlation matrix for column clustering
    col_matrix = numeric_corr.T.values
    
    # Compute the linkage matrix for columns
    col_linkage = linkage(col_matrix, method='average', metric='euclidean')
    
    # Compute dendrogram
    col_dendrogram = dendrogram(col_linkage, no_plot=True)
    
    # Get the order of columns after clustering
    col_order = col_dendrogram['leaves']
    clustered_cols = [sorted_corr.columns[i] for i in col_order]
    
    # Print clustering information
    print("\n===== COLUMN CLUSTERING INFORMATION =====")
    print(f"Column order after clustering: {', '.join(clustered_cols)}")
    
    # Analyze the clusters (based on a height threshold)
    height_threshold = 1.5  # Adjust this based on your dendrogram
    
    # Get cluster assignments using the height threshold
    from scipy.cluster.hierarchy import fcluster
    cluster_assignments = fcluster(col_linkage, height_threshold, criterion='distance')
    clusters = {}
    
    for i, col in enumerate(sorted_corr.columns):
        cluster_id = cluster_assignments[i]
        if cluster_id not in clusters:
            clusters[cluster_id] = []
        clusters[cluster_id].append(col)
    
    print("\nClusters identified:")
    for cluster_id, members in clusters.items():
        print(f"Cluster {cluster_id}: {', '.join(members)}")
    
    # Calculate average correlations within clusters
    print("\nAverage correlation profile for each cluster:")
    for cluster_id, members in clusters.items():
        if len(members) > 1:
            cluster_data = numeric_corr[members].mean(axis=1)
            top_features = cluster_data.abs().nlargest(3)
            print(f"Cluster {cluster_id} - Top correlated features:")
            for feature, corr in top_features.items():
                print(f"  {feature}: {corr:.3f}")
    
    # ==== VISUALIZATION ====
    # Create a figure with subplots: dendrogram on top, heatmap below
    fig = make_subplots(
        rows=2, 
        cols=1, 
        row_heights=[0.2, 0.8],
        vertical_spacing=0.03,
        shared_xaxes=True
    )
    
    # Add dendrogram trace
    dendro_leaves = clustered_cols
    
    # Create a custom dendrogram trace
    # First, we need to extract the paths from the dendrogram
    icoord = col_dendrogram['icoord']
    dcoord = col_dendrogram['dcoord']
    
    # Map the x-coordinates to match column indices
    x_mapping = {i: idx for idx, i in enumerate(col_order)}
    max_x = len(col_order) - 1
    
    for xs, ys in zip(icoord, dcoord):
        # Scale the coordinates
        xs_scaled = [(x / 10) * max_x for x in xs]
        
        # Add the dendrogram line segment
        fig.add_trace(
            go.Scatter(
                x=xs_scaled,
                y=ys,
                mode='lines',
                line=dict(color='black', width=1.5),
                hoverinfo='none',
                showlegend=False
            ),
            row=1, col=1
        )
    
    # Reorder correlation matrices based on dendrogram clustering
    sorted_corr = sorted_corr[clustered_cols]
    sorted_lower = sorted_lower[clustered_cols]
    sorted_upper = sorted_upper[clustered_cols]
    significant = significant[clustered_cols]
    
    # Define category boundaries in the formatted index
    count_formatted = [feature_name_mapping.get(f, f) for f in count_features 
                      if feature_name_mapping.get(f, f) in sorted_corr.index]
    surface_formatted = [feature_name_mapping.get(f, f) for f in surface_features 
                        if feature_name_mapping.get(f, f) in sorted_corr.index]
    
    count_end = len(count_formatted)
    surface_end = count_end + len(surface_formatted)
    
    # Prepare hover text
    hover_texts = []
    for i, feature in enumerate(sorted_corr.index):
        feature_texts = []
        for j, column in enumerate(sorted_corr.columns):
            corr_value = sorted_corr.loc[feature, column]
            lower = sorted_lower.loc[feature, column]
            upper = sorted_upper.loc[feature, column]
            is_sig = significant.loc[feature, column]
            
            if not np.isnan(corr_value):
                text = f"Feature: {feature}<br>"
                text += f"Group: {column}<br>"
                text += f"Correlation: {corr_value:.3f}<br>"
                text += f"CI ({confidence*100:.0f}%): [{lower:.3f}, {upper:.3f}]<br>"
                text += f"Statistically significant: {'Yes' if is_sig else 'No'}"
                feature_texts.append(text)
            else:
                feature_texts.append("")
        hover_texts.append(feature_texts)
    
    # Add the correlation heatmap
    fig.add_trace(
        go.Heatmap(
            z=sorted_corr.values,
            x=sorted_corr.columns,
            y=sorted_corr.index,
            colorscale='RdBu_r',
            zmid=0,
            zmin=-1,
            zmax=1,
            hoverinfo='text',
            hovertext=hover_texts,
            showscale=True,
            colorbar=dict(
                title="Correlation",
                titleside="right",
                titlefont=dict(size=18),
                len=0.8,  # Make colorbar same height as heatmap
                y=0.4,  # Center the colorbar
            )
        ),
        row=2, col=1
    )
    
    # Create lists to store annotations and shapes for each subplot
    dendrogram_annotations = []
    heatmap_annotations = []
    heatmap_shapes = []
    
    # Add correlation value annotations for the heatmap
    for i, feature in enumerate(sorted_corr.index):
        for j, column in enumerate(sorted_corr.columns):
            corr_value = sorted_corr.loc[feature, column]
            is_sig = significant.loc[feature, column]
            
            if not np.isnan(corr_value):
                text = f"{corr_value:.2f}"
                if is_sig:
                    text += "*"
                
                # Text color based on value for better readability
                text_color = 'white' if abs(corr_value) > 0.5 else 'black'
                
                # Special case for Word Count 
                emphasis = False
                if feature == 'Word Count':
                    emphasis = True
                    
                heatmap_annotations.append(dict(
                    x=column,
                    y=feature,
                    text=text,
                    font=dict(
                        color=text_color, 
                        size=16 if emphasis else 14,
                        family="Arial Black" if emphasis else "Arial"
                    ),
                    showarrow=False,
                    xref='x2',  # For second subplot (heatmap)
                    yref='y2'    # For second subplot (heatmap)
                ))
    
    # Add horizontal lines to separate categories
    heatmap_shapes.append(dict(
        type="line",
        x0=-0.5,
        y0=count_end - 0.5,
        x1=len(sorted_corr.columns) - 0.5,
        y1=count_end - 0.5,
        line=dict(color="black", width=2),
        xref='x2',  # For second subplot (heatmap)
        yref='y2'   # For second subplot (heatmap)
    ))
    
    heatmap_shapes.append(dict(
        type="line",
        x0=-0.5,
        y0=surface_end - 0.5,
        x1=len(sorted_corr.columns) - 0.5,
        y1=surface_end - 0.5,
        line=dict(color="black", width=2),
        xref='x2',  # For second subplot (heatmap)
        yref='y2'   # For second subplot (heatmap)
    ))
    
    
    
    # Add note about statistical significance (as a separate annotation in the figure layout)
    fig.add_annotation(
        x=-0.21,
        y=-0.2,
        xref="paper",
        yref="paper",
        text="* Indicates statistically significant correlation (CI does not include zero)",
        showarrow=False,
        font=dict(size=14),
        align="left",
        bgcolor="rgba(255, 255, 255, 0.7)",
        borderwidth=1,
        bordercolor="black",
        borderpad=4,
        xanchor="left"
    )
    
    # Set plot dimensions
    height = max(900, 35 * len(sorted_corr.index) + 300)  # Extra height for dendrogram
    width = max(1000, 110 * len(sorted_corr.columns))
    
    # Update layout
    fig.update_layout(
        title=dict(
            text=f"Clustered Feature Correlation with Accuracy Score ({confidence*100:.0f}% CI)",
            font=dict(size=22),
            y=0.98
        ),
        height=height,
        width=width,
        showlegend=False,
        margin=dict(l=200, r=100, t=150, b=100),
        annotations=heatmap_annotations + dendrogram_annotations  # Combine annotations
    )
    
    # Update axes for dendrogram
    fig.update_xaxes(
        row=1, col=1,
        showticklabels=False,
        showgrid=False,
        zeroline=False,
        range=[-1, len(sorted_corr.columns)]
    )
    
    fig.update_yaxes(
        row=1, col=1,
        showticklabels=False,
        showgrid=False,
        zeroline=False
    )
    
    # Update axes for heatmap
    fig.update_xaxes(
        row=2, col=1,
        tickfont=dict(size=16),
        constrain='domain'
    )
    
    fig.update_yaxes(
        row=2, col=1,
        tickfont=dict(size=16),
        autorange="reversed",  # Important to keep proper order
        constrain='domain'
    )
    
    # Add shapes (category separator lines)
    for shape in heatmap_shapes:
        fig.add_shape(shape)
    
    return fig


def create_clustered_correlation_heatmap_with_bootstrap(df_ai, df_human, text_column='response', llm_column='LLM', n_bootstrap=1000, n_samples=50, confidence=0.95):
    """
    Creates a clustered correlation heatmap with dendrogram, using confidence intervals calculated through bootstrapping.
    
    Args:
        df_ai: DataFrame with AI evaluations
        df_human: DataFrame with human evaluations
        text_column: Name of the column containing the text to analyze
        llm_column: Name of the column containing the LLM type
        n_bootstrap: Number of bootstrap samples for confidence intervals
        confidence: Confidence level for intervals (default 0.95)
        
    Returns:
        Plotly figure with clustered heatmap and DataFrame with correlations and intervals
    """
    import pandas as pd
    import numpy as np
    
    # Call the original bootstrap function to get correlations and confidence intervals
    print("Calculating bootstrap correlations...")
    fig_regular, results_df = create_comparative_correlation_heatmap_with_bootstrap(
        df_ai, df_human, text_column, llm_column, n_bootstrap, n_samples, confidence
    )
    
    # Extract the correlation matrices from the original function's results
    print("Extracting correlation data for clustering...")
    
    # Get unique features and groups from the results
    features = results_df['Feature'].unique()
    groups = results_df['Group'].unique()
    
    # Create empty dataframes for correlations and CIs
    point_correlations = pd.DataFrame(index=features, columns=groups)
    lower_ci = pd.DataFrame(index=features, columns=groups)
    upper_ci = pd.DataFrame(index=features, columns=groups)
    
    # Fill dataframes with values from result_df
    for _, row in results_df.iterrows():
        feature = row['Feature']
        group = row['Group']
        point_correlations.loc[feature, group] = row['Correlation']
        lower_ci.loc[feature, group] = row['Lower_CI']
        upper_ci.loc[feature, group] = row['Upper_CI']
    
    # Create the clustered heatmap
    print("Creating clustered heatmap with dendrogram...")
    fig = create_clustered_correlation_heatmap(
        point_correlations, 
        lower_ci, 
        upper_ci, 
        confidence=confidence
    )
    
    # Save the image and HTML
    fig.write_image('clustered_correlation_heatmap.png', scale=2)
    fig.write_html('clustered_correlation_heatmap.html')
    
    print("Clustered heatmap created and saved.")
    
    return fig, results_df

In [ ]:
# Para ejecutar el análisis comparativo entre AI y Human
fig, corr_df = create_clustered_correlation_heatmap_with_bootstrap(
    df_ai=df_ai,  # Tu dataframe de evaluaciones de IA
    df_human=df_human,  # Tu dataframe de evaluaciones de humanos
    text_column='response',  # Columna con el texto a analizar
    llm_column='LLM'  # Columna que identifica el tipo de LLM
)
fig.show()

In [ ]:
def create_comparative_correlation_heatmap_with_bootstrap(df_ai, df_human, text_column='response', llm_column='LLM', n_bootstrap=1000, n_samples=50, confidence=0.95):
    """
    Crea un heatmap de correlaciones con bootstrapping, comparando cada grupo contra:
    1. La distribución general total
    2. La distribución general por tipo de LLM (0, 1, 2)
    
    Args:
        df_ai: DataFrame con evaluaciones de IA
        df_human: DataFrame con evaluaciones de humanos
        text_column: Nombre de la columna que contiene el texto a analizar
        llm_column: Nombre de la columna que identifica el tipo de LLM
        n_bootstrap: Número de muestras bootstrap para calcular intervalos de confianza
        n_samples: Tamaño de cada submuestra en el bootstrapping
        confidence: Nivel de confianza para los intervalos (por defecto 95%)
        
    Returns:
        Tuple con (figura, dataframe de resultados, distribuciones de referencia)
    """
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns
    from sklearn.preprocessing import MinMaxScaler
    from scipy.stats import pearsonr
    from tqdm import tqdm
    
    def bootstrap_correlation(data, features, target, sample_size=50, n_iterations=1000, conf_level=0.95):
        """
        Calcula correlaciones utilizando técnica de submuestreo bootstrap.
        Toma 'sample_size' muestras aleatorias, repite 'n_iterations' veces y promedia los resultados.
        
        Args:
            data: DataFrame con los datos
            features: Lista de características a correlacionar
            target: Variable objetivo (ej. 'accuracy_score')
            sample_size: Tamaño de cada submuestra (por defecto 50)
            n_iterations: Número de iteraciones (por defecto 1000)
            conf_level: Nivel de confianza para los intervalos (por defecto 95%)
            
        Returns:
            Dict con correlaciones y sus intervalos de confianza
        """
        import numpy as np
        import pandas as pd
        from scipy.stats import pearsonr
        
        # Verificar que el sample_size no sea mayor que el total de datos disponibles
        n = len(data)
        if sample_size > n:
            print(f"ADVERTENCIA: Tamaño de muestra ({sample_size}) mayor que datos disponibles ({n}). Ajustando a {n}.")
            sample_size = n
        
        correlations = []
        
        # Realizar n_iterations de submuestreo
        for _ in range(n_iterations):
            # Seleccionar sample_size muestras aleatorias (con reemplazo)
            indices = np.random.choice(range(n), size=sample_size, replace=True)
            bootstrap_sample = data.iloc[indices]
            
            # Calcular correlaciones para esta muestra
            corr_sample = {}
            for feature in features:
                # Verificar si hay suficientes datos válidos
                valid_data = ~(bootstrap_sample[feature].isna() | bootstrap_sample[target].isna())
                if valid_data.sum() < 3:  # Necesitamos al menos 3 puntos para correlación
                    corr_sample[feature] = np.nan
                    continue
                    
                # Usar scipy.stats.pearsonr para obtener solo el coeficiente
                corr, _ = pearsonr(bootstrap_sample.loc[valid_data, feature], 
                                bootstrap_sample.loc[valid_data, target])
                corr_sample[feature] = corr
            
            correlations.append(corr_sample)
        
        # Convertir a DataFrame para facilitar el cálculo
        corr_df = pd.DataFrame(correlations)
        
        # Calcular la correlación puntual y los intervalos de confianza
        alpha = 1 - conf_level
        result = {}
        for feature in features:
            # Filtrar valores NaN
            feature_corrs = corr_df[feature].dropna()
            
            if len(feature_corrs) == 0:
                # No hay correlaciones válidas para esta característica
                result[feature] = {
                    'correlation': np.nan,
                    'lower_ci': np.nan,
                    'upper_ci': np.nan,
                    'sample_count': 0,
                    'all_correlations': []  # Para almacenar todas las correlaciones bootstrap
                }
                continue
            
            # Correlación puntual (promedio de bootstrap)
            point_estimate = feature_corrs.mean()
            
            # Intervalos de confianza
            if len(feature_corrs) >= 3:  # Necesitamos al menos 3 muestras para IC
                lower_bound = np.percentile(feature_corrs, alpha/2 * 100)
                upper_bound = np.percentile(feature_corrs, (1 - alpha/2) * 100)
            else:
                # No hay suficientes muestras para calcular ICs
                lower_bound = np.nan
                upper_bound = np.nan
            
            result[feature] = {
                'correlation': point_estimate,
                'lower_ci': lower_bound,
                'upper_ci': upper_bound,
                'sample_count': len(feature_corrs),
                'all_correlations': feature_corrs.tolist()  # Guardar todas las correlaciones para comparación
            }
        
        return result
    
    # Preparar los dataframes
    print("Procesando dataframe AI...")
    df_ai_processed = prepare_dataframe_for_analysis(df_ai, text_column=text_column)
    
    print("Procesando dataframe Human...")
    df_human_processed = prepare_dataframe_for_analysis(df_human, text_column=text_column)
    
    # Definir las categorías de características
    count_features = ['word_count', 'sentence_count']
    
    surface_features = [
        'avg_word_length', 'avg_sentence_length', 'reading_ease', 'grade_level',
        'connector_count'
    ]
    
    structure_features = [
        'unique_words_ratio', 'lexical_diversity',
        'verb_ratio', 'noun_ratio', 'adj_ratio', 'adv_ratio', 'pronoun_ratio',
        'named_entity_count', 'dependency_distance'
    ]
    
    # Todas las características
    all_features = count_features + surface_features + structure_features
    
    # Definir tipos de LLM
    llm_types = {0: "Humano", 1: "IA", 2: "IA CoT"}
    
    # Verificar qué características están disponibles en ambos dataframes
    available_features = [f for f in all_features 
                         if f in df_ai_processed.columns and f in df_human_processed.columns]
    
    # Verificar si accuracy_score está disponible en ambos dataframes
    if "accuracy_score" not in df_ai_processed.columns:
        raise ValueError("No se encontró 'accuracy_score' en el dataframe de IA")
    if "accuracy_score" not in df_human_processed.columns:
        raise ValueError("No se encontró 'accuracy_score' en el dataframe de Humano")
    
    # Crear dataframes para almacenar correlaciones e intervalos
    point_correlations = pd.DataFrame(index=available_features)
    lower_ci = pd.DataFrame(index=available_features)
    upper_ci = pd.DataFrame(index=available_features)
    
    # Para guardar las distribuciones completas de correlaciones
    all_bootstrap_correlations = {}
    
    # Diccionario para almacenar todas las distribuciones de referencia
    reference_distributions = {}
    
    # 1. Primero calculamos la distribución general TOTAL (combinando todos los datos)
    print("\nCalculando distribución general TOTAL (sin distinción de grupos)...")
    
    # Combinar ambos dataframes
    df_combined = pd.concat([df_ai_processed, df_human_processed])
    
    # Calcular distribución general
    general_bootstrap = bootstrap_correlation(
        df_combined, 
        available_features, 
        "accuracy_score", 
        sample_size=n_samples,
        n_iterations=n_bootstrap,
        conf_level=confidence
    )
    
    # Guardar distribución general total
    reference_distributions["General_Total"] = {}
    for feature in available_features:
        reference_distributions["General_Total"][feature] = {
            'correlation': general_bootstrap[feature]['correlation'],
            'lower_ci': general_bootstrap[feature]['lower_ci'],
            'upper_ci': general_bootstrap[feature]['upper_ci'],
            'all_correlations': general_bootstrap[feature]['all_correlations']
        }
    
    # 2. Calcular distribuciones generales por TIPO DE LLM (0, 1, 2)
    for llm_value in [0, 1, 2]:
        llm_name = llm_types[llm_value]
        print(f"\nCalculando distribución general para respuestas tipo {llm_name} (LLM={llm_value})...")
        
        # Filtrar ambos dataframes por este tipo de LLM y combinarlos
        if llm_column in df_ai_processed.columns and llm_column in df_human_processed.columns:
            df_ai_llm = df_ai_processed[df_ai_processed[llm_column] == llm_value].copy()
            df_human_llm = df_human_processed[df_human_processed[llm_column] == llm_value].copy()
            
            # Combinar para obtener todos los datos de este tipo de LLM
            df_combined_llm = pd.concat([df_ai_llm, df_human_llm])
            
            if len(df_combined_llm) > 10:  # Asegurar suficientes datos
                print(f"  - {len(df_combined_llm)} evaluaciones totales para LLM={llm_value}")
                
                # Calcular bootstrap para este tipo de LLM
                llm_bootstrap = bootstrap_correlation(
                    df_combined_llm, 
                    available_features, 
                    "accuracy_score", 
                    sample_size=n_samples,
                    n_iterations=n_bootstrap,
                    conf_level=confidence
                )
                
                # Guardar distribución de referencia para este tipo de LLM
                reference_distributions[f"General_LLM{llm_value}"] = {}
                for feature in available_features:
                    reference_distributions[f"General_LLM{llm_value}"][feature] = {
                        'correlation': llm_bootstrap[feature]['correlation'],
                        'lower_ci': llm_bootstrap[feature]['lower_ci'],
                        'upper_ci': llm_bootstrap[feature]['upper_ci'],
                        'all_correlations': llm_bootstrap[feature]['all_correlations']
                    }
            else:
                print(f"  - No hay suficientes datos para LLM={llm_value} (mínimo 10)")
    
    # 3. Calcular correlaciones con bootstrapping para TOTAL (AI vs Human)
    print("\nCalculando correlaciones con bootstrapping para AI (Total)...")
    ai_bootstrap = bootstrap_correlation(
        df_ai_processed, 
        available_features, 
        "accuracy_score", 
        sample_size=n_samples,
        n_iterations=n_bootstrap,
        conf_level=confidence
    )
    
    # Guardar resultados
    point_correlations["AI_Total"] = [ai_bootstrap[feature]['correlation'] for feature in available_features]
    lower_ci["AI_Total"] = [ai_bootstrap[feature]['lower_ci'] for feature in available_features]
    upper_ci["AI_Total"] = [ai_bootstrap[feature]['upper_ci'] for feature in available_features]
    
    print("Calculando correlaciones con bootstrapping para Human (Total)...")
    human_bootstrap = bootstrap_correlation(
        df_human_processed, 
        available_features, 
        "accuracy_score", 
        sample_size=n_samples,
        n_iterations=n_bootstrap,
        conf_level=confidence
    )
    
    # Guardar resultados
    point_correlations["Human_Total"] = [human_bootstrap[feature]['correlation'] for feature in available_features]
    lower_ci["Human_Total"] = [human_bootstrap[feature]['lower_ci'] for feature in available_features]
    upper_ci["Human_Total"] = [human_bootstrap[feature]['upper_ci'] for feature in available_features]
    
    # 4. Ahora calcular correlaciones para cada tipo de LLM
    for llm_value in [0, 1, 2]:
        llm_name = llm_types[llm_value]
        print(f"\nProcesando correlaciones para {llm_name} (LLM={llm_value})...")
        
        # Filtrar AI dataframe por tipo de LLM
        if llm_column in df_ai_processed.columns:
            df_ai_filtered = df_ai_processed[df_ai_processed[llm_column] == llm_value].copy()
            if len(df_ai_filtered) > 10:  # Asegurar suficientes datos para bootstrap
                print(f"  - AI: {len(df_ai_filtered)} evaluaciones encontradas")
                
                # Calcular correlaciones con bootstrap
                ai_llm_bootstrap = bootstrap_correlation(
                    df_ai_filtered, 
                    available_features, 
                    "accuracy_score", 
                    sample_size=n_samples,
                    n_iterations=n_bootstrap,
                    conf_level=confidence
                )
                
                # Guardar resultados
                point_correlations[f"AI_LLM{llm_value}"] = [ai_llm_bootstrap[feature]['correlation'] for feature in available_features]
                lower_ci[f"AI_LLM{llm_value}"] = [ai_llm_bootstrap[feature]['lower_ci'] for feature in available_features]
                upper_ci[f"AI_LLM{llm_value}"] = [ai_llm_bootstrap[feature]['upper_ci'] for feature in available_features]
                
                # Guardar todas las correlaciones bootstrap de este grupo
                for feature in available_features:
                    all_bootstrap_correlations[f"AI_LLM{llm_value}_{feature}"] = ai_llm_bootstrap[feature]['all_correlations']
            else:
                print(f"  - AI: No hay suficientes evaluaciones para LLM={llm_value} (mínimo 10)")
                point_correlations[f"AI_LLM{llm_value}"] = np.nan
                lower_ci[f"AI_LLM{llm_value}"] = np.nan
                upper_ci[f"AI_LLM{llm_value}"] = np.nan
        
        # Filtrar Human dataframe por tipo de LLM
        if llm_column in df_human_processed.columns:
            df_human_filtered = df_human_processed[df_human_processed[llm_column] == llm_value].copy()
            if len(df_human_filtered) > 10:  # Asegurar suficientes datos para bootstrap
                print(f"  - Human: {len(df_human_filtered)} evaluaciones encontradas")
                
                # Calcular correlaciones con bootstrap
                human_llm_bootstrap = bootstrap_correlation(
                    df_human_filtered, 
                    available_features, 
                    "accuracy_score", 
                    sample_size=n_samples,
                    n_iterations=n_bootstrap,
                    conf_level=confidence
                )
                
                # Guardar resultados
                point_correlations[f"Human_LLM{llm_value}"] = [human_llm_bootstrap[feature]['correlation'] for feature in available_features]
                lower_ci[f"Human_LLM{llm_value}"] = [human_llm_bootstrap[feature]['lower_ci'] for feature in available_features]
                upper_ci[f"Human_LLM{llm_value}"] = [human_llm_bootstrap[feature]['upper_ci'] for feature in available_features]
                
                # Guardar todas las correlaciones bootstrap de este grupo
                for feature in available_features:
                    all_bootstrap_correlations[f"Human_LLM{llm_value}_{feature}"] = human_llm_bootstrap[feature]['all_correlations']
            else:
                print(f"  - Human: No hay suficientes evaluaciones para LLM={llm_value} (mínimo 10)")
                point_correlations[f"Human_LLM{llm_value}"] = np.nan
                lower_ci[f"Human_LLM{llm_value}"] = np.nan
                upper_ci[f"Human_LLM{llm_value}"] = np.nan
    
    # Ordenar las características por categoría para visualización
    ordered_features = []
    for feature_group in [count_features, surface_features, structure_features]:
        ordered_features.extend([f for f in feature_group if f in available_features])
    
    # Reordenar filas del dataframe
    sorted_corr = point_correlations.loc[ordered_features, :].copy()
    sorted_lower = lower_ci.loc[ordered_features, :].copy()
    sorted_upper = upper_ci.loc[ordered_features, :].copy()
    
    # Crear figuras para los heatmaps
    # 1. Heatmap comparando con la distribución general TOTAL
    plt.figure(figsize=(14, len(ordered_features) * 0.5))
    
    # Crear heatmap con anotaciones - Comparando contra la distribución general TOTAL
    heatmap_data = sorted_corr.copy()
    annot_matrix = np.empty_like(heatmap_data, dtype=object)
    
    # DataFrame para guardar qué correlaciones son significativamente diferentes de la general
    is_diff_from_total = pd.DataFrame(False, index=sorted_corr.index, columns=sorted_corr.columns)
    
    for i, feature in enumerate(sorted_corr.index):
        for j, col in enumerate(sorted_corr.columns):
            corr_value = sorted_corr.iloc[i, j]
            
            # Verificar si es significativamente diferente de la distribución general TOTAL
            if not np.isnan(corr_value) and feature in reference_distributions["General_Total"]:
                general_lower = reference_distributions["General_Total"][feature]['lower_ci']
                general_upper = reference_distributions["General_Total"][feature]['upper_ci']
                
                # Es significativamente diferente si está fuera del intervalo de confianza general
                is_diff = corr_value < general_lower or corr_value > general_upper
                is_diff_from_total.iloc[i, j] = is_diff
                
                # Formatear el texto con asterisco si es significativamente diferente
                text = f"{corr_value:.2f}"
                if is_diff:
                    text += "*"
                annot_matrix[i, j] = text
            else:
                annot_matrix[i, j] = ""
    
    # Crear heatmap comparando con distribución general TOTAL
    heatmap = sns.heatmap(
        heatmap_data, 
        annot=annot_matrix,
        cmap='coolwarm',
        vmin=-1,
        vmax=1,
        center=0,
        fmt='',
        linewidths=.5,
    )
    
    # Añadir líneas para separar categorías
    count_end = sum(1 for feature in count_features if feature in ordered_features)
    surface_end = count_end + sum(1 for feature in surface_features if feature in ordered_features)
    
    plt.axhline(y=count_end, color='black', linewidth=2)
    plt.axhline(y=surface_end, color='black', linewidth=2)
    
    # Añadir etiquetas de categoría
    plt.text(-2.5, count_end / 2, 'CONTEO', verticalalignment='center', fontsize=12, fontweight='bold')
    plt.text(-2.5, count_end + (surface_end - count_end) / 2, 'SUPERFICIE', verticalalignment='center', fontsize=12, fontweight='bold')
    plt.text(-2.5, surface_end + (len(ordered_features) - surface_end) / 2, 'ESTRUCTURA', verticalalignment='center', fontsize=12, fontweight='bold')
    
    plt.title(f'Correlaciones (vs. Distribución General TOTAL) - IC {confidence*100:.0f}%', fontsize=16)
    plt.tight_layout()
    
    # Añadir nota sobre significancia
    plt.figtext(0.5, 0.00, "* Indica correlación significativamente diferente de la distribución general TOTAL", 
                ha="center", fontsize=10, bbox={"facecolor":"white", "alpha":0.5, "pad":5})
    
    # Guardar la imagen
    total_heatmap_fig = plt.gcf()
    plt.savefig('bootstrap_correlation_vs_total.png', dpi=300, bbox_inches='tight')
    
    # 2. Crear heatmaps separados para cada comparación con tipo de LLM
    llm_heatmap_figs = {}
    is_diff_from_llm = {}
    
    for llm_value in [0, 1, 2]:
        ref_key = f"General_LLM{llm_value}"
        if ref_key not in reference_distributions:
            continue  # Saltamos si no hay distribución de referencia para este tipo
        
        plt.figure(figsize=(14, len(ordered_features) * 0.5))
        
        # Matriz para anotaciones y DataFrame para significancias
        annot_matrix = np.empty_like(sorted_corr, dtype=object)
        is_diff_from_llm[llm_value] = pd.DataFrame(False, index=sorted_corr.index, columns=sorted_corr.columns)
        
        for i, feature in enumerate(sorted_corr.index):
            for j, col in enumerate(sorted_corr.columns):
                corr_value = sorted_corr.iloc[i, j]
                
                # Verificar si es significativamente diferente de la distribución para este LLM
                if not np.isnan(corr_value) and feature in reference_distributions[ref_key]:
                    llm_lower = reference_distributions[ref_key][feature]['lower_ci']
                    llm_upper = reference_distributions[ref_key][feature]['upper_ci']
                    
                    # Es significativamente diferente si está fuera del intervalo de confianza
                    is_diff = corr_value < llm_lower or corr_value > llm_upper
                    is_diff_from_llm[llm_value].iloc[i, j] = is_diff
                    
                    # Formatear el texto con asterisco si es significativamente diferente
                    text = f"{corr_value:.2f}"
                    if is_diff:
                        text += "*"
                    annot_matrix[i, j] = text
                else:
                    annot_matrix[i, j] = ""
        
        # Crear heatmap comparando con distribución para este tipo de LLM
        heatmap = sns.heatmap(
            sorted_corr, 
            annot=annot_matrix,
            cmap='coolwarm',
            vmin=-1,
            vmax=1,
            center=0,
            fmt='',
            linewidths=.5,
        )
        
        # Añadir líneas para separar categorías
        plt.axhline(y=count_end, color='black', linewidth=2)
        plt.axhline(y=surface_end, color='black', linewidth=2)
        
        # Añadir etiquetas de categoría
        plt.text(-2.5, count_end / 2, 'CONTEO', verticalalignment='center', fontsize=12, fontweight='bold')
        plt.text(-2.5, count_end + (surface_end - count_end) / 2, 'SUPERFICIE', verticalalignment='center', fontsize=12, fontweight='bold')
        plt.text(-2.5, surface_end + (len(ordered_features) - surface_end) / 2, 'ESTRUCTURA', verticalalignment='center', fontsize=12, fontweight='bold')
        
        llm_name = llm_types[llm_value]
        plt.title(f'Correlaciones (vs. Distribución de {llm_name} LLM={llm_value}) - IC {confidence*100:.0f}%', fontsize=16)
        plt.tight_layout()
        
        # Añadir nota sobre significancia
        plt.figtext(0.5, 0.00, f"* Indica correlación significativamente diferente de la distribución de {llm_name} (LLM={llm_value})", 
                    ha="center", fontsize=10, bbox={"facecolor":"white", "alpha":0.5, "pad":5})
        
        # Guardar la imagen
        llm_heatmap_figs[llm_value] = plt.gcf()
        plt.savefig(f'bootstrap_correlation_vs_llm{llm_value}.png', dpi=300, bbox_inches='tight')
    
    # Crear un DataFrame con los resultados completos
    result_df = pd.DataFrame()
    for col in sorted_corr.columns:
        for feature in sorted_corr.index:
            corr = sorted_corr.loc[feature, col]
            lower = sorted_lower.loc[feature, col] if col in sorted_lower.columns else np.nan
            upper = sorted_upper.loc[feature, col] if col in sorted_upper.columns else np.nan
            
            # Datos de la distribución general TOTAL para esta característica
            general_corr = reference_distributions["General_Total"][feature]['correlation'] if feature in reference_distributions["General_Total"] else np.nan
            general_lower = reference_distributions["General_Total"][feature]['lower_ci'] if feature in reference_distributions["General_Total"] else np.nan
            general_upper = reference_distributions["General_Total"][feature]['upper_ci'] if feature in reference_distributions["General_Total"] else np.nan
            
            # Determinar si es significativamente diferente de la distribución general TOTAL
            is_diff_total = False
            if not np.isnan(corr) and not np.isnan(general_lower) and not np.isnan(general_upper):
                is_diff_total = corr < general_lower or corr > general_upper
            
            # Extraer tipo de evaluador y tipo de respuesta
            if "AI" in col:
                evaluator = "IA"
            else:
                evaluator = "Humano"
                
            if "Total" in col:
                response_type = "Todas"
                llm_num = -1  # No aplica
            else:
                # Extraer el número del tipo de respuesta
                resp_num = int(col.split("LLM")[-1])
                response_type = llm_types.get(resp_num, f"Tipo {resp_num}")
                llm_num = resp_num
            
            # Crear fila base para este par (característica, grupo)
            row_data = {
                'Feature': feature,
                'Group': col,
                'Evaluador': evaluator,
                'Tipo_Respuesta': response_type,
                'LLM_Number': llm_num,
                'Correlation': corr,
                'Lower_CI': lower,
                'Upper_CI': upper,
                'General_Total_Correlation': general_corr,
                'General_Total_Lower_CI': general_lower,
                'General_Total_Upper_CI': general_upper,
                'Diff_From_General_Total': is_diff_total
            }
            
            # Añadir comparaciones con distribuciones por tipo de LLM
            for llm_value in [0, 1, 2]:
                ref_key = f"General_LLM{llm_value}"
                if ref_key in reference_distributions and feature in reference_distributions[ref_key]:
                    # Datos de la distribución de referencia para este tipo de LLM
                    llm_corr = reference_distributions[ref_key][feature]['correlation']
                    llm_lower = reference_distributions[ref_key][feature]['lower_ci']
                    llm_upper = reference_distributions[ref_key][feature]['upper_ci']
                    
                    # Determinar si es significativamente diferente
                    is_diff_llm = False
                    if not np.isnan(corr) and not np.isnan(llm_lower) and not np.isnan(llm_upper):
                        is_diff_llm = corr < llm_lower or corr > llm_upper
                    
                    # Añadir a la fila
                    row_data[f'General_LLM{llm_value}_Correlation'] = llm_corr
                    row_data[f'General_LLM{llm_value}_Lower_CI'] = llm_lower
                    row_data[f'General_LLM{llm_value}_Upper_CI'] = llm_upper
                    row_data[f'Diff_From_General_LLM{llm_value}'] = is_diff_llm
            
            # Añadir la fila al DataFrame de resultados
            result_df = result_df.append(row_data, ignore_index=True)
    
    return total_heatmap_fig, llm_heatmap_figs, result_df, reference_distributions

def run_bootstrap_correlation_analysis_multi_ref(df_ai, df_human, text_column='response', llm_column='LLM', n_bootstrap=1000, n_samples=50, confidence=0.95):
    """
    Ejecuta el análisis comparativo con bootstrapping, comparando contra múltiples distribuciones:
    1. Distribución general TOTAL
    2. Distribuciones generales por tipo de LLM (0, 1, 2)
    
    Args:
        df_ai: DataFrame con evaluaciones de IA
        df_human: DataFrame con evaluaciones de humanos
        text_column: Nombre de la columna que contiene el texto a analizar
        llm_column: Nombre de la columna que identifica el tipo de LLM
        n_bootstrap: Número de iteraciones bootstrap
        n_samples: Tamaño de cada submuestra
        confidence: Nivel de confianza (0-1)
        
    Returns:
        Tuple con (figuras, dataframe de resultados, distribuciones de referencia)
    """
    print(f"Iniciando análisis comparativo con múltiples distribuciones de referencia ({n_bootstrap} iteraciones con {n_samples} samples, {confidence*100:.0f}% confianza)...")
    
    # Ejecutar el análisis comparativo
    total_fig, llm_figs, results_df, reference_distributions = create_comparative_correlation_heatmap_with_bootstrap(
        df_ai, df_human, text_column, llm_column, n_bootstrap, n_samples, confidence
    )
    
    # Imprimir un resumen de los hallazgos
    print("\n===== RESUMEN DEL ANÁLISIS CON MÚLTIPLES DISTRIBUCIONES DE REFERENCIA =====\n")
    
    # Definir las categorías de características
    count_features = ['word_count', 'sentence_count']
    surface_features = [
        'avg_word_length', 'avg_sentence_length', 'reading_ease', 'grade_level',
        'connector_count'
    ]
    structure_features = [
        'unique_words_ratio', 'lexical_diversity',
        'verb_ratio', 'noun_ratio', 'adj_ratio', 'adv_ratio', 'pronoun_ratio',
        'named_entity_count', 'dependency_distance'
    ]
    
    # 1. Análisis de diferencias con respecto a la distribución general TOTAL
    print("\n----- DIFERENCIAS RESPECTO A LA DISTRIBUCIÓN GENERAL TOTAL -----")
    
    significant_diff_total = results_df[results_df['Diff_From_General_Total'] == True]
    print(f"\nEncontradas {len(significant_diff_total)} correlaciones significativamente diferentes de la distribución general TOTAL")
    
    # Analizar por grupo
    for group in results_df['Group'].unique():
        group_results = results_df[results_df['Group'] == group]
        sig_count = sum(group_results['Diff_From_General_Total'])
        print(f"\n{group}: {sig_count} correlaciones significativamente diferentes de {len(group_results)}")
        
        if sig_count > 0:
            sig_results = group_results[group_results['Diff_From_General_Total']]
            print("  Características que difieren significativamente de la distribución general TOTAL:")
            
            # Ordenar por magnitud de la diferencia con respecto a la general
            sig_results['diff_magnitude'] = abs(sig_results['Correlation'] - sig_results['General_Total_Correlation'])
            sig_results = sig_results.sort_values('diff_magnitude', ascending=False)
            
            for idx, row in sig_results.head(5).iterrows():
                diff = row['Correlation'] - row['General_Total_Correlation']
                print(f"    - {row['Feature']}: {row['Correlation']:.3f} vs general {row['General_Total_Correlation']:.3f} (diff: {diff:.3f})")
    
    # 2. Análisis de diferencias con respecto a las distribuciones por tipo de LLM
    for llm_value in [0, 1, 2]:
        if f"General_LLM{llm_value}_Correlation" not in results_df.columns:
            continue
            
        llm_name = {0: "Humano", 1: "IA", 2: "IA CoT"}.get(llm_value, f"Tipo {llm_value}")
        print(f"\n----- DIFERENCIAS RESPECTO A LA DISTRIBUCIÓN DE {llm_name.upper()} (LLM={llm_value}) -----")
        
        significant_diff_llm = results_df[results_df[f'Diff_From_General_LLM{llm_value}'] == True]
        print(f"\nEncontradas {len(significant_diff_llm)} correlaciones significativamente diferentes de la distribución de {llm_name} (LLM={llm_value})")
        
        # Analizar por grupo
        for group in results_df['Group'].unique():
            group_results = results_df[results_df['Group'] == group]
            sig_count = sum(group_results[f'Diff_From_General_LLM{llm_value}'])
            print(f"\n{group}: {sig_count} correlaciones significativamente diferentes de {len(group_results)}")
            
            if sig_count > 0:
                sig_results = group_results[group_results[f'Diff_From_General_LLM{llm_value}']]
                print(f"  Características que difieren significativamente de la distribución de {llm_name} (LLM={llm_value}):")
                
                # Ordenar por magnitud de la diferencia
                sig_results['diff_magnitude'] = abs(sig_results['Correlation'] - sig_results[f'General_LLM{llm_value}_Correlation'])
                sig_results = sig_results.sort_values('diff_magnitude', ascending=False)
                
                for idx, row in sig_results.head(5).iterrows():
                    diff = row['Correlation'] - row[f'General_LLM{llm_value}_Correlation']
                    print(f"    - {row['Feature']}: {row['Correlation']:.3f} vs {llm_name} {row[f'General_LLM{llm_value}_Correlation']:.3f} (diff: {diff:.3f})")
    
    # 3. Análisis por categoría de características y tipo de evaluador
    print("\n===== ANÁLISIS POR CATEGORÍA DE CARACTERÍSTICAS =====")
    
    for evaluator in ["IA", "Humano"]:
        print(f"\nEvaluaciones por {evaluator}:")
        
        evaluator_results = results_df[results_df['Evaluador'] == evaluator]
        
        # Contar significativas por categoría
        for category_name, category_features in [
            ("CONTEO", count_features), 
            ("SUPERFICIE", surface_features), 
            ("ESTRUCTURA", structure_features)
        ]:
            category_results = evaluator_results[evaluator_results['Feature'].isin(category_features)]
            
            if len(category_results) > 0:
                # Diferencias respecto a distribución TOTAL
                sig_count_total = sum(category_results['Diff_From_General_Total'])
                sig_percent_total = sig_count_total / len(category_results) * 100
                
                print(f"  {category_name}: {sig_count_total}/{len(category_results)} difieren de la distribución general ({sig_percent_total:.1f}%)")
                
                # Diferencias respecto a distribuciones por tipo de LLM
                for llm_value in [0, 1, 2]:
                    if f'Diff_From_General_LLM{llm_value}' in category_results.columns:
                        llm_name = {0: "Humano", 1: "IA", 2: "IA CoT"}.get(llm_value, f"Tipo {llm_value}")
                        sig_count_llm = sum(category_results[f'Diff_From_General_LLM{llm_value}'])
                        sig_percent_llm = sig_count_llm / len(category_results) * 100
                        
                        print(f"    vs {llm_name}: {sig_count_llm}/{len(category_results)} difieren ({sig_percent_llm:.1f}%)")
    
    # 4. Análisis por tipo de respuesta
    print("\n===== ANÁLISIS POR TIPO DE RESPUESTA =====")
    
    for resp_type in ["Todas", "Humano", "IA", "IA CoT"]:
        if resp_type == "Todas":
            resp_results = results_df[results_df['Tipo_Respuesta'] == resp_type]
        else:
            llm_value = {"Humano": 0, "IA": 1, "IA CoT": 2}.get(resp_type, -1)
            resp_results = results_df[results_df['LLM_Number'] == llm_value]
            
        if len(resp_results) == 0:
            continue
            
        print(f"\nRespuestas tipo {resp_type}:")
        
        # Diferencias respecto a distribución TOTAL
        sig_count_total = sum(resp_results['Diff_From_General_Total'])
        sig_percent_total = sig_count_total / len(resp_results) * 100
        
        print(f"  Diferencias vs distribución general: {sig_count_total}/{len(resp_results)} ({sig_percent_total:.1f}%)")
        
        # Top características diferentes (vs total)
        if sig_count_total > 0:
            sig_results = resp_results[resp_results['Diff_From_General_Total']]
            sig_results['diff_magnitude'] = abs(sig_results['Correlation'] - sig_results['General_Total_Correlation'])
            top_diffs = sig_results.sort_values('diff_magnitude', ascending=False).head(3)
            
            print("  Top características diferentes vs distribución general:")
            for idx, row in top_diffs.iterrows():
                diff = row['Correlation'] - row['General_Total_Correlation']
                print(f"    - {row['Feature']}: {row['Correlation']:.3f} vs general {row['General_Total_Correlation']:.3f} (diff: {diff:.3f})")
    
    # Guardar resultados detallados
    results_df.to_csv('bootstrap_correlation_results_multi_ref.csv', index=False)
    print("\nResultados detallados guardados en 'bootstrap_correlation_results_multi_ref.csv'")
    
    return total_fig, llm_figs, results_df, reference_distributions

In [ ]:
# Para ejecutar el análisis completo con múltiples distribuciones de referencia
total_fig, llm_figs, results, ref_dists = run_bootstrap_correlation_analysis_multi_ref(
    df_ai=df_ai,
    df_human=df_human,
    text_column='response',
    llm_column='LLM',
    n_bootstrap=1000,  # Número de iteraciones bootstrap
    n_samples=50,      # Tamaño de cada submuestra
    confidence=0.95    # Nivel de confianza
)

In [ ]:
df = load_data("eval")

In [ ]:

respuestas_completas = df['question_id'].unique()
len(respuestas_completas)

In [ ]:

# Primero, vamos a crear un diccionario para almacenar qué tipos de LLM tiene cada pregunta
preguntas_tipos = {}

# Iteramos por cada fila del dataframe
for _, row in df.iterrows():
    pregunta_id = row['question_id']
    
    llm_type = row['LLM']
    
    # Inicializamos el conjunto para esta pregunta si no existe
    if pregunta_id not in preguntas_tipos:
        preguntas_tipos[pregunta_id] = set()
    
    # Añadimos el tipo de LLM al conjunto de esta pregunta
    preguntas_tipos[pregunta_id].add(llm_type)
print(len(pregunta_id))
# Contadores para cada combinación
contador_combinaciones = {
    'tipos_0_1_2': 0,  # Tiene los tres tipos
    'tipos_0_1': 0,    # Solo tiene tipos 0 y 1
    'tipos_0_2': 0,    # Solo tiene tipos 0 y 2
    'tipos_1_2': 0,    # Solo tiene tipos 1 y 2
    'solo_0': 0,       # Solo tiene tipo 0
    'solo_1': 0,       # Solo tiene tipo 1
    'solo_2': 0        # Solo tiene tipo 2
}

# Conjuntos para guardar los IDs de cada combinación
ids_por_combinacion = {
    'tipos_0_1_2': [],
    'tipos_0_1': [],
    'tipos_0_2': [],
    'tipos_1_2': [],
    'solo_0': [],
    'solo_1': [],
    'solo_2': []
}

# Clasificamos cada pregunta según los tipos que tiene
for pregunta_id, tipos in preguntas_tipos.items():
    if tipos == {0, 1, 2}:
        contador_combinaciones['tipos_0_1_2'] += 1
        ids_por_combinacion['tipos_0_1_2'].append(pregunta_id)
    elif tipos == {0, 1}:
        contador_combinaciones['tipos_0_1'] += 1
        ids_por_combinacion['tipos_0_1'].append(pregunta_id)
    elif tipos == {0, 2}:
        contador_combinaciones['tipos_0_2'] += 1
        ids_por_combinacion['tipos_0_2'].append(pregunta_id)
    elif tipos == {1, 2}:
        contador_combinaciones['tipos_1_2'] += 1
        ids_por_combinacion['tipos_1_2'].append(pregunta_id)
    elif tipos == {0}:
        contador_combinaciones['solo_0'] += 1
        ids_por_combinacion['solo_0'].append(pregunta_id)
    elif tipos == {1}:
        contador_combinaciones['solo_1'] += 1
        ids_por_combinacion['solo_1'].append(pregunta_id)
    elif tipos == {2}:
        contador_combinaciones['solo_2'] += 1
        ids_por_combinacion['solo_2'].append(pregunta_id)

# Mostramos los resultados
print("Resumen de combinaciones de tipos LLM por pregunta:")
print(f"Preguntas con tipos 0, 1 y 2: {contador_combinaciones['tipos_0_1_2']}")
print(f"Preguntas con tipos 0 y 1: {contador_combinaciones['tipos_0_1']}")
print(f"Preguntas con tipos 0 y 2: {contador_combinaciones['tipos_0_2']}")
print(f"Preguntas con tipos 1 y 2: {contador_combinaciones['tipos_1_2']}")
print(f"Preguntas solo con tipo 0: {contador_combinaciones['solo_0']}")
print(f"Preguntas solo con tipo 1: {contador_combinaciones['solo_1']}")
print(f"Preguntas solo con tipo 2: {contador_combinaciones['solo_2']}")


In [ ]:
len(df["question_id"].unique())

In [ ]:
import plotly.graph_objects as go
import plotly.express as px
import pandas as pd
import re

# Raw data from your benchmark results
raw_data = [
    ("./output_2/rag_num_docs:1_embeddings:thenlper~gte-small_reader-model:phi3:3.8b.json", 0.396481),
    ("./output_2/rag_num_docs:6_embeddings:thenlper~gte-small_reader-model:llama3.1.json", 0.469722),
    ("./output_2/rag_num_docs:5_embeddings:thenlper~gte-small_reader-model:gemma2:2b.json", 0.495090),
    ("./output_2/rag_num_docs:5_embeddings:thenlper~gte-small_reader-model:llama3.1.json", 0.498773),
    ("./output_2/rag_num_docs:1_embeddings:thenlper~gte-small_reader-model:llama3.1.json", 0.517185),
    ("./output_2/rag_num_docs:6_embeddings:thenlper~gte-small_reader-model:gemma2:2b.json", 0.518412),
    ("./output_2/rag_num_docs:1_embeddings:thenlper~gte-small_reader-model:gemma2:2b.json", 0.518822),
    ("./output_2/rag_num_docs:6_embeddings:thenlper~gte-small_reader-model:phi3:3.8b.json", 0.528642),
    ("./output_2/rag_num_docs:5_embeddings:thenlper~gte-small_reader-model:phi3:3.8b.json", 0.538462),
    ("./output_2/rag_num_docs:1_embeddings:thenlper~gte-small_reader-model:qwen2:7b.json", 0.541735),
    ("./output_2/rag_num_docs:4_embeddings:thenlper~gte-small_reader-model:gemma2:2b.json", 0.552373),
    ("./output_2/rag_num_docs:6_embeddings:thenlper~gte-small_reader-model:qwen2:7b.json", 0.565876),
    ("./output_2/rag_num_docs:3_embeddings:thenlper~gte-small_reader-model:llama3.1.json", 0.567512),
    ("./output_2/rag_num_docs:4_embeddings:thenlper~gte-small_reader-model:llama3.1.json", 0.569149),
    ("./output_2/rag_num_docs:2_embeddings:thenlper~gte-small_reader-model:llama3.1.json", 0.569967),
    ("./output_2/rag_num_docs:2_embeddings:thenlper~gte-small_reader-model:gemma2:2b.json", 0.575696),
    ("./output_2/rag_num_docs:3_embeddings:thenlper~gte-small_reader-model:gemma2:2b.json", 0.577332),
    ("./output_2/rag_num_docs:2_embeddings:thenlper~gte-small_reader-model:phi3:3.8b.json", 0.580606),
    ("./output_2/rag_num_docs:5_embeddings:thenlper~gte-small_reader-model:qwen2:7b.json", 0.584288),
    ("./output_2/rag_num_docs:2_embeddings:thenlper~gte-small_reader-model:qwen2:7b.json", 0.591653),
    ("./output_2/rag_num_docs:3_embeddings:thenlper~gte-small_reader-model:qwen2:7b.json", 0.623568),
    ("./output_2/rag_num_docs:4_embeddings:thenlper~gte-small_reader-model:qwen2:7b.json", 0.632979),
    ("./output_2/rag_num_docs:4_embeddings:thenlper~gte-small_reader-model:phi3:3.8b.json", 0.695990),
    ("./output_2/rag_num_docs:3_embeddings:thenlper~gte-small_reader-model:phi3:3.8b.json", 0.709083),
]

def parse_filename(filename):
    """Extract number of documents and model name from filename"""
    # Extract number of documents
    num_docs_match = re.search(r'num_docs:(\d+)', filename)
    num_docs = int(num_docs_match.group(1)) if num_docs_match else None
    
    # Extract model name
    model_match = re.search(r'reader-model:([^.]+)', filename)
    if model_match:
        model_raw = model_match.group(1)
        # Clean up model names for better display
        if 'phi3' in model_raw:
            model = 'Phi-3 3.8B'
        elif 'llama3' in model_raw:
            model = 'Llama 3.1'
        elif 'gemma2' in model_raw:
            model = 'Gemma2 2B'
        elif 'qwen2' in model_raw:
            model = 'Qwen2 7B'
        else:
            model = model_raw
    else:
        model = 'Unknown'
    
    return num_docs, model

# Parse data and create DataFrame
data_list = []
for filename, score in raw_data:
    num_docs, model = parse_filename(filename)
    data_list.append({
        'num_docs': num_docs,
        'model': model,
        'score': score,
        'filename': filename
    })

df = pd.DataFrame(data_list)

# Define colors for each model (using professional color palette)
model_colors = {
    'Phi-3 3.8B': '#1f77b4',     # Blue
    'Llama 3.1': '#ff7f0e',      # Orange
    'Gemma2 2B': '#2ca02c',      # Green
    'Qwen2 7B': '#d62728'        # Red
}

# Create the plot
fig = go.Figure()

# Add traces for each model
for model in df['model'].unique():
    model_data = df[df['model'] == model].sort_values('num_docs')
    
    fig.add_trace(go.Scatter(
        x=model_data['num_docs'],
        y=model_data['score'],
        mode='lines+markers',
        name=model,
        line=dict(color=model_colors[model], width=2.5),
        marker=dict(
            color=model_colors[model],
            size=10,
            line=dict(color='white', width=2)
        ),
        hovertemplate='<b>%{fullData.name}</b><br>' +
                     'Retrieved Documents: %{x}<br>' +
                     'Performance Score: %{y:.4f}<br>' +
                     '<extra></extra>'
    ))

# Update layout for paper-quality appearance with larger text
fig.update_layout(
    title={
        'text': 'RAG System Performance vs Number of Retrieved Documents',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 24, 'family': 'Arial, sans-serif', 'color': '#2c3e50'}
    },
    xaxis={
        'title': 'Number of Retrieved Documents',
        'title_font': {'size': 18, 'family': 'Arial, sans-serif'},
        'tickfont': {'size': 16, 'family': 'Arial, sans-serif'},
        'showgrid': True,
        'gridcolor': '#e8e8e8',
        'gridwidth': 1,
        'zeroline': False,
        'dtick': 1,  # Show every integer
        'range': [0.5, 6.5]
    },
    yaxis={
        'title': 'Performance Score',
        'title_font': {'size': 18, 'family': 'Arial, sans-serif'},
        'tickfont': {'size': 16, 'family': 'Arial, sans-serif'},
        'showgrid': True,
        'gridcolor': '#e8e8e8',
        'gridwidth': 1,
        'zeroline': False,
        'tickformat': '.3f'
    },
    legend={
        'x': 0.02,
        'y': 0.98,
        'bgcolor': 'rgba(255,255,255,0.8)',
        'bordercolor': '#cccccc',
        'borderwidth': 1,
        'font': {'size': 16, 'family': 'Arial, sans-serif'}
    },
    plot_bgcolor='white',
    paper_bgcolor='white',
    width=900,
    height=700,
    margin=dict(l=100, r=50, t=100, b=80)
)

# Add subtle background grid
fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='rgba(128,128,128,0.2)')
fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='rgba(128,128,128,0.2)')

# Show the plot
fig.show()

# Optional: Save the plot as a high-quality image
# fig.write_image("rag_benchmark_results.png", width=900, height=700, scale=2)
# fig.write_html("rag_benchmark_results.html")

# Print summary statistics
print("\n=== RAG Benchmark Summary ===")
print(f"Total configurations tested: {len(df)}")
print(f"Models evaluated: {', '.join(df['model'].unique())}")
print(f"Number of documents range: {df['num_docs'].min()}-{df['num_docs'].max()}")
print(f"Performance score range: {df['score'].min():.4f}-{df['score'].max():.4f}")

print("\n=== Best Performance by Model ===")
for model in df['model'].unique():
    model_data = df[df['model'] == model]
    best_config = model_data.loc[model_data['score'].idxmax()]
    print(f"{model}: {best_config['score']:.4f} (with {best_config['num_docs']} documents)")

print("\n=== Overall Best Configuration ===")
best_overall = df.loc[df['score'].idxmax()]
print(f"Model: {best_overall['model']}")
print(f"Documents: {best_overall['num_docs']}")
print(f"Score: {best_overall['score']:.4f}")